---
>「語っている対象を測ることができ、それを数で表せるとき、人はその対象について何かを知っている」
>
> ケルビン卿（ウィリアム・トムソン）

---

# LLMの評価 — 言語モデルとその応用システムを「測る」技術

本テキストでは、大規模言語モデル（LLM）と、LLMを組み込んだアプリケーション（RAG、エージェント）の**評価（evaluation）**を体系的に学ぶ

画像分類であれば、正解ラベルと予測を比べて正解率を出せば評価は終わる
LLMではそうはいかない

- 同じ質問に対して、正しい答え方が何通りもある
- 同じ入力でも、実行のたびに出力が変わる
- プロンプトの言い回しを少し変えただけで、成績が上下する

それでも、モデルを選ぶ、プロンプトを直す、検索の方式を変える、といった判断はすべて「どちらが良いか」の比較である
比較の物差しが信用できなければ、改良したつもりで悪化させていても気づけない

**評価は、LLMを使った開発における計測器である**
本テキストでは、その計測器を自分の手で組み立て、計測器そのものの誤差まで確かめる

本テキストでは以下を扱う：

1. **LLMの評価が難しい理由**: 正解の多様性、確率的な出力、プロンプトへの敏感さ、評価データの汚染
2. **評価データの作り方**: 設計、層別、規模と統計的な誤差、ブートストラップによる信頼区間
3. **自動指標**: 完全一致、F1、BLEU、ROUGE、埋め込み類似度、パープレキシティ
4. **選択式ベンチマーク**: 対数尤度による採点と、生成させて抽出する採点
5. **LLMによる採点（LLM-as-a-judge）**: 採点基準、ペア比較、バイアス、人手ラベルとの一致率
6. **RAGの評価**: 検索段と生成段を分けて測る
7. **エージェントの評価**: 成否、コスト、ツール呼び出し、軌跡、成功率のばらつき
8. **回帰試験**: 変更のたびに評価を回し、悪化を検出する
9. **落とし穴**: 評価セットへの過適合、グッドハートの法則、ベンチマークの飽和

---

# この講義でできるようになること

- 目的に合った評価データを設計し、その規模から「何点差までが誤差か」を見積もれるようになる
- 正解率に信頼区間を付け、2つのモデルやプロンプトの差が偶然かどうかを判断できるようになる
- 自動指標、LLMによる採点、人手評価のそれぞれが、何を測れて何を測れないかを説明できるようになる
- RAGとエージェントを、部品ごとに分けて評価できるようになる
- プロンプトやモデルを変えるたびに自動で回す回帰試験を書けるようになる

# 全体像

評価の作業は、対象が何であっても次の4つの部品からなる

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-overview.png" width=720>

上の図では、評価データから判断までが左から右へ流れる
下の枠は、変更のたびに同じ流れをもう一度たどること（回帰試験）を表す

| 部品 | 内容 | 本テキストの節 |
| --- | --- | --- |
| **評価データ** | 何を入力し、何を正解とみなすか | 評価データの作り方 |
| **採点器** | 出力を点数に変える仕組み（文字列の比較、LLMによる採点、人手） | 自動指標、選択式ベンチマーク、LLMによる採点 |
| **集計と統計** | 点数を平均し、その誤差を見積もる | 評価データの作り方 |
| **運用** | 変更のたびに回し、前回と比べる | 回帰試験として回す |

評価対象が複雑になるほど、採点する箇所が増える

| 評価対象 | 採点する箇所 |
| --- | --- |
| LLM単体 | 最終的な出力 |
| RAG | 検索結果（正しい文書を取れたか）と、生成結果（文書に基づいて答えたか） |
| エージェント | 最終結果、途中のツール呼び出し、ステップ数とコスト |

既存の章との関係は次のとおりである

- 分類の評価指標（正解率、適合率、再現率、F値）は `mlsys-text-2-ML基礎.ipynb` の「評価手法」を参照
- 検索の指標（Precision@k、Recall@k、MRR）の導入は `mlsys-text-J-NLP-1-Basics.ipynb` を参照
- パープレキシティと、調整の前後の比較は `mlsys-text-J-NLP-8-LLM調整.ipynb` を参照
- エージェントの実装と失敗の型は `mlsys-text-M-LLM-5-Agent設計.ipynb`、提供時の品質の測定とカナリア公開は `mlsys-text-S-推論サービング.ipynb`、安全性の評価とウィルソンの区間は `mlsys-text-T-安全性と公平性.ipynb` を参照
- 機械翻訳のBLEUは `mlsys-text-J-NLP-4-Transformer-full.ipynb` を参照
- RAGの構成は `mlsys-text-M-LLM-2-Application.ipynb`、エージェントの構成と評価指標の一覧は `mlsys-text-M-LLM-3-Agent.ipynb` を参照

本テキストは、それらの章で名前だけ挙げた指標を、実際に実装して動かすことを目的とする

実習では、APIキーが不要な小型の公開モデルを使う
小型のモデルは間違いが多いが、評価の実習ではそれがかえって都合がよい
全問正解するモデルでは、採点器の良し悪しを比べられないからである

規模の異なる3つのモデル（0.5B、1.5B、3B）を使い、モデルを大きくすると偏りや成績がどう変わるかも、各節で確かめる

---

# 準備

## ライブラリの導入

- `transformers`: 公開モデルの読み込みと推論
- `sacrebleu`: BLEUの計算

In [ ]:
!pip install -q transformers sacrebleu

In [ ]:
# ライブラリの読み込みと乱数シードの固定
import ast, json, math, operator, os, random, re, time, unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## モデルの読み込み

評価対象として、規模の異なる2つの対話用モデルを用意する

- `Qwen/Qwen2.5-1.5B-Instruct`（以下、**大モデル**）
- `Qwen/Qwen2.5-0.5B-Instruct`（以下、**小モデル**）

どちらもHugging Faceから鍵なしで取得でき、2つ合わせてもT4のGPUメモリに収まる

2つ用意するのは、評価の目的の多くが「AとBのどちらが良いか」の比較だからである

In [ ]:
# 評価対象のLLMを2つ読み込む（どちらも鍵なしで取得できる公開モデル）
def load_llm(name):
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"  # まとめて生成するとき、文末をそろえるために左側を埋める
    model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()
    if device.type == "cuda":
        model.half()  # 半精度にしてGPUメモリを節約する
    return {"name": name, "tok": tok, "model": model}

LLM_L = load_llm("Qwen/Qwen2.5-1.5B-Instruct")  # 大きい方（約15億パラメータ）
LLM_S = load_llm("Qwen/Qwen2.5-0.5B-Instruct")  # 小さい方（約5億パラメータ）

### 比較用の3Bモデル

小型のモデルでは、評価にまつわる偏り（特定の記号ばかり答える、示した順序で判定が変わる、など）が極端な形で現れる

偏りが見えやすいのは実習には都合がよいが、「それは小さいモデルだからではないか」という疑問が残る

そこで、3つ目のモデルとして `Qwen/Qwen2.5-3B-Instruct`（以下、**3Bモデル**）を用意する

- 各節の主な実験は、大モデル（1.5B）と小モデル（0.5B）で行う
- 節の終わりで3Bモデルを加え、**モデルを大きくすると結果がどう変わるか**を確かめる
- 3つのモデルを同時に載せても、GPUメモリの使用量は10GB程度であり、T4（16GB）に収まる
  - メモリが足りない環境では、次のセルと、3Bモデルを使うセル（先頭のコメントに「3Bモデル」または「自己選好の再検証」とあるコードセル）を飛ばしても、残りのセルは動く
- 2026年時点で、3Bモデルは研究目的の利用に限る独自のライセンスで公開されている（ほかの2つはApache 2.0）
  - 利用の前に、モデルカードでライセンスを確認する

In [ ]:
# 比較用に、さらに大きい公開モデルを読み込む
LLM_XL = load_llm("Qwen/Qwen2.5-3B-Instruct")  # 3Bモデル（約30億パラメータ）
if device.type == "cuda":
    print(f"GPUメモリの使用量: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 推論の補助関数

以降の実習で繰り返し使う2つの関数を定義する

- `chat`: プロンプトを渡して応答文を得る
  - 複数のプロンプトをまとめて処理し、評価にかかる時間を短くする
  - `temperature=0` のときは、毎回最も確率の高いトークンを選ぶ（貪欲法）ため、出力は決定的になる
- `next_token_logprobs`: プロンプトの直後に、指定したトークンが来る対数確率を得る
  - 文章を生成させずに、モデルが「A」と「B」のどちらを選びたがっているかを直接読み取れる
  - 選択式ベンチマークの採点と、ペア比較の採点で使う

`USAGE` には消費したトークン数を積算する
APIを使う場合はこれが料金に直結するため、評価のコストの目安として記録しておく

In [ ]:
# 対話形式のプロンプトを作り、まとめて生成する関数
USAGE = {"prompt": 0, "completion": 0}  # 消費したトークン数の累計（コストの目安）

def build_prompt(llm, user, system=None):
    msgs = [{"role": "system", "content": system}] if system else []
    # user が文字列なら1回の発話、リストなら複数ターンの対話履歴として扱う
    msgs += user if isinstance(user, list) else [{"role": "user", "content": user}]
    return llm["tok"].apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def chat(llm, users, system=None, max_new_tokens=64, temperature=0.0, batch_size=16):
    """users（文字列またはそのリスト）への応答を返す（temperature=0 のときは貪欲法で決定的に生成する）"""
    single = isinstance(users, str)
    users = [users] if single else list(users)
    tok, model, outs = llm["tok"], llm["model"], []
    for i in range(0, len(users), batch_size):
        texts = [build_prompt(llm, u, system) for u in users[i:i + batch_size]]
        enc = tok(texts, return_tensors="pt", padding=True).to(device)
        sampling = dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if temperature > 0 else dict(do_sample=False)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
        new = gen[:, enc.input_ids.shape[1]:]
        USAGE["prompt"] += int(enc.attention_mask.sum())
        USAGE["completion"] += int((new != tok.pad_token_id).sum())
        outs += [t.strip() for t in tok.batch_decode(new, skip_special_tokens=True)]
    return outs[0] if single else outs

@torch.no_grad()
def next_token_logprobs(llm, users, candidates, system=None, batch_size=16):
    """各プロンプトの直後に candidates の各トークンが来る対数確率を返す（形は [プロンプト数, 候補数]）"""
    tok, model = llm["tok"], llm["model"]
    ids = [tok.encode(c, add_special_tokens=False) for c in candidates]
    assert all(len(x) == 1 for x in ids), "候補は1トークンで表せるものに限る"
    ids = torch.tensor([x[0] for x in ids], device=device)
    rows = []
    for i in range(0, len(users), batch_size):
        texts = [build_prompt(llm, u, system) for u in users[i:i + batch_size]]
        enc = tok(texts, return_tensors="pt", padding=True).to(device)
        logits = model(**enc).logits[:, -1, :].float()  # 左側を埋めたので、最後の位置が次トークンの予測になる
        rows.append(torch.log_softmax(logits, dim=-1)[:, ids].cpu())
    return torch.cat(rows).numpy()

---

# なぜLLMの評価は難しいか

分類モデルの評価と比べて、LLMの評価には4つの難しさがある

| 難しさ | 内容 | 本テキストでの対処 |
| --- | --- | --- |
| **正解が1つでない** | 同じ内容を表す文が無数にあり、文字列の一致では正誤を決められない | 自動指標の限界を知り、LLMによる採点を併用する |
| **出力が確率的** | サンプリングにより、実行のたびに出力が変わる | 複数回実行し、ばらつきごと報告する |
| **プロンプトに敏感** | 指示の言い回しや選択肢の順序で成績が変わる | 複数の書き方で測り、差を誤差と比べる |
| **評価データの汚染** | 評価問題が学習データに混入していると、暗記で解けてしまう | 自作の問題、新しい問題、非公開の問題を使う |

まず、最初の2つを実際に観察する

In [ ]:
# 同じ質問を5回サンプリングして、出力のばらつきを見る
q = "機械学習における過学習とは何か\n1文で答えよ"
torch.manual_seed(SEED)
samples = chat(LLM_L, [q] * 5, temperature=0.7, max_new_tokens=80)
for i, s in enumerate(samples):
    print(f"[{i}] {s}")

# 貪欲法（temperature=0）では、2回実行しても同じ出力になる
g1, g2 = chat(LLM_L, q, max_new_tokens=80), chat(LLM_L, q, max_new_tokens=80)
print("\n貪欲法の出力:", g1)
print("貪欲法で2回の出力が一致:", g1 == g2)

## 観察1: 正解が1つでなく、出力は毎回変わる

- 5つの出力は文面がすべて異なる
  - 内容がおおむね正しいものも、怪しいものも混ざる
  - 小型の多言語モデルでは、途中から別の言語に切り替わる出力も現れる
  - 「訓練データに過剰に適合し、未知のデータで性能が下がること」という模範解答を用意しても、文字列として一致する出力は1つも無い
- `temperature=0` にすれば出力は固定できるが、これは「たまたま選ばれた1つの出力」を固定しただけである
  - 実際の利用でサンプリングを使うなら、評価もサンプリングした出力に対して行い、複数回の結果を集計する必要がある

したがって、LLMの評価では次の2点を常に意識する

1. **採点器**: 文面が違っても内容が正しければ正解とできるか
2. **試行回数**: 1回の出力ではなく、分布として性能を捉えているか

## 観察2: プロンプトへの敏感さ

プロンプトの言い回しによる成績の変化は、評価データがあって初めて数値で示せる

この点は、次の節で評価データを作ったあとに実験する

## 評価データの汚染

**汚染（contamination）**とは、評価に使う問題と正解が、モデルの学習データに含まれてしまうことである

- 公開されたベンチマークは、Web上に問題と解答が置かれる
- LLMの学習データはWebから大量に収集されるため、公開から時間が経ったベンチマークは学習データに混入しやすい
- 混入した問題に正解しても、それは推論の能力ではなく暗記の結果かもしれない

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-contamination.png" width=650>

上の図の赤い矢印が、公開された評価データが学習データに入り込む経路である

これは `mlsys-text-2-ML基礎.ipynb` で学んだ「テストデータを学習に使ってはならない」という原則の破れと同じである
ただしLLMでは、学習データが巨大で中身を確認できないため、破れているかどうかを利用者が確かめにくい

対処として、次の方法が使われる

- **自分で問題を作る**: 本テキストの実習はすべてこの方法である
- **プログラムで問題を生成する**: 数値や組合せを毎回変えれば、暗記では解けない
- **モデルの学習より後に作られたデータを使う**: 新しいニュース、新しい論文、新しいコードなど
- **評価データを公開しない**: 公開ベンチマークの一部を非公開にして、主催者だけが採点する

---

# 評価データの作り方

評価データは「何を測りたいか」から逆算して設計する

## 設計の手順

1. **測りたい能力を1文で書く**
   - 悪い例: 「賢さを測る」
   - 良い例: 「3桁までの四則演算を、数値だけで正しく答えられるか」
2. **入力の分布を決める**
   - 実際の利用で来る入力に近づける
   - 利用者の質問の記録があれば、そこから抜き出すのが最もよい
3. **層別（stratification）する**
   - 入力を種類や難しさで層に分け、各層に十分な件数を割り当てる
   - 平均点だけでなく、層ごとの点数を見られるようにする
4. **正解と採点方法を決める**
   - 採点が機械的にできる形（数値、選択肢、短い語句）にできるなら、そうする
5. **件数を決める**
   - 見分けたい差の大きさから、必要な件数を見積もる（この節の後半で扱う）

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-stratified.png" width=650>

上の図は、層別した評価データの模式図である
層ごとの正解率が同じでも、各層に何件を割り当てるかによって、全体の点数は変わる

## 失敗例から作る

実運用では、評価データの最も良い供給源は**実際に起きた失敗**である

- 利用者から報告された誤答、開発中に見つけた誤答を、入力と期待する出力の組として保存する
- 1件直すたびに、その事例を評価データに加える
- ソフトウェア開発で、バグを直すたびに再発防止の試験を追加するのと同じ考え方である

ただし、失敗例だけで作った評価データには偏りがある

- あるモデルが間違えた問題だけを集めると、そのモデルの正解率は定義上0%に近くなる
- この集合で別のモデルと比べると、元のモデルが不当に低く見える
- 失敗例は**回帰試験**（直したはずの誤りが再発していないかの確認）に使い、モデル間の比較には、失敗とは無関係に集めた集合を使う

## 実習用の評価データ

ここでは、整数の計算問題をプログラムで生成する

- 測りたい能力: 3桁までの加算、減算、2桁までの乗算を、数値だけで正しく答えられるか
- 層: 演算の種類と桁数で5つに分ける
- 件数: 各層40問、合計200問
- 正解: プログラムで計算できる
- 汚染: 乱数で生成するため、特定の問題が学習データに入っている心配が小さい

In [ ]:
# 計算問題をプログラムで生成する（5つの層、各40問）
def make_arith(n_per=40, seed=SEED):
    rng = random.Random(seed)
    spec = {
        "add2":   lambda: (rng.randint(10, 99), "+", rng.randint(10, 99)),
        "add3":   lambda: (rng.randint(100, 999), "+", rng.randint(100, 999)),
        "sub3":   lambda: (rng.randint(500, 999), "-", rng.randint(100, 499)),
        "mul1x2": lambda: (rng.randint(2, 9), "×", rng.randint(11, 99)),
        "mul2x2": lambda: (rng.randint(11, 99), "×", rng.randint(11, 99)),
    }
    fn = {"+": operator.add, "-": operator.sub, "×": operator.mul}
    data = []
    for kind, gen in spec.items():
        for _ in range(n_per):
            a, op, b = gen()
            data.append({"id": f"{kind}-{len(data):03d}", "kind": kind, "expr": f"{a} {op} {b}", "answer": fn[op](a, b)})
    return data

ARITH = make_arith()
print(len(ARITH), "問")
for d in ARITH[::40]:
    print(d)

## 採点方法を決める

モデルの出力は文章であるため、そこから数値を取り出す規則が要る

- 出力に含まれる整数のうち、**最後のもの**を答えとみなす
  - 「37 × 48 = 1776」のように式を書き直してから答える出力に対応するためである
- 桁区切りのカンマは取り除く

この規則も評価の一部である
規則が変われば点数も変わるため、評価結果を報告するときは、取り出しの規則まで書く

In [ ]:
# 計算問題を解かせて採点する
PROMPT_V1 = "{expr} を計算し、答えの数値だけを書け"

def extract_int(text):
    """出力に含まれる最後の整数を取り出す（見つからなければ None）"""
    nums = re.findall(r"-?\d[\d,]*", text)
    return int(nums[-1].replace(",", "")) if nums else None

assert extract_int("37 × 48 = 1,776 です") == 1776 and extract_int("わからない") is None

def run_arith(llm, template=PROMPT_V1, data=ARITH):
    outs = chat(llm, [template.format(expr=d["expr"]) for d in data], max_new_tokens=16, batch_size=32)
    df = pd.DataFrame(data)
    df["output"] = outs
    df["correct"] = [extract_int(o) == d["answer"] for o, d in zip(outs, data)]
    return df

t0 = time.time()
res_L = run_arith(LLM_L)
res_S = run_arith(LLM_S)
print(f"実行時間 {time.time() - t0:.1f} 秒")
print(f"大モデルの正解率: {res_L.correct.mean():.3f}")
print(f"小モデルの正解率: {res_S.correct.mean():.3f}")
print(res_L.sample(5, random_state=SEED)[["expr", "answer", "output", "correct"]].to_string(index=False))

## 層ごとに見る

全体の正解率だけでは、どこが弱いのかがわからない

層ごとの正解率を並べる

In [ ]:
# 層ごとの正解率を表にする
by_kind = pd.DataFrame({
    "large": res_L.groupby("kind", sort=False).correct.mean(),
    "small": res_S.groupby("kind", sort=False).correct.mean(),
})
by_kind.loc["all"] = [res_L.correct.mean(), res_S.correct.mean()]
print(by_kind.round(3))

ax = by_kind.plot.bar(figsize=(7, 3.2), rot=0)
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Accuracy by stratum")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

### 結果の読み方

- 2桁の加算（`add2`）は、どちらのモデルもほぼ全問正解する
  - この層だけで評価すると、2つのモデルに差が無いように見える
  - 簡単すぎる問題は、モデルを見分ける役に立たない
- 2桁どうしの乗算（`mul2x2`）は、どちらのモデルも半分ほどしか解けず、やはり差が無い
- 2つのモデルの差は、ほとんどが**3桁の加算（`add3`）の層だけ**から生じている
  - 全体の正解率だけを見ると「大モデルは計算全般に強い」と読んでしまう
  - 層ごとに見ると、「小モデルは3桁の加算（繰り上がりの連続）に固有の弱点がある」という、より具体的な事実がわかる
- 乗算の層では、小モデルのほうがわずかに高い
  - 40問中1〜2問の差であり、誤差の範囲である（誤差の見積もり方は、この節の後半で扱う）
  - 「小モデルのほうが乗算に強い」と読んではならない

全体の正解率は、各層の件数の配分で決まる
`add3` の層を増やせば2つのモデルの差は広がり、減らせば縮まる

**全体の点数は評価データの設計者が決めた配分の結果であり、絶対的な意味は無い**
意味を持つのは、同じ評価データの上での比較と、層ごとの内訳である

## モデルを大きくすると成績は上がるか

3Bモデルにも同じ200問を解かせる

パラメータ数は、小モデルの約6倍、大モデルの約2倍である

実行する前に、3つのモデルの順位を予想してみよ

In [ ]:
# 3Bモデルにも同じ200問を解かせ、3つのモデルを層ごとに比べる
res_XL = run_arith(LLM_XL)
by_kind3 = pd.DataFrame({name: r.groupby("kind", sort=False).correct.mean()
                         for name, r in [("small(0.5B)", res_S), ("large(1.5B)", res_L), ("3B", res_XL)]})
by_kind3.loc["all"] = [res_S.correct.mean(), res_L.correct.mean(), res_XL.correct.mean()]
print(by_kind3.round(3))

ax = by_kind3.plot.bar(figsize=(7.5, 3.2), rot=0)
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Accuracy by stratum (three model sizes)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

# 3Bモデルが2桁の加算で間違えた問題を見る
bad = res_XL[(res_XL.kind == "add2") & ~res_XL.correct]
print(f"\n3Bモデルが add2 で間違えた問題: {len(bad)}問")
print(bad.head(4)[["expr", "answer", "output"]].to_string(index=False))

### 結果の読み方

- 筆者の実行では、3Bモデルの全体の正解率は0.73であり、大モデル（0.85）を下回り、小モデル（0.75）と同程度であった
- 層ごとに見ると、最も易しいはずの2桁の加算（`add2`）で、最も大きいモデルが最も悪い
  - 誤答を読むと、「59 + 63」に「222」と答えるというように、先頭の桁を誤っている
  - 乗算の層では、3つのモデルにほとんど差が無い
- 「大きいモデルのほうが良い」という予想は、この評価データとこのプロンプトでは外れた

### 本来はどうなるはずか、なぜここでは違うのか

- **一般に言われている傾向**
  - 同じ系列で同じ方法により学習されたモデルどうしでは、パラメータ数の多いモデルのほうが、幅広い課題で成績が高い
  - この系列の公開ベンチマークの平均点も、大きいモデルほど高いと報告されている
  - したがって、本来は「3B ≧ 1.5B ≧ 0.5B」の順になると期待される
- **ここで逆になった理由として考えられること**
  - **プロンプトの形式**: このプロンプトは、途中の式を書かせず、数値だけをいきなり答えさせている
    - 計算のように何段階かの処理が要る課題では、途中経過を書かせたほうが正確になることが知られている（Wei et al., 2022）
    - 3Bモデルは、この制約の影響を特に強く受けた可能性がある
  - **1つのプロンプトしか試していない**: 平均的に優れていることと、特定の言い回しで優れていることは別である
  - **課題が狭い**: 整数の計算という1種類の課題であり、モデルの総合的な能力の一部しか見ていない
- **断定できないこと**
  - 3Bモデルが数値だけを書くときに、なぜ先頭の桁を誤るのかという内部の理由は、この実験からは断定できない
  - なお、半精度での計算や、まとめて生成する処理が原因ではないことは、筆者が別に確かめた（単精度で1問ずつ生成しても同じ誤答になる）

「3Bモデルは計算が苦手である」と結論するのは早い

プロンプトの形式が原因であるという見立ては、この節の後半（プロンプトへの敏感さ）で、言い回しを変えて確かめる

**パラメータ数や公開ベンチマークの順位から、自分の課題での優劣を決めてはならない**
自分の評価データで測って初めてわかる

## 規模と統計的な誤差

評価データは、起こりうる入力全体からの**標本**である
200問での正解率は、別の200問を引けば別の値になる

したがって、正解率は1つの数値ではなく、**誤差の幅を持った推定値**として扱う

### ブートストラップ法

**ブートストラップ法（bootstrap）**は、手元の標本から誤差の幅を見積もる汎用的な方法である

1. $n$問の採点結果（正解なら1、不正解なら0）から、重複を許して$n$個を無作為に取り出す
2. 取り出した$n$個の平均（正解率）を計算する
3. 1と2を数千回繰り返し、得られた正解率の分布を作る
4. 分布の下側2.5%点と上側2.5%点を、**95%信頼区間（confidence interval）**とする

「別の評価データを引いたら正解率がどれだけ動くか」を、手元のデータの引き直しで模擬していることになる

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-bootstrap.png" width=700>

上の図では、左の採点結果から引き直した列（中央）ごとに平均を求め、その分布（右）の中央95%を信頼区間としている

数式を導かなくても、平均以外の指標（F1、順位の指標など）にそのまま使えるのが利点である

In [ ]:
# ブートストラップ法で正解率の95%信頼区間を求める
def bootstrap_ci(x, n_boot=5000, alpha=0.05, seed=SEED):
    """x（事例ごとの点数）の平均と、その (1-alpha) 信頼区間を返す"""
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))  # 重複を許した引き直し
    means = x[idx].mean(axis=1)
    return x.mean(), np.quantile(means, alpha / 2), np.quantile(means, 1 - alpha / 2)

for name, res in [("大モデル", res_L), ("小モデル", res_S)]:
    m, lo, hi = bootstrap_ci(res.correct)
    print(f"{name}: 正解率 {m:.3f}  95%信頼区間 [{lo:.3f}, {hi:.3f}]  幅 ±{(hi - lo) / 2:.3f}")

### 結果の読み方

- 200問では、信頼区間の幅はおよそ±5〜6ポイントになる
- 「正解率は○○%だった」と1つの数値で報告すると、この幅が見えなくなる

正解率$p$、問題数$n$のとき、正解率の標準誤差は次の式で近似できる

$$\mathrm{SE} = \sqrt{\frac{p(1-p)}{n}}$$

95%信頼区間はおよそ $p \pm 1.96\,\mathrm{SE}$ である

- $p=0.75$、$n=200$ なら $\mathrm{SE} \approx 0.031$、信頼区間は±6ポイント程度となり、ブートストラップ法の結果と合う
- 幅は $1/\sqrt{n}$ に比例するため、幅を半分にするには問題数を4倍にしなければならない

ブートストラップ法にも、この近似式にも、苦手な場面がある

- 件数が少なく、正解率が0%や100%に近い場合である
  - 10問すべて正解のとき、引き直しても常に全問正解になり、区間の幅が0になってしまう
- このような場合は、ウィルソンの区間（Wilson score interval）を使う
  - `mlsys-text-T-安全性と公平性.ipynb` で、レッドチーミングの違反率（めったに起きない事象の割合）を見積もる場面で扱っている

## 2つのモデルの差は本物か

次に、2つのモデルの**差**の信頼区間を求める

ここで重要なのは、2つのモデルが**同じ問題**を解いていることである

- 問題ごとに「大モデルの点 − 小モデルの点」を計算する（値は +1、0、−1 のいずれか）
- この差の列に対してブートストラップ法を使う
- これを**対応のあるブートストラップ（paired bootstrap）**と呼ぶ

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-paired.png" width=680>

上の図では、問題ごとに2つのモデルの点を縦に並べ、その差（下の行）だけを取り出している

問題ごとの差を取ると、「この問題はどちらのモデルにも難しい」という問題側のばらつきが打ち消される
そのため、2つの正解率の信頼区間を別々に求めて比べるよりも、小さな差を検出しやすい

差の信頼区間が0をまたがなければ、差は偶然では説明しにくいと判断する

In [ ]:
# 対応のあるブートストラップで、2つのモデルの差の信頼区間を求める
def paired_diff_ci(a, b, **kw):
    """同じ問題に対する点数 a, b の差（a - b）の平均と信頼区間を返す"""
    return bootstrap_ci(np.asarray(a, dtype=float) - np.asarray(b, dtype=float), **kw)

d, lo, hi = paired_diff_ci(res_L.correct, res_S.correct)
print(f"差（大 − 小）: {d:+.3f}  95%信頼区間 [{lo:+.3f}, {hi:+.3f}]")
print("判断:", "差がある" if lo > 0 or hi < 0 else "差は誤差の範囲")

# 引き直しで得られる差の分布を描く
rng = np.random.default_rng(SEED)
diff = res_L.correct.to_numpy(float) - res_S.correct.to_numpy(float)
boot = diff[rng.integers(0, len(diff), size=(5000, len(diff)))].mean(axis=1)
plt.figure(figsize=(6, 3))
plt.hist(boot, bins=np.arange(boot.min(), boot.max() + 0.01, 0.005) - 0.0025, color="tab:blue", alpha=0.8)  # 差は0.005刻みの値しか取らない
plt.axvline(0, color="k", linestyle="--", label="no difference")
plt.axvline(lo, color="tab:red", label="95% CI")
plt.axvline(hi, color="tab:red")
plt.xlabel("accuracy difference (large - small)")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()

### 「2点差」は誤差か

論文や製品の発表では、「従来より2ポイント向上した」という主張をよく見る

2ポイントの差が意味を持つかどうかは、評価データの件数で決まる

次のセルでは、**真の実力がまったく同じ2つのモデル**を仮想的に作り、$n$問で比べたときに、偶然だけで2ポイント以上の差がつく確率を計算する

- 2つのモデルは、どの問題にも確率0.7で正解する
- 2つのモデルの正誤は、問題ごとに独立とする

In [ ]:
# 実力が同じ2つのモデルで、偶然に2ポイント以上の差がつく確率を調べる
rng = np.random.default_rng(SEED)
p_true, trials = 0.7, 20000
rows = []
for n in [50, 100, 200, 500, 1000, 2000, 5000, 10000]:
    a = rng.binomial(n, p_true, size=trials) / n  # モデルAの正解率（trials 回分）
    b = rng.binomial(n, p_true, size=trials) / n  # モデルBの正解率
    rows.append({"n": n,
                 "half_width_95": 1.96 * math.sqrt(p_true * (1 - p_true) / n),
                 "P(|diff|>=2pt)": np.mean(np.abs(a - b) >= 0.02 - 1e-9)})
tab = pd.DataFrame(rows)
print(tab.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(tab.n, tab.half_width_95 * 100, "o-")
ax[0].set_xscale("log"); ax[0].set_xlabel("number of questions n"); ax[0].set_ylabel("95% CI half width (points)")
ax[1].plot(tab.n, tab["P(|diff|>=2pt)"], "o-", color="tab:red")
ax[1].set_xscale("log"); ax[1].set_xlabel("number of questions n"); ax[1].set_ylabel("P(|diff| >= 2 points)")
plt.tight_layout()
plt.show()

### 結果の読み方

- 100問では、実力が同じでも、偶然だけで2ポイント以上の差がつく確率が8割近くある
  - 100問で比べた「2点差」は、何も示していない
- 1000問でも、その確率は3割を超える
- 2ポイントの差を確信を持って言うには、数千問が要る

目安として、互いに独立な2つの正解率の差の標準誤差は $\sqrt{2p(1-p)/n}$ である
$p=0.7$ のとき、差2ポイントが1.96倍の標準誤差を超えるには、$n \approx 4000$ が必要となる

同じ問題で比べる（対応のある比較）と、必要な件数はこれより少なくて済む
2つのモデルが同じ問題で同じように正解・不正解するほど、差のばらつきが小さくなるからである

**実務での判断基準**

- 評価結果を見たら、まず件数を確認する
- 差を主張するときは、差の信頼区間を付ける
- 数十問の評価データは、大きな差（数十ポイント）の検出と、個別の失敗例の観察に使う

## プロンプトへの敏感さを測る

評価データと誤差の見積もりがそろったので、保留していた「プロンプトへの敏感さ」を測る

同じ計算問題を、指示の言い回しだけを変えた4つのプロンプトで大モデルに解かせる

内容はどれも同じであり、人間なら成績は変わらないはずである

In [ ]:
# 指示の言い回しを変えた4つのプロンプトで、大モデルの正解率を比べる
TEMPLATES = {
    "v1": PROMPT_V1,
    "v2": "次の計算の答えを数値だけで答えよ: {expr}",
    "v3": "{expr} = ?",
    "v4": "お手数ですが、{expr} の計算結果を教えていただけますか",
}
res_tmpl = {"v1": res_L}
for k in ["v2", "v3", "v4"]:
    res_tmpl[k] = run_arith(LLM_L, TEMPLATES[k])

for k, res in res_tmpl.items():
    m, lo, hi = bootstrap_ci(res.correct)
    d, dlo, dhi = paired_diff_ci(res.correct, res_L.correct)
    print(f"{k}: 正解率 {m:.3f} [{lo:.3f}, {hi:.3f}]   v1との差 {d:+.3f} [{dlo:+.3f}, {dhi:+.3f}]   {TEMPLATES[k]}")

# v1では正解し、v3では不正解になった問題の出力を見る
lost = res_L.correct & ~res_tmpl["v3"].correct
print(f"\nv1で正解、v3で不正解: {int(lost.sum())}問")
print(res_tmpl["v3"][lost].head(4)[["expr", "answer", "output"]].to_string(index=False))

### 結果の読み方

- 計算の内容は同じでも、言い回しによって正解率が動く
  - 筆者の実行では、最も高いv1と最も低いv3の間に18ポイントの開きがあった
  - これは、大モデルと小モデルの差（10ポイント）よりも大きい
- v1との差の信頼区間が0をまたがないプロンプトは、偶然ではなく、言い回しの影響で点が変わっている
  - 信頼区間の端がちょうど0にかかるもの（筆者の実行ではv2）は、差があるとも無いとも言い切れない
- v3で不正解になった出力を読むと、原因がわかる
  - 「数値だけを書け」という指示が無くなったため、モデルは説明の文章を書き始める
  - 答えの数値に届く前に、16トークンの上限で出力が切れている
  - これは「計算の能力」ではなく「出力の形式」の違いだが、点数には同じように現れる

ここから、評価結果の報告について2つの教訓が得られる

1. **モデルの点数は、プロンプトと取り出しの規則を含めた「評価の手順全体」に対する点数である**
   - 「モデルAの正解率は○○%」という数値は、プロンプトを明記しなければ再現できない
2. **モデルを比べるときは、複数の言い回しで測る**
   - 1つのプロンプトだけで比べると、たまたまそのプロンプトと相性の良いモデルが勝つ
   - 言い回しによる変動幅を、モデル間の差と並べて報告する

## モデルを大きくすると敏感さは減るか

大きいモデルは、言い回しの違いに左右されにくいと期待したくなる

3Bモデルでも同じ4つのプロンプトを試し、変動幅を比べる

In [ ]:
# 3Bモデルでも4つのプロンプトを試し、言い回しによる変動幅を大モデルと比べる
res_tmpl_XL = {"v1": res_XL}
for k in ["v2", "v3", "v4"]:
    res_tmpl_XL[k] = run_arith(LLM_XL, TEMPLATES[k])

sens = pd.DataFrame({"large(1.5B)": {k: r.correct.mean() for k, r in res_tmpl.items()},
                     "3B": {k: r.correct.mean() for k, r in res_tmpl_XL.items()}})
print(sens.round(3))
print("\n各プロンプトでの差（3B − 大）と95%信頼区間")
for k in TEMPLATES:
    d, lo, hi = paired_diff_ci(res_tmpl_XL[k].correct, res_tmpl[k].correct)
    print(f"  {k}: {d:+.3f} [{lo:+.3f}, {hi:+.3f}]")
print("\n変動幅（最大 − 最小）")
print((sens.max() - sens.min()).round(3).to_string())
print("\n3Bモデルのv3の出力例:", [o for o in res_tmpl_XL["v3"].output[:3]])

### 結果の読み方

- 筆者の実行では、変動幅は大モデルが18ポイント、3Bモデルが34ポイントであった
  - モデルを大きくしても、プロンプトへの敏感さは減っていない
- **どのプロンプトで比べるかによって、2つのモデルの順位が入れ替わる**
  - v1とv4では、大モデルが3Bモデルを上回る
  - v2とv3では、3Bモデルが大モデルを上回る
  - v1だけで比べた人は「1.5Bのほうが計算に強い」と報告し、v2だけで比べた人は「3Bのほうが強い」と報告することになる
- 3Bモデルのv3の出力を見ると、「59 + 63 equals 122.」のように式を書き直してから答えており、正しく計算できている
  - 同じ問題に、v1（数値だけを書け）では「222」と答えていた
  - このモデルは、式を書き直す余地があれば正しく計算でき、数値だけをいきなり書かされると誤る
  - 大モデルは逆に、数値だけを書かせたほうが成績が良い

先ほどの「3Bモデルは計算が苦手」という読みは誤りで、正しくは「3Bモデルは、v1の言い回しと相性が悪い」である

各プロンプトでの差の信頼区間は、4つとも0をまたいでいない
順位の入れ替わりは、200問という件数による偶然ではなく、プロンプトの違いによるものである

### 本来はどうなるはずか、なぜここでは違うのか

- **素朴な期待**: 大きいモデルほど指示をよく理解するので、言い回しの違いに左右されにくくなる
- **文献が示していること**: この期待は、実際には支持されていない
  - プロンプトの書式を変えるだけで成績が数十ポイント動くこと、その敏感さはモデルを大きくしても、例示を増やしても無くならないことが報告されている（Sclar et al., 2024）
  - 今回の「モデルを大きくしても敏感さが減らない」という結果は、想定外に見えるが、文献とは合っている
- **順位が入れ替わった理由**
  - v2とv3では、3Bモデルは式を書き直してから答えており、正解が多い
  - v1では数値だけをいきなり書いて誤り、v4では丁寧な前置きの文が長くなって、答えに届く前に出力が切れることが多い
  - 途中の式を書けるかどうかで、3Bモデルの正解率が大きく変わっている
  - 1.5Bは、式を書き直すと出力が長くなって16トークンの上限に掛かりやすく、数値だけを書く形式のほうが成績が良い
  - つまり、順位の入れ替わりは、計算の能力の差というより、**各モデルの答え方の癖と、出力の上限や取り出しの規則との噛み合わせ**で生じている
- **この実験から言えないこと**
  - 4つのプロンプト、2つのモデル、1種類の課題だけの結果であり、「大きいモデルほど敏感である」と一般化することはできない

モデルを比べるときに複数の言い回しで測る必要があることを、この結果はよく示している

In [ ]:
# 失敗例を取り出して、回帰試験用の集合として保存する
fails = res_L[~res_L.correct]
print("大モデルが間違えた問題:", len(fails), "問")
print(fails.kind.value_counts().to_string())
print(fails.head(5)[["expr", "answer", "output"]].to_string(index=False))

HARD_SET = fails[["id", "kind", "expr", "answer"]].to_dict("records")  # 改良後に再び解かせるための集合

### 失敗例の使い方

- 失敗した問題を保存しておけば、プロンプトやモデルを変えたあとに「前に間違えた問題が解けるようになったか」を確かめられる
- 失敗が特定の層に集中していれば、その層の問題を増やして詳しく調べる
- 先に述べたとおり、この集合は大モデルの失敗だけを集めたものであり、モデル間の比較には使わない

---

# 自動指標

**自動指標（automatic metric）**は、モデルの出力と模範解答（参照文）をプログラムで比べて点数を出す方法である

速く、安く、何度実行しても同じ結果になるため、開発中に繰り返し回す評価に向く

一方で、どの指標も「文字列が似ているか」「文として自然か」といった表面の性質を測っているにすぎず、内容の正しさを直接測ってはいない

この節では6つの指標を実装し、人間の判断とずれる例を具体的に確かめる

| 指標 | 測っているもの | 参照文 |
| --- | --- | --- |
| **完全一致（Exact Match, EM）** | 正規化した文字列が参照文と同一か | 必要 |
| **F1** | 出力と参照文で共通する文字（または単語）の割合 | 必要 |
| **BLEU** | 出力のn-gramが参照文に現れる割合（適合率に基づく） | 必要 |
| **ROUGE-L** | 出力と参照文の最長共通部分列の長さ | 必要 |
| **埋め込み類似度** | 文の埋め込みベクトルどうしのコサイン類似度 | 必要 |
| **パープレキシティ（Perplexity, PPL）** | 言語モデルから見た文の「意外さ」 | 不要 |

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-metric-views.png" width=720>

上の図は、3種類の指標が回答の何を見ているかを並べたものである
左は文字の一致、中央はベクトルの向き、右は各トークンの確率であり、どれも内容の正誤を直接は見ていない

## 文字列の重なりに基づく指標

- **完全一致**は、質問応答の評価で古くから使われる
  - 全角と半角、空白、句読点の違いで不一致にならないよう、比較の前に**正規化**する
- **F1**は、出力と参照文を要素の集まりとみなし、適合率と再現率の調和平均を取る
  - 適合率: 出力の要素のうち、参照文にも含まれる割合
  - 再現率: 参照文の要素のうち、出力にも含まれる割合
  - 英語では単語を要素にするが、日本語は単語の境目に空白が無いため、ここでは**文字**を要素にする
  - 形態素解析器で単語に分けてから計算する方法もある
- **BLEU**は `mlsys-text-J-NLP-4-Transformer-full.ipynb` で扱った
  - ここでは `sacrebleu` の文字単位の分割（`tokenize="char"`）を使う
- **ROUGE-L**は要約の評価で使われ、**最長共通部分列（Longest Common Subsequence, LCS）**の長さから適合率と再現率を求める
  - 部分列は連続していなくてもよいため、語順が保たれていれば、間に別の語が挟まっても点が入る

In [ ]:
# 文字列の重なりに基づく指標を実装する
import sacrebleu

def normalize(s):
    """全角と半角をそろえ、小文字にし、空白と句読点を取り除く"""
    s = unicodedata.normalize("NFKC", s).lower()
    return re.sub(r"[\s、。,.!?「」『』()]", "", s)

def exact_match(pred, ref):
    return float(normalize(pred) == normalize(ref))

def char_f1(pred, ref):
    p, r = Counter(normalize(pred)), Counter(normalize(ref))
    common = sum((p & r).values())  # 両方に現れる文字の数（重複は少ない方まで数える）
    if common == 0:
        return 0.0
    prec, rec = common / sum(p.values()), common / sum(r.values())
    return 2 * prec * rec / (prec + rec)

def lcs_len(a, b):
    """最長共通部分列の長さを動的計画法で求める"""
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            dp[i + 1][j + 1] = dp[i][j] + 1 if x == y else max(dp[i][j + 1], dp[i + 1][j])
    return dp[-1][-1]

def rouge_l(pred, ref):
    p, r = normalize(pred), normalize(ref)
    l = lcs_len(p, r)
    if l == 0:
        return 0.0
    prec, rec = l / len(p), l / len(r)
    return 2 * prec * rec / (prec + rec)

def bleu(pred, ref):
    return sacrebleu.sentence_bleu(pred, [ref], tokenize="char").score / 100

# 動作確認
assert exact_match("３７７６ メートル。", "3776メートル") == 1.0
assert lcs_len("abcde", "axcxe") == 3
assert abs(char_f1("abc", "abd") - 2 / 3) < 1e-9
print("EM:", exact_match("東京", "東京都"), " F1:", round(char_f1("東京", "東京都"), 3),
      " ROUGE-L:", round(rouge_l("東京", "東京都"), 3), " BLEU:", round(bleu("東京", "東京都"), 3))

## 埋め込み類似度

文字列の重なりは、言い換えに弱い
「沸点」と「沸騰する温度」は同じ意味だが、共通する文字は少ない

そこで、文を**埋め込みベクトル**に変換し、ベクトルどうしのコサイン類似度で比べる方法が使われる

- 埋め込みモデルは、意味の近い文が近いベクトルになるように学習されている
- 言い換えを同じものとして扱えることが期待できる
- 学習済みモデルの内部表現を使ってトークンごとに照合する**BERTScore**も、同じ発想の指標である

ここでは、多言語の小型埋め込みモデル `intfloat/multilingual-e5-small` を使う

In [ ]:
# 埋め込みモデルを読み込み、文をベクトルに変換する関数を定義する
EMB_NAME = "intfloat/multilingual-e5-small"
emb_tok = AutoTokenizer.from_pretrained(EMB_NAME)
emb_model = AutoModel.from_pretrained(EMB_NAME).to(device).eval()

@torch.no_grad()
def embed(texts, prefix="query: "):
    """文のリストを、長さ1に正規化した埋め込みベクトル（numpy配列）に変換する"""
    # このモデルは、文の先頭に "query: " または "passage: " を付けて使う約束になっている
    enc = emb_tok([prefix + t for t in texts], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    h = emb_model(**enc).last_hidden_state
    mask = enc.attention_mask.unsqueeze(-1)
    v = (h * mask).sum(dim=1) / mask.sum(dim=1)  # 埋め草を除いた平均
    return torch.nn.functional.normalize(v, dim=-1).cpu().numpy()

def emb_sim(pred, ref):
    v = embed([pred, ref])
    return float(v[0] @ v[1])

print("言い換え:", round(emb_sim("水の沸点は100℃になる", "水は摂氏100度で沸騰する"), 3))
print("無関係  :", round(emb_sim("今日の昼食はカレーにする", "水は摂氏100度で沸騰する"), 3))

### 結果の読み方

- 言い換えの組は、無関係な組よりも類似度が高い
- ただし、無関係な組でも類似度は0.8前後ある
  - このモデルのコサイン類似度は、狭い範囲（およそ0.7〜1.0）に集まる性質がある
  - 類似度は**絶対値ではなく、同じモデルの中での大小**で読む
  - 「0.8以上なら正解」のような閾値を、モデルを確かめずに決めてはならない

## パープレキシティ

**パープレキシティ（perplexity）**は、言語モデルがその文をどれだけ「予想どおり」と感じるかを表す

文をトークン列 $x_1, \dots, x_T$ とすると、次の式で定義される

$$\mathrm{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^{T}\log p(x_t \mid x_1, \dots, x_{t-1})\right)$$

- 各トークンの対数確率の平均を取り、符号を反転して指数関数に入れたものである
- 値が小さいほど、モデルにとって自然な文である
- 「毎回、平均して何個の候補から迷っているか」と解釈できる

パープレキシティには2通りの使い方がある

- **言語モデル自体の評価**: 人が書いた文章に対するパープレキシティが低いモデルほど、言語をよく捉えている
- **生成文の評価**: 別の言語モデルで生成文のパープレキシティを測り、文の自然さの目安にする

参照文が要らないのが利点だが、測っているのは**自然さ**であって**正しさ**ではない

パープレキシティを、モデルの追加学習（調整）の前後で比べる使い方は、`mlsys-text-J-NLP-8-LLM調整.ipynb` で扱っている

In [ ]:
# 言語モデルから見た文のパープレキシティを計算する
@torch.no_grad()
def perplexity(llm, text):
    tok, model = llm["tok"], llm["model"]
    # このトークナイザは文頭の記号を付けないので、最初のトークンにも文脈を与えるために終端記号を先頭に置く
    ids = torch.tensor([[tok.eos_token_id] + tok.encode(text, add_special_tokens=False)], device=device)
    logits = model(ids).logits[0, :-1].float()            # 位置 t の出力は、位置 t+1 のトークンの予測
    nll = torch.nn.functional.cross_entropy(logits, ids[0, 1:])  # 対数確率の平均（符号反転）
    return math.exp(nll.item())

for t in ["今日は天気が良いので、公園を散歩した", "今日は天気が良いので、公園を飲んだ", "公園した散歩を良いので天気が今日は"]:
    print(f"PPL {perplexity(LLM_L, t):9.1f}  {t}")

### 結果の読み方

- 自然な文のパープレキシティが最も低い
- 語を1つ不自然なものに替えると上がり、語順を崩すとさらに上がる

パープレキシティは「文として自然か」をよく反映している

次に、6つの指標を同じ土俵に並べ、人間の判断と比べる

## 指標が人の判断とずれる例

3つの質問について、模範解答（参照文）と7種類の回答を用意した

| 種類 | 内容 | 人の判断 |
| --- | --- | --- |
| `exact` | 参照文と同一 | 正しい |
| `paraphrase` | 同じ内容を別の言い回しで述べる | 正しい |
| `short` | 答えの語句だけ | 正しい |
| `verbose` | 前置きや補足を付けて正しく答える | 正しい |
| `wrong` | 参照文の数値や名称だけを誤ったものに替える | 誤り |
| `negation` | 参照文を否定形にする | 誤り |
| `offtopic` | 自然な文だが、質問に答えていない | 誤り |

`human` の列は、筆者が付けた正誤（1が正しい、0が誤り）である

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-overlap-pitfall.png" width=650>

上の図は、これから確かめる現象の要点である
参照文と数文字しか違わない誤答は高得点になり、言い換えた正答は低得点になる

In [ ]:
# 人の判断と比べるための回答例（参照文、回答の種類、回答、人の判断）
METRIC_CASES = [
    ("富士山の標高は3776メートルである", [
        ("exact", "富士山の標高は3776メートルである", 1),
        ("paraphrase", "日本一高い富士山は、高さが3776mある", 1),
        ("short", "3776メートル", 1),
        ("verbose", "ご質問ありがとうございます、富士山は日本を代表する山として知られており、その標高は3776メートルです", 1),
        ("wrong", "富士山の標高は3667メートルである", 0),
        ("negation", "富士山の標高は3776メートルではない", 0),
        ("offtopic", "富士山は静岡県と山梨県にまたがる山である", 0),
    ]),
    ("水は1気圧のもとで摂氏100度で沸騰する", [
        ("exact", "水は1気圧のもとで摂氏100度で沸騰する", 1),
        ("paraphrase", "1気圧では、水の沸点は100℃になる", 1),
        ("short", "100度", 1),
        ("verbose", "良い質問です、水の沸点は圧力によって変わりますが、標準的な1気圧の場合はちょうど摂氏100度で沸騰します", 1),
        ("wrong", "水は1気圧のもとで摂氏90度で沸騰する", 0),
        ("negation", "水は1気圧のもとで摂氏100度では沸騰しない", 0),
        ("offtopic", "水は1気圧のもとで摂氏0度で凍る", 0),
    ]),
    ("日本の首都は東京である", [
        ("exact", "日本の首都は東京である", 1),
        ("paraphrase", "東京が日本の首都だ", 1),
        ("short", "東京", 1),
        ("verbose", "はい、お答えします、日本の首都は関東地方に位置する東京であり、政治と経済の中心となっています", 1),
        ("wrong", "日本の首都は京都である", 0),
        ("negation", "日本の首都は東京ではない", 0),
        ("offtopic", "日本の首都は世界有数の大都市である", 0),
    ]),
]

rows = []
for ref, cands in METRIC_CASES:
    for kind, cand, human in cands:
        rows.append({"kind": kind, "candidate": cand, "human": human,
                     "EM": exact_match(cand, ref), "F1": char_f1(cand, ref), "BLEU": bleu(cand, ref),
                     "ROUGE-L": rouge_l(cand, ref), "Emb": emb_sim(cand, ref), "PPL": perplexity(LLM_L, cand)})
mdf = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(mdf.head(7).round(3).to_string(index=False))

上の表は、1つ目の質問（富士山の標高）に対する7種類の回答の点数である

- `wrong` と `negation` は人の判断では誤りだが、F1、BLEU、ROUGE-L、埋め込み類似度のいずれも、正しい `paraphrase` や `short` より高い

3つの質問をまとめ、回答の種類ごとに平均して確かめる

In [ ]:
# 回答の種類ごとに指標を平均し、人の判断との相関を調べる
order = ["exact", "paraphrase", "short", "verbose", "wrong", "negation", "offtopic"]
mdf["logPPL"] = np.log(mdf["PPL"])  # PPLは桁が大きく変わるので、対数を取ってから平均する
METRICS = ["EM", "F1", "BLEU", "ROUGE-L", "Emb", "logPPL"]
by_type = mdf.groupby("kind")[["human"] + METRICS].mean().loc[order]
print(by_type.round(3))

# 人の判断（0か1）と各指標の相関係数（PPLは小さいほど良いので、符号を反転して使う）
score = mdf[METRICS].copy()
score["logPPL"] = -score["logPPL"]
corr = score.corrwith(mdf["human"])
print("\n人の判断との相関係数")
print(corr.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
by_type[["F1", "BLEU", "ROUGE-L", "Emb"]].plot.bar(ax=ax[0], rot=30)
ax[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax[0].set_title("Metric score by answer type (first 4 types are correct)")
ax[0].set_xlabel("")
corr.rename({"logPPL": "-logPPL"}).plot.bar(ax=ax[1], rot=0, color="tab:gray")
ax[1].axhline(0, color="k", linewidth=0.8)
ax[1].set_title("Correlation with human judgment")
plt.tight_layout()
plt.show()

### 結果の読み方

人が「誤り」とした3種類（`wrong`、`negation`、`offtopic`）と、「正しい」とした4種類を見比べる

- **数値や名称の誤り（`wrong`）と否定（`negation`）は、どの重なり指標でも高得点になる**
  - 参照文と1〜3文字しか違わないからである
  - 内容は正反対でも、文字列としてはほぼ同じである
  - 埋め込み類似度も高い値のままである
    - 埋め込みは「何について述べているか」をよく捉えるが、数値の違いや否定には鈍い
- **言い換え（`paraphrase`）、短答（`short`）、丁寧な回答（`verbose`）は、正しいのに低得点になる**
  - 完全一致は `exact` 以外すべて0である
  - F1、BLEU、ROUGE-Lは、長さが参照文と違うだけで大きく下がる
- **パープレキシティは正誤と関係しない**
  - 誤った文も、文としては自然だからである

人の判断との相関係数を見ると、F1、BLEU、ROUGE-Lは**負**になっている
この例では、誤った回答のほうが、正しい回答よりも平均して高い点を得ている

本来、これらの指標は、翻訳や要約のシステムを多数の文で比べる場面では、人の評価と正の相関を持つことが報告されており、だからこそ広く使われてきた

ここで相関が負になったのは、指標の弱点を突く回答（1〜3文字だけを変えた誤答、長さの違う正答）を意図的に集め、1文ごとの正誤と比べたからである
実際のデータでここまで相関が低いとは限らない
しかし、**誤答のうち最も危険なもの（もっともらしく、参照文とほとんど同じ文面で、内容だけ違う）を、重なりに基づく指標は見逃す**という性質は一般に成り立つ

### 指標の使い分け

| 指標 | 向いている場面 | 向いていない場面 |
| --- | --- | --- |
| 完全一致 | 答えが数値、選択肢、固有名詞など短く定まる | 文章で答える課題 |
| F1 | 短い語句を抜き出す質問応答 | 長い回答、否定や数値の正誤が重要な課題 |
| BLEU、ROUGE | 翻訳や要約で、多数の文を集計してシステムどうしを比べる | 1文ごとの正誤の判定 |
| 埋め込み類似度 | 話題が合っているかの確認、言い換えを許す照合 | 事実の正誤の判定 |
| パープレキシティ | 言語モデルの比較、文の自然さの確認 | 内容の正しさの判定 |

自動指標で測れるように、**課題の側を設計する**のも有効である

- 計算問題のように、答えを数値だけにする
- 選択式にする（次の節）
- 出力をJSONなどの決まった形式にさせ、項目ごとに完全一致で比べる

文章で答える課題の正誤は、自動指標だけでは測れない
その場合は、人手による評価か、LLMによる採点（後の節）を使う

## パープレキシティと汚染の検出

パープレキシティには、評価データの汚染を調べる使い道もある

モデルが学習中に何度も見た文章は、次のトークンを高い確率で言い当てられるため、パープレキシティが極端に低くなる

次のセルでは、有名な文章の冒頭と、その一部の語を入れ替えた文で、パープレキシティを比べる

In [ ]:
# 有名な文と、語を入れ替えた文のパープレキシティを比べる
PAIRS = [
    ("吾輩は猫である。名前はまだ無い。", "吾輩は犬である。住所はまだ無い。"),
    ("The quick brown fox jumps over the lazy dog", "The quick brown cat jumps over the lazy fox"),
    ("To be, or not to be, that is the question", "To eat, or not to eat, that is the problem"),
]
for famous, altered in PAIRS:
    print(f"PPL {perplexity(LLM_L, famous):8.1f}  {famous}")
    print(f"PPL {perplexity(LLM_L, altered):8.1f}  {altered}\n")

### 結果の読み方

- 有名な文は、語を入れ替えた文よりもパープレキシティが数倍低い
  - 入れ替えた文も文法的には自然であるから、この差は自然さの差ではなく、**学習データで見たことがあるか**の差である
- 評価に使う文章のパープレキシティが、同じ種類の新しい文章と比べて不自然に低ければ、そのデータは学習に使われた疑いがある

これは疑いを示す手がかりであって、証明ではない
パープレキシティが低くなる理由は、暗記のほかにも、文が単純である、定型的であるなど複数ある

---

# 選択式ベンチマークの仕組み

LLMの性能比較でよく使われるベンチマークの多くは、**多肢選択式（multiple choice）**である

- 答えが記号1つに定まるため、採点が完全一致で済む
- 前の節で見た「文章の正誤を自動で判定できない」という問題を、課題の形式を変えることで避けている

この節では、小さな選択式ベンチマークを自作し、採点の仕組みを2通り実装する

選択式の評価は単純に見えるが、**採点方法と選択肢の並べ方によって点数が変わる**ことを実験で確かめる

## 問題を作る

5つの分野（科学、地理、歴史、情報、数学）から8問ずつ、合計40問の4択問題を用意する

- 正解の選択肢を先頭に書いておき、出題するときに並べ替える
- `open` は、選択肢を見せなくても答えが1つに定まる問題かどうかを表す
  - 後の節で、同じ問題を記述式として使うための印である

In [ ]:
# 多肢選択問題（正解を先頭に書き、出題時に並べ替える）
# open が False の問題は、選択肢が無いと答えを1つに決められないので、記述式では使わない
MC_RAW = [
    ("科学", "水の化学式は何か", ["H2O", "CO2", "NaCl", "O2"], True),
    ("科学", "光合成で植物が取り込む気体は何か", ["二酸化炭素", "酸素", "窒素", "水素"], True),
    ("科学", "太陽系で最も大きい惑星は何か", ["木星", "土星", "地球", "海王星"], True),
    ("科学", "人体の内臓で最も重い臓器は何か", ["肝臓", "心臓", "腎臓", "胃"], True),
    ("科学", "次のうち音が最も速く伝わるものは何か", ["鉄", "水", "空気", "真空"], False),
    ("科学", "原子番号1の元素は何か", ["水素", "ヘリウム", "炭素", "酸素"], True),
    ("科学", "DNAでアデニンと対になる塩基は何か", ["チミン", "グアニン", "シトシン", "ウラシル"], True),
    ("科学", "力の大きさを表すSI単位は何か", ["ニュートン", "ジュール", "ワット", "パスカル"], True),
    ("地理", "日本で最も面積が大きい都道府県はどこか", ["北海道", "岩手県", "長野県", "新潟県"], True),
    ("地理", "オーストラリアの首都はどこか", ["キャンベラ", "シドニー", "メルボルン", "パース"], True),
    ("地理", "日本で最も面積が大きい湖は何か", ["琵琶湖", "霞ヶ浦", "サロマ湖", "猪苗代湖"], True),
    ("地理", "エベレストがある山脈は何か", ["ヒマラヤ山脈", "アルプス山脈", "アンデス山脈", "ロッキー山脈"], True),
    ("地理", "カナダの首都はどこか", ["オタワ", "トロント", "バンクーバー", "モントリオール"], True),
    ("地理", "次のうち赤道が通る国はどこか", ["エクアドル", "エジプト", "インド", "アルゼンチン"], False),
    ("地理", "日本の標準時子午線が通る兵庫県の市はどこか", ["明石市", "姫路市", "神戸市", "西宮市"], True),
    ("地理", "世界で最も面積が大きい国はどこか", ["ロシア", "カナダ", "中国", "アメリカ合衆国"], True),
    ("歴史", "江戸幕府を開いた人物は誰か", ["徳川家康", "織田信長", "豊臣秀吉", "足利尊氏"], True),
    ("歴史", "明治元年は西暦何年か", ["1868年", "1853年", "1889年", "1912年"], True),
    ("歴史", "鎌倉幕府を開いた人物は誰か", ["源頼朝", "平清盛", "北条時宗", "足利義満"], True),
    ("歴史", "第二次世界大戦が終結したのは西暦何年か", ["1945年", "1939年", "1941年", "1950年"], True),
    ("歴史", "『源氏物語』の作者は誰か", ["紫式部", "清少納言", "小野小町", "和泉式部"], True),
    ("歴史", "フランス革命が始まったのは西暦何年か", ["1789年", "1776年", "1815年", "1848年"], True),
    ("歴史", "15世紀のヨーロッパで活版印刷術を実用化した人物は誰か", ["グーテンベルク", "ガリレオ", "コペルニクス", "ルター"], True),
    ("歴史", "慶應義塾を創設した人物は誰か", ["福澤諭吉", "大隈重信", "新島襄", "津田梅子"], True),
    ("情報", "1バイトは何ビットか", ["8ビット", "4ビット", "16ビット", "32ビット"], True),
    ("情報", "10進数の10を2進数で表すとどうなるか", ["1010", "1001", "1100", "0110"], True),
    ("情報", "Webページの構造を記述するマークアップ言語は何か", ["HTML", "SQL", "TCP", "CPU"], True),
    ("情報", "整列済みの配列に対する二分探索の時間計算量はどれか", ["O(log n)", "O(n)", "O(n log n)", "O(n^2)"], True),
    ("情報", "ドメイン名をIPアドレスに変換する仕組みは何か", ["DNS", "DHCP", "HTTP", "FTP"], True),
    ("情報", "後入れ先出し（LIFO）で動くデータ構造は何か", ["スタック", "キュー", "ヒープ", "ハッシュ表"], True),
    ("情報", "16進数のFを10進数で表すといくつか", ["15", "10", "16", "255"], True),
    ("情報", "ニューラルネットワークで勾配を出力側から入力側へ伝える計算法は何か", ["誤差逆伝播法", "主成分分析", "k平均法", "決定木"], True),
    ("数学", "三角形の内角の和は何度か", ["180度", "90度", "270度", "360度"], True),
    ("数学", "2の10乗はいくつか", ["1024", "512", "2048", "1000"], True),
    ("数学", "次のうち素数はどれか", ["29", "21", "27", "33"], False),
    ("数学", "次のうち円周率に最も近い値はどれか", ["3.14", "2.72", "1.41", "1.62"], False),
    ("数学", "関数 x^2 を x で微分すると何になるか", ["2x", "x", "x^2", "2"], True),
    ("数学", "1から10までの整数の和はいくつか", ["55", "45", "50", "60"], True),
    ("数学", "正六角形の1つの内角は何度か", ["120度", "108度", "135度", "90度"], True),
    ("数学", "10を底とする1000の対数はいくつか", ["3", "2", "10", "100"], True),
]
MC = [{"id": f"mc{i:02d}", "cat": c, "q": q, "options": o, "open": op} for i, (c, q, o, op) in enumerate(MC_RAW)]
print(len(MC), "問", Counter(m["cat"] for m in MC))

## 出題の形式

各問題を次の形のプロンプトにする

```text
次の問題に、選択肢の記号（A〜D）1文字だけで答えよ

問題: 水の化学式は何か
A. H2O
B. CO2
C. NaCl
D. O2
```

正解の位置は、問題番号に応じてA、B、C、Dに均等に割り振る
`shift` を変えると、すべての問題で選択肢が1つずつ巡回し、正解の位置がずれる
4通りの `shift` を試すと、どの選択肢もA〜Dの各位置に1回ずつ置かれる

In [ ]:
# 選択肢を並べ替えて出題用のプロンプトを作る
LETTERS = ["A", "B", "C", "D"]
MC_INST = "次の問題に、選択肢の記号（A〜D）1文字だけで答えよ"

def mc_layout(i, shift=0):
    """i番目の問題で、各位置に元の何番目の選択肢を置くかを返す（元の0番が正解）"""
    pos = (i + shift) % 4                        # 正解を置く位置
    order = [(k - pos) % 4 for k in range(4)]    # 選択肢を pos だけ巡回させる
    return order, pos

def mc_prompt(m, order, inst=MC_INST):
    lines = [f"{LETTERS[k]}. {m['options'][j]}" for k, j in enumerate(order)]
    return f"{inst}\n\n問題: {m['q']}\n" + "\n".join(lines)

order0, pos0 = mc_layout(0)
print(mc_prompt(MC[0], order0))
print("\n正解の位置:", LETTERS[pos0])

## 採点方法1: 選択肢の対数尤度

LLMは、次に来るトークンの確率分布を出力する

プロンプトの直後に「A」「B」「C」「D」の各トークンが来る確率を取り出し、最も高いものをモデルの答えとする

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-mc-scoring.png" width=700>

上の図の上段がこの方法である
下段は、このあとで扱うもう1つの方法（文章で答えさせ、記号を取り出す）である

- 文章を生成させないため、速い（1問につき順伝播1回）
- 出力から答えを取り出す処理が要らず、必ずA〜Dのいずれかが得られる
- モデルの出力確率を取り出せることが前提である
  - 重みが公開されたモデルでは可能である
  - APIで提供されるモデルでは、確率を取得できない、または一部しか取得できないことがある

多くの公開ベンチマークの評価プログラムは、この方式か、選択肢の本文それぞれの尤度を比べる方式を採用している

In [ ]:
# 採点方法1: 次のトークンが A〜D である対数確率を比べる
def mc_loglik(llm, shift=0, items=MC):
    """各問題の予測位置、正解位置、A〜Dの確率を返す"""
    layouts = [mc_layout(i, shift) for i in range(len(items))]
    prompts = [mc_prompt(m, order) for m, (order, _) in zip(items, layouts)]
    lp = next_token_logprobs(llm, prompts, LETTERS)
    prob = np.exp(lp) / np.exp(lp).sum(axis=1, keepdims=True)  # A〜Dの4つの中での確率に直す
    return {"pred": prob.argmax(axis=1), "gold": np.array([p for _, p in layouts]), "prob": prob, "layouts": layouts}

ll_L, ll_S = mc_loglik(LLM_L), mc_loglik(LLM_S)
for name, r in [("大モデル", ll_L), ("小モデル", ll_S)]:
    m, lo, hi = bootstrap_ci(r["pred"] == r["gold"])
    print(f"{name}: 正解率 {m:.3f}  95%信頼区間 [{lo:.3f}, {hi:.3f}]")
print("大モデルの1問目の確率（A〜D）:", ll_L["prob"][0].round(3), " 正解:", LETTERS[ll_L["gold"][0]])

### 結果の読み方

- 4択であるから、でたらめに答えても正解率は25%になる
  - 選択式の点数は、この**偶然の水準（chance level）**との差で読む
  - 小モデルの正解率は、偶然の水準とほとんど変わらない
    - 計算問題では大モデルと10ポイントしか違わなかったモデルが、ここでは何も知らないかのような点数になっている
    - この理由は、あとの実験で明らかになる
- 40問しか無いため、信頼区間は±10〜15ポイントある
  - 前の節で学んだとおり、この規模では大きな差しか見分けられない

## 採点方法2: 生成させて抽出する

もう1つの方法は、モデルに文章で答えさせ、その中から記号を取り出すものである

- 確率を取り出せないモデルでも使える
- 理由を書かせてから答えさせる（段階的に考えさせる）ことができる
- 出力から答えを取り出す規則が必要であり、**取り出しに失敗することがある**

ここでは、理由を1文で述べたあとに「答え: 記号」の形で答えるよう指示する

取り出しの規則を2つ用意し、点数がどう変わるかを見る

- **厳格な規則**: 「答え: X」の形に合うものだけを答えとして認める
- **緩い規則**: 形に合わなければ、出力の最後に現れたA〜Dの文字を答えとみなす

In [ ]:
# 採点方法2: 文章で答えさせ、記号を取り出す
MC_INST_GEN = "次の問題について、まず理由を1文で述べ、最後の行に「答え: 記号」の形でA〜Dのいずれかを答えよ"

def extract_choice(text, strict=True):
    """出力から答えの記号を取り出す（見つからなければ None）"""
    t = unicodedata.normalize("NFKC", text)
    m = re.findall(r"答え\s*[:は]?\s*([A-D])", t)
    if m:
        return m[-1]
    if strict:
        return None
    m = re.findall(r"(?<![A-Za-z])([A-D])(?![A-Za-z])", t)  # 英単語の一部でない A〜D
    return m[-1] if m else None

assert extract_choice("水素と酸素からなる\n答え: B") == "B"
assert extract_choice("正解はCです") is None and extract_choice("正解はCです", strict=False) == "C"

def mc_generate(llm, shift=0, items=MC):
    layouts = [mc_layout(i, shift) for i in range(len(items))]
    prompts = [mc_prompt(m, order, MC_INST_GEN) for m, (order, _) in zip(items, layouts)]
    outs = chat(llm, prompts, max_new_tokens=96)
    return {"outputs": outs, "gold": np.array([p for _, p in layouts])}

gen_L, gen_S = mc_generate(LLM_L), mc_generate(LLM_S)
print(gen_L["outputs"][0], "\n")

rows = []
for name, ll, gen in [("large", ll_L, gen_L), ("small", ll_S, gen_S)]:
    gold = [LETTERS[g] for g in gen["gold"]]
    strict = [extract_choice(o, strict=True) for o in gen["outputs"]]
    loose = [extract_choice(o, strict=False) for o in gen["outputs"]]
    ll_pred = [LETTERS[p] for p in ll["pred"]]
    rows.append({"model": name,
                 "loglik": np.mean([p == g for p, g in zip(ll_pred, gold)]),
                 "gen_strict": np.mean([p == g for p, g in zip(strict, gold)]),
                 "gen_loose": np.mean([p == g for p, g in zip(loose, gold)]),
                 "fail_strict": sum(p is None for p in strict),
                 "fail_loose": sum(p is None for p in loose),
                 "agree(loglik,gen_loose)": np.mean([p == q for p, q in zip(ll_pred, loose)])})
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 結果の読み方

- 同じモデル、同じ問題でも、**採点方法によって正解率が変わる**
  - 筆者の実行では、大モデルは対数尤度で0.85、生成で0.725であった
  - 小モデルは逆に、生成のほうがやや高かった
  - 採点方法を変えるだけで、2つのモデルの差が大きくも小さくもなる
- `fail_strict` は、厳格な規則で答えを取り出せなかった件数である
  - 取り出せなかった出力は不正解として数えるため、指示した形式を守れないモデルほど点が下がる
  - これは知識の不足ではなく、**指示に従う能力**の不足である
  - 緩い規則にすると取り出せる件数が増え、点数も変わる
- `agree` は、対数尤度による答えと、生成による答えが一致した割合である
  - 2つの方法は、同じモデルの同じ知識を測っているはずだが、答えは6〜8割しか一致しない
  - 理由を書かせると、その理由に引きずられて答えが変わることがある

### 理由を書かせると下がった理由

理由を書かせてから答えさせる方法は、一般には正解率を上げるとされている（Wei et al., 2022）

ここでは、大モデルの正解率が逆に下がった

- この効果が報告されているのは、主に計算や多段の推論を要する課題である
  - 今回の問題の多くは、知識を思い出すだけで答えられるものであり、段階的に考える利益が小さい
- 小型のモデルが書く1文の理由は、それ自体が誤りを含みやすい
  - 先に誤った理由を書くと、答えがそれに引きずられる
- 40問しか無いため、0.85と0.725の差は5問の違いである
  - 信頼区間は大きく重なっており、「理由を書かせると悪化する」とは断定できない

どちらの方法が正しいということはない

- 対数尤度は「モデルが最初の1文字として何を出したがるか」を測る
- 生成は「実際に使うときと同じ手順で答えさせたときの結果」を測る

**ベンチマークの点数を比べるときは、採点方法がそろっていることを確認する**
別々の採点方法で得た点数を並べても、モデルの差なのか採点方法の差なのかを区別できない

## 選択肢の順序への敏感さ

人間であれば、選択肢の並び順が変わっても答えは変わらない

LLMではどうか

`shift` を0から3まで変え、すべての問題について、正解がA、B、C、Dのそれぞれの位置にある場合を採点する

- 同じ問題が、並び順だけを変えて4回出題される
- 順序に影響されないモデルなら、4回とも同じ選択肢（内容）を選ぶはずである

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-mc-rotation.png" width=650>

上の図は、1つの問題を4通りの並びで出題する様子である
色の付いた選択肢が正解であり、並びごとにA、B、C、Dへ移っていく

In [ ]:
# 正解の位置を4通りに変えて採点し、順序への敏感さを調べる
def mc_order_experiment(llm):
    n = len(MC)
    correct = np.zeros((n, 4), dtype=bool)   # correct[i, p]: i番目の問題で、正解が位置pにあるときに正解したか
    pred_count = np.zeros(4)                 # A〜Dを答えた回数
    content_prob = np.zeros((n, 4))          # 並び順について平均した、元の各選択肢の確率
    for shift in range(4):
        r = mc_loglik(llm, shift)
        for i in range(n):
            order, pos = r["layouts"][i]
            correct[i, pos] = r["pred"][i] == pos
            pred_count[r["pred"][i]] += 1
            for k, j in enumerate(order):
                content_prob[i, j] += r["prob"][i, k] / 4
    return correct, pred_count / pred_count.sum(), content_prob

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for k, (name, llm) in enumerate([("large", LLM_L), ("small", LLM_S)]):
    correct, pred_dist, content_prob = mc_order_experiment(llm)
    print(f"[{name}]")
    print("  正解の位置ごとの正解率 (A,B,C,D):", correct.mean(axis=0).round(3))
    print("  答えた記号の割合       (A,B,C,D):", pred_dist.round(3))
    print(f"  4通りすべてで正解: {correct.all(axis=1).mean():.3f}   少なくとも1通りで正解: {correct.any(axis=1).mean():.3f}")
    print(f"  4通りの確率を平均してから答えを決めた場合の正解率: {(content_prob.argmax(axis=1) == 0).mean():.3f}")
    ax[0].bar(np.arange(4) + 0.4 * k - 0.2, correct.mean(axis=0), width=0.4, label=name)
    ax[1].bar(np.arange(4) + 0.4 * k - 0.2, pred_dist, width=0.4, label=name)
for a, title in zip(ax, ["Accuracy by position of the correct option", "Share of predicted letters"]):
    a.set_xticks(range(4)); a.set_xticklabels(LETTERS); a.set_title(title)
    a.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
ax[1].axhline(0.25, color="k", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

### 結果の読み方

- **正解の位置によって正解率が変わる**
  - 問題の内容は同じであるから、差はすべて並び順の影響である
- **答える記号に偏りがある**
  - 正解はA〜Dに均等に置いたので、偏りのないモデルなら各記号を25%ずつ答えるはずである
  - 小モデルは、ほとんどすべての問題で「A」を答えている
    - 正解がAにあれば必ず正解し、それ以外ではほぼ必ず間違える
    - 先ほどの「偶然の水準と変わらない正解率」は、知識が無いことを示していたのではなく、**記号の偏りが知識を覆い隠していた**結果である
    - もし評価データの正解がすべてAに置かれていたら、このモデルは満点を取る
  - 大モデルにも、程度は小さいが同じ向きの偏りがある
- **「4通りすべてで正解」と「少なくとも1通りで正解」の間に開きがある**
  - この開きの分だけ、並び順という偶然に点数が左右されている
  - 「4通りすべてで正解」は、並び順に頼らずに本当に知っている問題の割合の目安になる

この現象は小型モデルで極端に現れるが、大規模なモデルでも報告されている

### 対処

- **正解の位置を均等にする**: 正解がAに偏った評価データでは、Aを答えやすいモデルが不当に高得点になる
- **並び順を変えて複数回測る**: 順序による変動幅を、点数と一緒に報告する
- **並び順について平均する**: 4通りの確率を選択肢の内容ごとに平均してから答えを決めると、位置の偏りが打ち消される
  - 上のセルの最後の行がこの方法の結果である
  - 小モデルの正解率は、この方法で大きく上がる
    - 記号の偏りを取り除くと、小モデルも偶然の水準を上回る知識を持っていたことがわかる
  - 評価の回数は4倍になる

選択式ベンチマークの点数は、採点方法、取り出しの規則、選択肢の並び順という、問題の内容とは無関係な要因に左右される

公開ベンチマークの順位表を読むときは、これらの条件がそろえられているかを確認する

## モデルを大きくすると偏りはどう変わるか

小モデルの記号の偏りは極端であった

同じ実験を3Bモデルでも行い、モデルの大きさと偏りの関係を見る

表の列の意味は次のとおりである

| 列 | 内容 |
| --- | --- |
| `loglik`、`generate` | 対数尤度による正解率、生成と抽出（緩い規則）による正解率 |
| `acc_min_pos`、`acc_max_pos` | 正解の位置（A〜D）ごとの正解率の最小と最大 |
| `max_letter_share` | 最もよく答えた記号の割合（偏りが無ければ0.25） |
| `all4`、`any` | 4通りの並びすべてで正解した割合、少なくとも1通りで正解した割合 |
| `averaged` | 4通りの確率を平均してから答えを決めた場合の正解率 |

In [ ]:
# 3つのモデルで、採点方法による違いと、順序への敏感さを比べる
ll_XL, gen_XL = mc_loglik(LLM_XL), mc_generate(LLM_XL)
rows, order_res = [], {}
for name, llm, ll, gen in [("small(0.5B)", LLM_S, ll_S, gen_S), ("large(1.5B)", LLM_L, ll_L, gen_L), ("3B", LLM_XL, ll_XL, gen_XL)]:
    correct, pred_dist, content_prob = mc_order_experiment(llm)
    order_res[name] = (correct, pred_dist)
    loose = [extract_choice(o, strict=False) for o in gen["outputs"]]
    rows.append({"model": name,
                 "loglik": np.mean(ll["pred"] == ll["gold"]),
                 "generate": np.mean([p == LETTERS[g] for p, g in zip(loose, gen["gold"])]),
                 "acc_min_pos": correct.mean(axis=0).min(), "acc_max_pos": correct.mean(axis=0).max(),
                 "max_letter_share": pred_dist.max(),
                 "all4": correct.all(axis=1).mean(), "any": correct.any(axis=1).mean(),
                 "averaged": (content_prob.argmax(axis=1) == 0).mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for k, (name, (correct, pred_dist)) in enumerate(order_res.items()):
    ax[0].bar(np.arange(4) + 0.27 * (k - 1), correct.mean(axis=0), width=0.27, label=name)
    ax[1].bar(np.arange(4) + 0.27 * (k - 1), pred_dist, width=0.27, label=name)
for a, title in zip(ax, ["Accuracy by position of the correct option", "Share of predicted letters"]):
    a.set_xticks(range(4)); a.set_xticklabels(LETTERS); a.set_title(title)
    a.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
ax[1].axhline(0.25, color="k", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

### 結果の読み方

- **記号の偏りは、モデルが大きいほど小さい**
  - 最もよく答えた記号の割合は、筆者の実行では 0.96 → 0.31 → 0.28 と、偏りの無い値（0.25）に近づく
  - 正解の位置による正解率の開きも、0.05〜1.00 → 0.75〜0.88 → 0.85〜0.90 と狭まる
- **並び順に頼らずに正解できる問題が増える**
  - 「4通りすべてで正解」は 0.05 → 0.65 → 0.80 と上がる
- **それでも、偏りは無くならない**
  - 3Bモデルでも、「少なくとも1通りで正解」と「4通りすべてで正解」の間に15ポイントの開きがある
  - 並び順という偶然で正否が変わる問題が、まだ15%ある
- **採点方法による違いも残る**
  - 3Bモデルでも、対数尤度と生成では正解率が10ポイント以上違う

小型モデルで見た現象は、小型モデルだけの欠陥ではなく、**程度を変えてどの規模のモデルにも現れる性質**である

さらに大規模なモデルでも、選択肢の順序への敏感さや記号の偏りが残ることが報告されている（参考文献を参照）
モデルが大きいからといって、順序を変えた確認を省いてよいことにはならない

---

# LLMによる採点（LLM-as-a-judge）

文章で答える課題の正誤は、文字列の比較では判定できなかった

人が読んで採点すれば判定できるが、時間と費用がかかり、変更のたびに回すことはできない

そこで、**採点そのものをLLMに行わせる**方法が広く使われている
これを**LLM-as-a-judge**と呼ぶ

- 言い換えや冗長な回答でも、内容で判定できる
- 人手より速く安い
- 一方で、採点者であるLLM自身が間違える
  - 採点者の誤りは、評価結果にそのまま混入する
  - **採点者を使う前に、採点者自体を評価しなければならない**

この節では、採点基準の書き方、2つの採点形式（単独採点とペア比較）、採点者が持つ偏り、そして人手ラベルによる採点者の検証を扱う

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-judge-flow.png" width=720>

上の図の左側が採点の流れ、右側の枠が採点者そのものの検証である

## 採点者を検証するための答案集

採点者を検証するには、**人が正誤を付けた答案**が要る

ここでは、前の節の40問のうち、選択肢なしで答えが定まる36問を記述式で出題し、大モデルと小モデルが実際に出力した回答（合計72件）を固定して使う

- 回答は `質問 + "\n1文で簡潔に答えよ"` というプロンプトに対する貪欲法の出力であり、48トークンで打ち切っている
- 各回答には、筆者が次の基準で正誤（1が正解、0が不正解）を付けた
  - 質問で問われた事項（名称、人物、数値、年）が模範解答と合っていれば正解とする
  - 表記の違い（「二酸化炭素」と「CO2」など）は問わない
  - 付け加えた説明に誤りがあっても、問われた事項が合っていれば正解とする
  - 答えを複数挙げ、その中に誤りがあれば不正解とする

答案をその場で生成せずに固定するのは、人手ラベルと答案の対応を保つためである
答案が変われば、人手ラベルも付け直さなければならない

In [ ]:
# 固定した答案集: (問題ID, 大モデルの回答, 人の判定, 小モデルの回答, 人の判定)
ANSWERS = [
    ('mc00', "水の化学式はH2Oです。", 1, "水の化学式はH2Oです。", 1),
    ('mc01', "光合成では植物が取り込む気体はCO2です。", 1, "植物が取り込む気体は、水と二氧化碳です。", 0),
    ('mc02', "太陽系で最も大きい惑星は木星です。", 1, "太陽系で最も大きな惑星は木星です。", 1),
    ('mc03', "人間の内臓で最も重いのは心です。", 0, "人間の内臓で最も重い臓器は、肺です。", 0),
    ('mc05', "アルカリ金属（鉛）", 0, "原子番号1の元素は鉛です。", 0),
    ('mc06', "アデニンの対となる塩基はジオシナ。", 0, "アデニンと対応する塩基は、2-メタクリル酸シロキサンです。", 0),
    ('mc07', "力の大きさを表すSI単位はニュートン（N）です。", 1, "力の大きさをSI単位として表すのは、「牛」です。\n\n1牛（N）は、重さが約1000kg（約220000g）とされる重量", 0),
    ('mc08', "日本の面積最大の都道府県は、北海道です。", 1, "日本の最大の都市は東京都です。", 0),
    ('mc09', "オーストラリアの首都はデビルです。", 0, "オーストラリアの首都はマッカーシーです。", 0),
    ('mc10', "日本の最も面積が大きい湖は、静岡県の伊豆大島にある伊豆湖です。", 0, "日本の最大の湖は、東海道西岸にある鹿児島県の鹿児島湖です。", 0),
    ('mc11', "エベレストがある山脈は、モンブラン山脈です。", 0, "エベレスト（ Everest）がある山脈は、中国の Nepal と India の交界部にある。", 0),
    ('mc12', "カナダの首都はバンクーバーです。", 0, "カナダの首都はオーストラリアのマッカーシーです。", 0),
    ('mc14', "神戸市", 0, "兵庫県の市中には、日本標準時子午線が通っているのは以下のような2つです：\n\n1. 佐賀県佐賀市：東北地方の都市\n\n2. 長野", 0),
    ('mc15', "世界で最も面積が大きい国はアメリカ合衆国です。", 0, "世界で最も面積が大きい国は、中国です。", 0),
    ('mc16', "江戸幕府を開いた人物は徳川家康です。", 1, "江戸幕府の開発には、以下の人物が関わったことが挙げられます：\n\n1. 伊藤博文（いとう ひろし）：江戸幕府の創立者であり、", 0),
    ('mc17', "明治元年は西暦1868年です。", 1, "明治元年は1868年です。", 1),
    ('mc18', "鎌倉幕府を開いた人物は武田信玄です。", 0, "鎌倉幕府の開発者は、元禄時代の松平家です。", 0),
    ('mc19', "第二次世界大戦は1945年に終結しました。", 1, "第二次世界大戦（20世紀初頭）が終結したのは、1945年7月7日に日本軍の突撃によって東京を攻撃し、その後の戦争", 1),
    ('mc20', "『源氏物語』の作者は紫式部です。", 1, "源氏物語の作者は、日本古代の文学家、元禄時代（1680年-1725年）の松本茂です。", 0),
    ('mc21', "フランス革命は1789年に始まりました。", 1, "フランス革命は1789年に始まりました。1789年5月20日、ローマの大殿で君主制を廃止し、共和制を導入しました。この時期", 1),
    ('mc22', "15世紀のヨーロッパで活版印刷術を実用化した人物は、イタリアのクリストフ・ブリオッチャです。", 0, "15世紀のヨーロッパで活版印刷術を実用化した人物は、ロバート・デュイエスです。", 0),
    ('mc23', "慶應義塾を創設した人物は、伊藤博文です。", 0, "慶應義塾大学の創立者は、日本近代教育の先駆けとして有名な松永道雄です。", 0),
    ('mc24', "1バイトは1ビットです。", 0, "1バイトは8ビットです。", 1),
    ('mc25', "10進数の10は、2進数で表すと1010です。", 1, "10進数の10を2進数で表すと、10は「1010」です。", 1),
    ('mc26', "HTML (HyperText Markup Language)", 1, "Webページの構造を記述するマークアップ言語は、HTML（HyperText Markup Language）です。以下はHTMLの基本的な要素と構造について説明します：\n\n1. <html>：HTML", 1),
    ('mc27', "整列済み配列に対する二分探索の時間計算量はO(log n)です。", 1, "整列済みの配列に対する二分探索の時間計算量は、O(log n)です。この計算法は、配列が完全な二分探索を可能にするための最適な方法です。", 1),
    ('mc28', "ドメイン名とIPアドレスの対応関係は、DNS（Domain Name System）によって管理されています。", 1, "ドメイン名とIPアドレスの間には直接的な関係はありませんが、DNS（デジタルネームサーバー）システムでは、それぞれの場所から情報を伝えるためのルートパスとして使用されます。", 1),
    ('mc29', "後入先出(LIFO)のデータ構造はスタック(Stack)です。", 1, "後入れ先出し（LIFO）で動くデータ構造は、リストや配列です。", 0),
    ('mc30', "16進数のFは10進数で24です。", 0, "16進数のFを10進数で表すと、\n\n15 (16进制) = 23 (10进制)\n\n理由は以下の通りです：\n\n- 16進数の", 0),
    ('mc31', "ニューラルネットワークの勾配を出力側から入力側へ伝える方法は、逆伝播法（Backpropagation）です。", 1, "ニューラルネットワークの勾配伝播法（Gradient Descent）を使用する場合、出力端から入力端への伝播を行うための計算方法です。この法則は、学習データに対する誤", 0),
    ('mc32', "三角形の内角の和は180度です。", 1, "三角形の内角の和は180度です。", 1),
    ('mc33', "2の10乗は1024です。", 1, "2の10乗は、1000です。", 0),
    ('mc36', "x^2 の x で微分すると 2x になります。", 1, "x の微分は x の 2 個の次元の変数行列を表す。つまり、x^2 の微分は d(x^2)/dx = 2x。", 1),
    ('mc37', "1から10までの整数の和は55です。", 1, "1から10までの整数の和は55です。", 1),
    ('mc38', "正六角形の1つの内角は120度です。", 1, "正六角形の1つの内角は120度です。", 1),
    ('mc39', "10を底とする1000の対数は2です。", 0, "10の10乗 = 1000", 0),
]

QMAP = {m["id"]: m for m in MC}
JUDGE_SET = []  # 採点対象を1件ずつ並べたもの
for qid, ans_l, h_l, ans_s, h_s in ANSWERS:
    for author, ans, h in [("large", ans_l, h_l), ("small", ans_s, h_s)]:
        JUDGE_SET.append({"qid": qid, "q": QMAP[qid]["q"], "ref": QMAP[qid]["options"][0],
                          "answer": ans, "human": h, "author": author})
human = np.array([it["human"] for it in JUDGE_SET])
print(len(JUDGE_SET), "件  人の判定による正解率: 大モデル",
      round(float(human[0::2].mean()), 3), " 小モデル", round(float(human[1::2].mean()), 3))
for it in JUDGE_SET[2:4]:
    print(it)

## 採点基準（ルーブリック）の書き方

採点者への指示を**採点基準（rubric、ルーブリック）**と呼ぶ

人に採点を頼むときと同じで、基準が曖昧だと採点がぶれる

良い採点基準の条件は次のとおりである

1. **判定する観点を1つに絞る**
   - 「正確さ」「わかりやすさ」「丁寧さ」をまとめて1つの点数にさせない
   - 観点ごとに別の採点を行う
2. **段階を少なくする**
   - 二値（正解、不正解）か、多くても3〜5段階にする
   - 10点満点や100点満点は、段階の違いを説明できないため、採点がぶれやすい
3. **境界事例の扱いを書く**
   - 表記の違い、余分な説明、複数の答えの併記など、迷う場合の判定を明記する
4. **模範解答を与える**
   - 模範解答が無いと、採点者は自分の知識で正誤を決めることになり、採点者の知識の誤りが混入する
5. **出力の形式を固定する**
   - 判定を機械的に取り出せるようにする

次のセルで、3種類の採点プロンプトを用意する

| 名前 | 模範解答 | 採点基準 |
| --- | --- | --- |
| `vague` | 与えない | 「良い回答か」とだけ尋ねる |
| `ref` | 与える | 「正解か不正解か」とだけ尋ねる |
| `rubric` | 与える | 境界事例の扱いまで書く |

In [ ]:
# 3種類の採点プロンプトと、判定を取り出す関数
def prompt_vague(it):
    return f"次の回答は良い回答か\n「良い」または「悪い」のどちらか一語だけで答えよ\n\n質問: {it['q']}\n回答: {it['answer']}"

def prompt_ref(it):
    return (f"次の回答は正解か不正解か\n「正解」または「不正解」のどちらか一語だけで答えよ\n\n"
            f"質問: {it['q']}\n模範解答: {it['ref']}\n回答: {it['answer']}")

RUBRIC = """あなたは採点者である
質問、模範解答、採点対象の回答を読み、次の基準で判定せよ

採点基準:
1. 質問で問われた事項（名称、人物、数値、年）が模範解答と同じであれば「正解」とする
2. 言い回し、表記（漢字とカタカナ、日本語名と記号など）、詳しさの違いは問わない
3. 問われた事項が模範解答と異なる場合、または答えを特定できない場合は「不正解」とする
4. 答えを複数挙げ、その中に模範解答と異なるものがある場合は「不正解」とする

出力は「正解」または「不正解」のどちらか一語だけとする"""

def prompt_rubric(it):
    return f"{RUBRIC}\n\n質問: {it['q']}\n模範解答: {it['ref']}\n回答: {it['answer']}"

def parse_verdict(out):
    """採点者の出力を 1（正解）、0（不正解）に直す（どちらとも読めなければ None）"""
    if "不正解" in out or "悪い" in out:
        return 0
    if "正解" in out or "良い" in out:
        return 1
    return None

assert parse_verdict("不正解です") == 0 and parse_verdict("正解") == 1 and parse_verdict("判断できない") is None

def run_judge(llm, make_prompt, items=JUDGE_SET):
    outs = chat(llm, [make_prompt(it) for it in items], max_new_tokens=8)
    return [parse_verdict(o) for o in outs]

print(prompt_rubric(JUDGE_SET[2]))

## 採点者を検証する: 一致率とCohenのκ

採点者の判定と人の判定を比べる

最も単純な指標は**一致率**（同じ判定をした割合）であるが、これには落とし穴がある

- 答案の9割が不正解である場合、何も読まずにすべて「不正解」と答える採点者でも、一致率は90%になる

そこで、**偶然でも起きる一致**を差し引いた指標を使う
これが**Cohenのκ（カッパ）係数**である

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

- $p_o$: 実際の一致率
- $p_e$: 2人の採点者が、それぞれの判定の割合を保ったまま無関係に採点した場合に、偶然一致する確率
  - 二値の場合、$p_e = P_1(\text{正解})P_2(\text{正解}) + P_1(\text{不正解})P_2(\text{不正解})$

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-kappa.png" width=600>

上の図は、一致率を0から1の帯で表したものである
偶然でも一致する分（$p_e$）を除いた残りのうち、実際にどれだけ一致したかの割合がκである

κの読み方は次のとおりである

- $\kappa = 1$: 完全に一致
- $\kappa = 0$: 偶然と同じ程度の一致
- $\kappa < 0$: 偶然よりも一致しない

κは、もともと人間の採点者どうしの一致を測るために作られた指標である
人どうしでもκが1になることは少なく、どの値から十分とみなすかは課題による
目安として、0.6を下回る採点者の結果をそのまま信じるのは危険である

In [ ]:
# Cohenのκを実装し、3種類の採点プロンプトを2つの採点者で比べる
def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = np.mean(a == b)
    pe = sum(np.mean(a == c) * np.mean(b == c) for c in np.unique(np.concatenate([a, b])))
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

assert abs(cohen_kappa([1, 1, 0, 0], [1, 0, 0, 0]) - 0.5) < 1e-9
assert cohen_kappa([0] * 9 + [1], [0] * 10) == 0.0  # すべて「不正解」と答える採点者は、一致率が9割でもκは0

verdicts, rows = {}, []
for jname, judge in [("large", LLM_L), ("small", LLM_S)]:
    for pname, make in [("vague", prompt_vague), ("ref", prompt_ref), ("rubric", prompt_rubric)]:
        raw = run_judge(judge, make)
        v = np.array([0 if x is None else x for x in raw])  # 判定を取り出せなかったものは不正解として扱う
        verdicts[(jname, pname)] = v
        rows.append({"judge": jname, "prompt": pname, "agreement": np.mean(v == human), "kappa": cohen_kappa(v, human),
                     "judged_correct": v.mean(), "unparsed": sum(x is None for x in raw)})
print(f"人の判定で正解の割合: {human.mean():.3f}")
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 結果の読み方

- `judged_correct` は、採点者が「正解（良い）」と判定した割合である
  - 人の判定で正解の割合と比べると、採点者が**甘い**のか**厳しい**のかがわかる
- 模範解答を与えない `vague` は、どちらの採点者でもκが最も低い
  - 採点者は回答の正誤を自分の知識で決めるしかない
  - 小モデルは、すべての回答を「良い」と判定している
    - 一致率は5割近くあるが、κは0である
    - 何も読まずに全部「良い」と答えるのと同じであり、採点者として機能していない
- 模範解答を与える（`ref`）と、κが大きく上がる
  - 採点が「知識を問う課題」から「2つの文を照合する課題」に変わり、易しくなるからである
- 採点基準を詳しく書いた `rubric` は、筆者の実行では `ref` を上回らなかった
  - 大モデルでは、判定が厳しくなり（`judged_correct` が下がり）、κはわずかに下がった
  - 小モデルでは、ほとんどの回答を「正解」と判定するようになり、κは大きく下がった
  - 長い指示は、それを読んで適用する力の無いモデルには負担になる
  - 「基準を詳しく書けば良くなる」とは限らず、**採点プロンプトを変えたら、必ずκを測り直す**
- 採点者として使えるのは、採点対象の課題を十分にこなせるモデルである
  - 小モデルを採点者にした行は、どのプロンプトでもκが低い

一致率とκを見比べると、一致率だけでは採点者の質を過大に評価しやすいこともわかる

In [ ]:
# 最も一致した採点者について、人の判定との食い違いを読む
best = max(verdicts, key=lambda k: cohen_kappa(verdicts[k], human))
v = verdicts[best]
print("採点者:", best, " κ =", round(cohen_kappa(v, human), 3))
print(pd.crosstab(pd.Series(human, name="human"), pd.Series(v, name="judge")))
print()
for it, h, j in zip(JUDGE_SET, human, v):
    if h != j:
        print(f"人={h} 採点者={j} | {it['q']} | 模範解答: {it['ref']} | 回答: {it['answer'][:60]!r}")

### 食い違いを読む

κの値を見るだけで終わらせず、**食い違った事例を1件ずつ読む**

食い違いには3種類ある

- **採点者の誤り**: 模範解答と明らかに異なる回答を正解とした、またはその逆
- **採点基準の不備**: 基準に書かれていない境界事例で、判定が割れた
  - 基準を書き足して、再び測る
- **人の誤り**: 読み直すと、人のラベルのほうが間違っていた
  - 人手ラベルも誤りを含む
  - 可能であれば2人以上で採点し、人どうしのκも測っておく

採点者の誤りの方向にも注目する

- 不正解を正解とする誤り（甘い採点）が多ければ、評価結果は実際より高く出る
- 2つのシステムを比べる目的であれば、甘さが両方に同じだけ働くかぎり、順位は保たれる
- 絶対的な水準（「正解率90%以上で出荷する」など）を判断する目的では、甘い採点者は使えない

## 採点者を大きくするとどうなるか

小モデルは採点者として使えず、大モデルは模範解答を与えれば使える、という結果であった

3Bモデルを採点者にして、同じ答案集を採点させる

In [ ]:
# 3Bモデルを採点者にして、同じ3種類の採点プロンプトで人の判定との一致を測る
rows = []
for pname, make in [("vague", prompt_vague), ("ref", prompt_ref), ("rubric", prompt_rubric)]:
    raw = run_judge(LLM_XL, make)
    v = np.array([0 if x is None else x for x in raw])
    verdicts[("3B", pname)] = v
    rows.append({"judge": "3B", "prompt": pname, "agreement": np.mean(v == human), "kappa": cohen_kappa(v, human),
                 "judged_correct": v.mean(), "unparsed": sum(x is None for x in raw)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 採点者の大きさとκの関係を1枚の図にまとめる
kap = pd.DataFrame({j: {p: cohen_kappa(verdicts[(j, p)], human) for p in ["vague", "ref", "rubric"]}
                    for j in ["small", "large", "3B"]}).T
ax = kap.plot.bar(figsize=(6, 3), rot=0)
ax.set_xlabel("judge model")
ax.set_ylabel("Cohen's kappa vs human")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### 結果の読み方

- **模範解答を与えない `vague` で、最も大きな違いが出る**
  - 筆者の実行では、κは 0（小）→ 0.51（大）→ 0.78（3B）と上がった
  - 模範解答が無い場合、採点者は自分の知識で正誤を決める
  - 知識の多いモデルほど、自力で正しく判定できる
- **模範解答を与える `ref` では、大モデルと3Bモデルに差が無い**
  - どちらもκは0.86である
  - 模範解答との照合という易しい課題では、1.5Bの時点で頭打ちになっている
  - 残った食い違いは、境界事例（余分な誤りを含む回答など）であり、採点者を大きくしても解消しない
- **詳しい採点基準（`rubric`）の害は、3Bモデルでは現れない**
  - 小モデルではκが大きく下がり、大モデルではわずかに下がったが、3Bモデルでは `ref` と同じ値である
  - 長い指示を読みこなせるモデルでは、基準を詳しく書いても悪化しない

### 詳しい採点基準が効かなかった理由

採点基準を明確に書くほど人との一致が上がる、というのが一般に言われていることであり、この節の前半でもそう説明した

それにもかかわらず、どの採点者でも `rubric` は `ref` を上回らなかった

- 小モデルでは、長い指示を読んで適用する力が足りず、かえって悪化したと考えられる
- 大モデルと3Bモデルでは、`ref` の時点でκが0.86に達しており、改善の余地が小さかった
  - 残った食い違いは数件であり、72件の答案集では、κの0.03程度の差は誤差の範囲である
- 答案が1文の事実の回答であり、もともと判定に迷う境界事例が少なかった
  - 採点基準の効果は、長い文章や、複数の観点が絡む課題で現れやすい

「採点基準は不要である」という結論にはならない
この答案集と、この規模のモデルでは、効果を確認できなかった、というのが正確な読み方である

採点者が大きくなるほど、**採点プロンプトの書き方による違いが小さくなる**

逆に言えば、採点者に小さく安いモデルを使う場合ほど、模範解答を与えること、採点プロンプトを検証することが重要になる

## ペア比較

採点には、1つの回答に点を付ける**単独採点（pointwise）**のほかに、2つの回答を見せてどちらが良いかを選ばせる**ペア比較（pairwise）**がある

- 絶対的な点数を付けるよりも、2つを比べるほうが判断しやすい
  - 人による評価でも同じ性質が知られている
- 模範解答を用意しにくい課題（文章の読みやすさ、要約の質など）でも使える
- 2つのシステムのどちらが良いかを直接知りたいときに向く
- 多数のシステムを比べるときは、総当たりの勝敗から順位を計算する

ペア比較には、特有の偏りがある

- **位置バイアス（position bias）**: 回答の内容に関係なく、先に示した方（または後に示した方）を選びやすい
- **自己選好（self-preference）**: 採点者が、自分自身（または同系統のモデル）の出力を高く評価しやすい
- **長さへの偏り**: 長く詳しい回答を、内容の正誤とは別に高く評価しやすい

位置バイアスは、**同じ組を順序を入れ替えて2回採点する**ことで測定でき、打ち消すこともできる

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-position-bias.png" width=680>

上の図は、位置バイアスのある採点者の例である
順序を入れ替えた2回とも「A」を選んでおり、1回目は回答x、2回目は回答yを選んだことになる

次のセルでは、大モデルの回答と小モデルの回答の組（36組）を、順序を入れ替えて2回ずつ採点させる

採点には、選択式ベンチマークと同じく、次のトークンが「A」か「B」かの確率を使う

In [ ]:
# ペア比較: 同じ組を順序を入れ替えて2回採点する
PAIR_INST = "質問に対する2つの回答を、模範解答に照らして比べよ\n正しく答えている方を選び、AまたはBの1文字だけで答えよ"

def pair_prompt(q, ref, first, second):
    return f"{PAIR_INST}\n\n質問: {q}\n模範解答: {ref}\n回答A: {first}\n回答B: {second}"

def pair_judge(llm):
    """各組について、(大モデルの回答を先に示したときにAを選ぶ確率, 小モデルの回答を先に示したときにAを選ぶ確率) を返す"""
    p_lf, p_sf = [], []
    for first_is_large, store in [(True, p_lf), (False, p_sf)]:
        prompts = []
        for qid, ans_l, _, ans_s, _ in ANSWERS:
            x, y = (ans_l, ans_s) if first_is_large else (ans_s, ans_l)
            prompts.append(pair_prompt(QMAP[qid]["q"], QMAP[qid]["options"][0], x, y))
        lp = next_token_logprobs(llm, prompts, ["A", "B"])
        store.extend(np.exp(lp[:, 0]) / np.exp(lp).sum(axis=1))
    return np.array(p_lf), np.array(p_sf)

h_l = np.array([a[2] for a in ANSWERS])   # 大モデルの回答に対する人の判定
h_s = np.array([a[4] for a in ANSWERS])   # 小モデルの回答に対する人の判定
clear = h_l != h_s                        # 人の判定で、一方だけが正解の組
print(f"一方だけが正解の組: {clear.sum()} 組（大モデルだけ正解 {int((h_l > h_s).sum())}、小モデルだけ正解 {int((h_s > h_l).sum())}）")

pair_res, rows = {}, []
for jname, judge in [("large", LLM_L), ("small", LLM_S)]:
    p_lf, p_sf = pair_judge(judge)
    pair_res[jname] = (p_lf, p_sf)
    first_rate = np.mean(np.concatenate([p_lf, p_sf]) > 0.5)         # 先に示した回答（A）を選んだ割合
    consistent = np.mean((p_lf > 0.5) == (p_sf < 0.5))               # 順序を入れ替えても同じ回答を選んだ割合
    pref_large = (p_lf + (1 - p_sf)) / 2                             # 2つの順序で平均した、大モデルの回答を選ぶ確率
    acc_clear = np.mean((pref_large[clear] > 0.5) == (h_l[clear] > h_s[clear]))
    rows.append({"judge": jname, "chose_first": first_rate, "consistent": consistent,
                 "acc_on_clear_pairs": acc_clear, "prefers_large(all)": np.mean(pref_large > 0.5),
                 "prefers_large(tie pairs)": np.mean(pref_large[~clear] > 0.5)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 結果の読み方

- `chose_first` は、先に示した回答（A）を選んだ割合である
  - すべての組を両方の順序で示しているので、偏りのない採点者なら50%になる
  - 50%から離れているほど、位置バイアスが強い
- `consistent` は、順序を入れ替えても同じ回答を選んだ割合である
  - これが低い採点者は、内容ではなく位置で選んでいる
  - **順序を入れ替えると判定が変わる組は、「引き分け」または「判定不能」として扱う**のが安全である
- `acc_on_clear_pairs` は、人の判定で一方だけが正解である組について、採点者が正しい方を選べた割合である
  - 2つの順序の確率を平均してから判定しているため、位置バイアスは打ち消されている
- `prefers_large` は、大モデルの回答を選んだ割合である
  - `tie pairs` は、人の判定で両方とも正解、または両方とも不正解の組だけで集計したものである
  - これらの組では、正誤の観点ではどちらを選ぶ理由も無い
  - それでも一方に偏るなら、採点者は正誤以外の何か（文体、長さ、自分の出力との近さ）で選んでいる

筆者の実行では、どちらの採点者にも極端な位置バイアスが現れた

- 小モデルは、すべての組で先に示した回答（A）を選んだ
- 大モデルは、8割以上の組で後に示した回答（B）を選んだ
- 順序を入れ替えても同じ回答を選んだ組は、1割に満たない

つまり、この2つの採点者によるペア比較は、1回の採点だけでは**回答の内容ではなく、示した順序を答えているにすぎない**

2つの順序の確率を平均すると、大モデルは、人の判定で優劣が明らかな組の多くで正しい方を選べている
位置バイアスの下に、内容に基づく判断が弱く残っていることになる

順序を入れ替えずに「AとBのどちらが良いか」を1回だけ尋ねた評価結果は、信用できない

### 自己選好について

自己選好を調べるには、「大モデルが採点すると大モデルの回答が有利になり、小モデルが採点すると小モデルの回答が有利になるか」を見る

`prefers_large(tie pairs)` を2つの採点者で比べると、筆者の実行ではほとんど差が無く、自己選好は確認できなかった

ただし、この実験から「自己選好は無い」と結論することもできない

- 組の数が少なく、割合の誤差が大きい
- 位置バイアスが強すぎて、ほかの偏りが見えにくい
- 2つのモデルの回答は長さや文体が異なるため、偏りがあったとしても、「自分の出力だから」選んだのか、「短い回答だから」選んだのかを区別できない
- 2つのモデルは同じ系列であり、互いの出力が似ている

自己選好は、より大規模なモデルと多数の組を使った研究で報告されている（参考文献を参照）

実務では、**評価されるモデルと同じ系列のモデルを採点者に使わない**、または複数の系列の採点者で結果が変わらないことを確かめる、という対策を取る

自己選好については、この節の後半で、条件を変えてもう一度調べる

## ペア比較の偏りは、モデルを大きくすると減るか

大モデルと小モデルによるペア比較は、どちらも位置バイアスに支配されていた

3Bモデルを採点者に加える

In [ ]:
# 3Bモデルを採点者にしてペア比較を行い、位置バイアスを3つの採点者で比べる
def pair_summary(jname, p_lf, p_sf):
    pref_large = (p_lf + (1 - p_sf)) / 2
    return {"judge": jname,
            "chose_first": np.mean(np.concatenate([p_lf, p_sf]) > 0.5),
            "consistent": np.mean((p_lf > 0.5) == (p_sf < 0.5)),
            "acc_on_clear_pairs": np.mean((pref_large[clear] > 0.5) == (h_l[clear] > h_s[clear])),
            "prefers_large(tie pairs)": np.mean(pref_large[~clear] > 0.5)}

pair_res["3B"] = pair_judge(LLM_XL)
print(pd.DataFrame([pair_summary(j, *pair_res[j]) for j in ["small", "large", "3B"]]).round(3).to_string(index=False))

### 結果の読み方

- 位置バイアスは、3Bモデルで弱まる
  - 先に示した回答を選ぶ割合は、筆者の実行では 1.00（小）、0.15（大）、0.33（3B）であり、3Bモデルが最も0.5に近い
  - 順序を入れ替えても同じ回答を選んだ割合は、0 → 0.08 → 0.28 と上がる
- それでも、3Bモデルは7割の組で、順序を入れ替えると判定が変わる
  - 単独採点ではκが0.86に達したモデルでも、ペア比較では位置の影響のほうが大きい
- ペア比較で順序を入れ替えて2回採点することは、採点者の大きさによらず必要である

位置バイアスは、大規模なモデルを採点者にした場合にも報告されている（Zheng et al., 2023、Wang et al., 2024）

ただし、報告されているのは「順序を入れ替えると一部の判定が変わる」という程度であり、ここで見たような「ほぼすべての判定が位置で決まる」ほどではない

ここで極端に現れた理由として、次のことが考えられる

- 採点者が小型であり、2つの回答と模範解答を照合する処理そのものが難しい
- 回答が短く、内容の差を判断する手がかりが少ない
- 判定を、最初の1トークンが「A」か「B」かの確率だけで決めており、モデルがもともと持っている記号の出やすさの偏り（選択式の節で見たもの）が、そのまま現れる

## 自己選好を、条件を変えて調べ直す

先ほどの実験では、自己選好を確認できなかった

位置バイアスが強すぎたこと、模範解答を与えたために採点者が自分の考えを使う余地が小さかったことが、理由として考えられる

条件を変えて、3通りの方法で調べ直す

1. **単独採点での甘さ**: 位置の影響を受けない単独採点で、採点者は自分が書いた誤答を「正解」としやすいか
2. **模範解答を見せないペア比較**: 採点者が自分の知識だけで選ぶとき、自分の回答を選びやすいか
3. **実力の近い2つのモデル**: 大モデルの回答と3Bモデルの回答を、それぞれのモデル自身に比べさせる

まず1つ目を調べる

人の判定で不正解である回答だけを取り出し、採点者（`ref` のプロンプト）がそれを誤って「正解」とした割合を、回答を書いたモデルごとに求める

In [ ]:
# 自己選好の再検証1: 単独採点で、採点者は自分が書いた誤答に甘いか
wrong = human == 0
author = np.array([it["author"] for it in JUDGE_SET])
rows = []
for j in ["small", "large", "3B"]:
    v = verdicts[(j, "ref")]
    rows.append({"judge": j,
                 "false_accept(answers by large)": v[wrong & (author == "large")].mean(),
                 "false_accept(answers by small)": v[wrong & (author == "small")].mean()})
print(f"人の判定で不正解の回答: 大モデルの回答 {int((wrong & (author == 'large')).sum())}件、小モデルの回答 {int((wrong & (author == 'small')).sum())}件")
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 結果の読み方（再検証1）

自己選好があれば、採点者は**自分が書いた誤答**を、他のモデルが書いた誤答よりも多く「正解」とするはずである

- 小モデルは、大モデルの誤答も自分の誤答も、同じ4割を誤って正解としている
- 大モデルは、自分の誤答（7%）よりも、小モデルの誤答（14%）のほうを多く誤って正解としている
  - 自己選好とは逆の向きである
- 3Bモデルは、どちらの誤答も正解としていない

単独採点では、自己選好は見られない

次に2つ目を調べる

模範解答を見せずに、大モデルの回答と小モデルの回答を比べさせる
採点者は自分の知識で選ぶしかないため、自分の回答（＝自分が正しいと考えている内容）を選びやすいはずである

位置バイアスを打ち消すため、2つの順序の確率を平均してから、どちらを選んだかを決める

In [ ]:
# 自己選好の再検証2: 模範解答を見せないペア比較（採点者は自分の知識だけで選ぶ）
PAIR_INST_NOREF = "質問に対する2つの回答を比べよ\n正しく答えている方を選び、AまたはBの1文字だけで答えよ"
PAIR_Q = [QMAP[a[0]]["q"] for a in ANSWERS]
PAIR_REF = [QMAP[a[0]]["options"][0] for a in ANSWERS]
ANS_L, ANS_S = [a[1] for a in ANSWERS], [a[3] for a in ANSWERS]

def pair_probs(llm, first, second, with_ref=True):
    """first を回答A、second を回答Bとして示したときに、Aを選ぶ確率"""
    if with_ref:
        prompts = [pair_prompt(q, r, x, y) for q, r, x, y in zip(PAIR_Q, PAIR_REF, first, second)]
    else:
        prompts = [f"{PAIR_INST_NOREF}\n\n質問: {q}\n回答A: {x}\n回答B: {y}" for q, x, y in zip(PAIR_Q, first, second)]
    lp = next_token_logprobs(llm, prompts, ["A", "B"])
    return np.exp(lp[:, 0]) / np.exp(lp).sum(axis=1)

def prefer_x(llm, xs, ys, with_ref=True):
    """2つの順序について平均した、xs の側の回答を選ぶ確率"""
    return (pair_probs(llm, xs, ys, with_ref) + 1 - pair_probs(llm, ys, xs, with_ref)) / 2

JUDGES = [("small", LLM_S), ("large", LLM_L), ("3B", LLM_XL)]
differ = np.array([x != y for x, y in zip(ANS_L, ANS_S)])  # 2つの回答の文面が異なる組
rows = []
for j, llm in JUDGES:
    pref = prefer_x(llm, ANS_L, ANS_S, with_ref=False)
    m, lo, hi = bootstrap_ci(pref[differ] > 0.5)
    rows.append({"judge": j, "prefers_large": m, "ci_low": lo, "ci_high": hi, "mean_prob_large": pref[differ].mean(),
                 "acc_on_clear_pairs": np.mean((pref[clear] > 0.5) == (h_l[clear] > h_s[clear]))})
print(f"文面が異なる組: {int(differ.sum())}組   人の判定: 大モデルだけ正解 {int((h_l > h_s).sum())}、小モデルだけ正解 {int((h_s > h_l).sum())}")
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 結果の読み方（再検証2）

`prefers_large` は、文面の異なる32組のうち、採点者が大モデルの回答を選んだ割合である

- 筆者の実行では、小モデルが採点者のときに0.47、大モデルのときに0.63、第三者である3Bモデルのときに0.69であった
- 小モデルは、ほかの採点者よりも、大モデルの回答を選ぶ割合が低い
  - 向きだけを見れば、「小モデルは自分の回答を選びやすい」という自己選好と合っている

しかし、これを自己選好の証拠とすることはできない

- **信頼区間が重なっている**
  - 小モデルの信頼区間（0.31〜0.63）と、ほかの採点者の信頼区間は大きく重なる
- **小モデルは、内容を見て選んでいない**
  - 小モデルは、どの組でも先に示した回答を選んでおり、順序について平均した確率（`mean_prob_large`）はちょうど0.50である
  - 0.47という値は、0.5のまわりのごく小さな揺れを数えたものにすぎない
- **回答の質の差で説明できる**
  - 人の判定では、大モデルだけが正解の組が8、小モデルだけが正解の組が1である
  - 第三者が大モデルの回答を多く選ぶのは、実際に大モデルの回答のほうが正しいからである
  - 小モデルがそうしないのは、自分の回答を好むからではなく、正誤を見分ける力が無いからだと考えても説明がつく

最後に3つ目を調べる

実力の差が大きいと、選び方の違いが「好み」なのか「見分ける力」なのかを区別できない

そこで、実力の近い大モデル（1.5B）と3Bモデルの回答を、それぞれのモデル自身と、第三者である小モデルに比べさせる

- 3Bモデルの回答は、答案集と同じプロンプトでその場で生成する
  - 人手ラベルは付けていないので、どちらが正しいかは見ない
  - 採点者によって選び方が変わるかどうかだけを見る
- 模範解答は見せる

In [ ]:
# 自己選好の再検証3: 大モデルの回答と3Bモデルの回答を、それぞれのモデル自身に比べさせる
ANS_XL = chat(LLM_XL, [q + "\n1文で簡潔に答えよ" for q in PAIR_Q], max_new_tokens=48)  # 答案集と同じ条件で3Bモデルに回答させる
differ3 = np.array([x != y for x, y in zip(ANS_L, ANS_XL)])
rows = []
for j, llm in JUDGES:
    pref = prefer_x(llm, ANS_L, ANS_XL, with_ref=True)
    m, lo, hi = bootstrap_ci(pref[differ3] > 0.5)
    rows.append({"judge": j, "prefers_large(1.5B)": m, "ci_low": lo, "ci_high": hi, "mean_prob_large": pref[differ3].mean()})
print(f"文面が異なる組: {int(differ3.sum())}組")
print(pd.DataFrame(rows).round(3).to_string(index=False))
for q, a, b in list(zip(PAIR_Q, ANS_L, ANS_XL))[8:11]:
    print(f"\n質問: {q}\n  1.5B: {a!r}\n  3B  : {b!r}")

### 結果の読み方（再検証3）

表の値は、文面の異なる30組のうち、採点者が**大モデル（1.5B）の回答**を選んだ割合である

- 筆者の実行では、大モデルが採点者のときに0.57、3Bモデルが採点者のときに0.47であった
  - 大モデルは自分の回答を57%で選び、3Bモデルは自分の回答を53%で選んだことになる
  - どちらの採点者も、自分の回答をやや多く選んでおり、向きは自己選好と合っている
- しかし、その差（10ポイント）は、信頼区間の幅（およそ±17ポイント）よりも小さい
  - 偶然でも、この程度の差は生じる

### 自己選好についての結論

条件を変えて3通りに調べたが、**この実習の規模では、自己選好を確認できなかった**

- 単独採点では、まったく見られなかった
- ペア比較では、向きは自己選好と合うものの、誤差の範囲を出なかった

### 本来はどうなるはずか、なぜここでは出なかったのか

- **文献が示していること**
  - LLMの採点者は、自分の出力を他のモデルや人の出力よりも高く評価する傾向があると報告されている（Zheng et al., 2023、Panickssery et al., 2024）
  - その傾向は、採点者が「どれが自分の出力か」を見分けられるほど強いことも報告されている
- **ここで出なかった理由として考えられること**
  - **モデルの規模**: 報告は大規模なモデルについてのものであり、小型のモデルは、自分の出力を見分ける力が弱いと考えられる
  - **回答の形式**: 1文の事実の回答であり、モデルごとの文体の個性が出にくい（報告の多くは、要約のような長い文章を対象にしている）
  - **位置バイアス**: 小型のモデルでは位置の影響が非常に強く、ほかの偏りがあっても埋もれてしまう
  - **組の数**: 30組前後では、小さな偏りは誤差に埋もれる
  - **モデルの系列**: 3つとも同じ系列であり、互いの出力が似ているため、「自分」と「他者」の違いが小さい
- **断定できないこと**
  - これらのうち、どれが主な理由かは、この実験からは断定できない

これは「自己選好は存在しない」という意味ではない

- 30組の比較で検出できるのは、数十ポイントに及ぶ大きな差だけである
- 10ポイント程度の偏りがあるかどうかを判定するには、数百組が必要になる（評価データの節の「2点差は誤差か」と同じ計算である）

**「差が検出されなかった」ことと「差が無い」ことは違う**

評価の結果を報告するときは、何が出なかったかも、検出できる差の大きさと合わせて書く

自己選好は、より大規模なモデルと多数の組を使った研究で報告されている（参考文献を参照）
先に述べた実務上の対策（評価されるモデルと同じ系列のモデルを採点者に使わない、複数の系列の採点者で確かめる）は、自分の実験で確認できなかった場合でも取っておくのが安全である

## 外部APIのモデルを採点者にする（任意）

ここまでの実習では、鍵なしで動く小型モデルを採点者にした

実務では、評価対象よりも高性能なモデルを採点者に使うのが普通である

次のセルは、OpenAI互換のChat Completions API（`mlsys-text-M-LLM-2-Application.ipynb` を参照）で採点する例である

**このセルは実行不要（APIキーが必要）**
環境変数 `OPENAI_API_KEY` が設定されていなければ、何もせずに終了する

採点者を替えても、検証の手順は変わらない
同じ答案集を採点させ、人の判定とのκを測る

In [ ]:
# このセルは実行不要（APIキーが必要）: 外部APIのモデルを採点者にして、人の判定とのκを測る
import requests

API_URL = "https://api.openai.com/v1/chat/completions"
API_MODEL = "gpt-4o-mini"  # 利用できるモデル名に置き換える

def api_chat(prompt):
    r = requests.post(API_URL, timeout=60,
                      headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
                      json={"model": API_MODEL, "temperature": 0, "messages": [{"role": "user", "content": prompt}]})
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

if os.environ.get("OPENAI_API_KEY"):
    raw = [parse_verdict(api_chat(prompt_rubric(it))) for it in JUDGE_SET]
    v_api = np.array([0 if x is None else x for x in raw])
    print(f"一致率 {np.mean(v_api == human):.3f}  κ {cohen_kappa(v_api, human):.3f}")
else:
    print("OPENAI_API_KEY が設定されていないため、このセルは何もしない")

## LLMによる採点のまとめ

- 採点者は計測器であり、計測器には誤差がある
- 使う前に、人手ラベルの付いた数十〜数百件で一致率とκを測る
- 模範解答と、境界事例まで書いた採点基準を与える
- ペア比較では、順序を入れ替えて2回採点する
- 採点プロンプトや採点者のモデルを変えたら、検証をやり直す
- 評価結果を報告するときは、採点者のモデル名、採点プロンプト、人との一致の程度を添える

---

# RAGの評価

RAG（検索拡張生成）は、質問に関連する文書を**検索**し、その文書を根拠にLLMが回答を**生成**する仕組みである（構成は `mlsys-text-M-LLM-2-Application.ipynb` を参照）

RAGの回答が間違っていたとき、原因は2通りある

- **検索の失敗**: 答えの書かれた文書を取ってこられなかった
- **生成の失敗**: 文書は取れていたのに、LLMがそれを使わなかった、または読み違えた

最終的な回答だけを採点しても、どちらが原因かはわからない
原因がわからなければ、検索を直すべきか、プロンプトやモデルを直すべきかを決められない

そこで、RAGは**検索段と生成段を分けて評価する**

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-rag-stages.png" width=720>

上の図のとおり、検索段の指標は「上位k件の文書」に対して、生成段の指標は「回答」に対して計算する

| 段 | 指標 | 測っているもの | 必要な正解データ |
| --- | --- | --- | --- |
| 検索 | Recall@k | 関連文書のうち、上位k件に入った割合 | 質問ごとの関連文書 |
| 検索 | MRR | 最初の関連文書が何位に現れたか | 質問ごとの関連文書 |
| 検索 | nDCG@k | 関連の強い文書ほど上位にあるか | 質問ごとの関連文書と関連度 |
| 生成 | 正確さ | 回答が模範解答と合っているか | 模範解答 |
| 生成 | **忠実性（faithfulness）** | 回答の内容が、渡された文書に書かれているか | 不要 |
| 生成 | **関連性（answer relevance）** | 回答が質問に答えているか | 不要 |

## 文書集合と質問集合

架空の計算機室の利用案内を18件の短い文書として用意する

架空の内容にするのは、LLMが事前の知識で答えられないようにするためである
実在の事柄を題材にすると、文書を読まずに正解できてしまい、検索の効果を測れない

文書の中には、話題の近いものをわざと混ぜてある（GPUサーバとCPUサーバ、アカウント申請と一時アカウントなど）
紛らわしい文書が無いと、どんな検索器でも正解できてしまい、検索器の差を測れない

質問は18問あり、各質問に次の情報を付ける

- `rel`: 関連文書の番号と関連度（2は答えが直接書かれている、1は話題が関連している）
- `key`: 正解の回答に必ず含まれる語句
- 最後の2問は、**どの文書にも答えが書かれていない質問**である
  - 正しい振る舞いは「資料に記載なし」と答えることである
  - 文書に無いことを、もっともらしく作って答えないかを調べる

In [ ]:
# 架空の計算機室の利用案内（文書集合）と質問集合
DOCS = [
    "開室時間: 計算機室は平日の9時から21時まで開いている、土曜日は10時から17時まで、日曜日と祝日は閉室する",
    "GPUサーバの予約: GPUサーバは予約システムで前日の18時までに予約する、1回の予約は最長48時間で、延長は1回だけ認められる",
    "ジョブの上限: 1人が同時に実行できるジョブは4件までである、上限を超えたジョブは待ち行列に入り、空きが出た順に開始される",
    "ストレージ: ホーム領域の容量は1人あたり50GBである、大きなデータは共有領域 /data に置く、共有領域で90日間アクセスが無いファイルは自動で削除される",
    "バックアップ: ホーム領域は毎日午前3時にバックアップされ、30日分が保存される、共有領域 /data はバックアップされない",
    "アカウント申請: 利用アカウントは指導教員の承認を得て申請する、発行までは通常3営業日かかる",
    "パスワード: パスワードは12文字以上とし、180日ごとに変更する、5回続けて認証に失敗するとアカウントは30分間ロックされる",
    "印刷: 印刷は1人あたり月300枚まで無料である、上限を超えた分は1枚につき5円が研究室の予算から引かれる",
    "飲食: 計算機室での食事は禁止である、ふた付きの容器に入った飲み物だけ持ち込める",
    "障害連絡: サーバの障害に気づいたら、管理者のメーリングリスト admin@example.org に連絡する、夜間と休日の連絡への対応は翌営業日になる",
    "ソフトウェア: ソフトウェアの追加は各自がコンテナで行う、管理者権限が必要な導入は申請フォームから依頼し、審査に1週間ほどかかる",
    "卒業時の扱い: 卒業や修了のあと、アカウントは3月末から60日後に停止される、必要なデータはそれまでに各自で持ち出す",
    "会議室の予約: 会議室は予約システムで当日の9時までに予約する、1回の予約は最長2時間で、延長はできない",
    "一時アカウント: 来訪者向けの一時アカウントは即日発行され、14日後に自動で停止される",
    "夜間利用: 21時以降に計算機室を使う場合は、前日までに夜間利用届を出す、夜間に使えるのは24時までである",
    "貸出ノートPC: ノートPCは最長7日間借りられる、返却が遅れると次の貸出が30日間できなくなる",
    "CPUサーバ: CPUサーバは予約なしで使える、1つのジョブの実行時間は最長72時間である",
    "スキャナ: スキャナの利用は無料で、枚数の上限は無い、読み取ったデータは共有領域に7日間だけ保存される",
]
RAG_Q = [
    {"q": "土曜日は何時に閉まるか", "rel": {0: 2, 14: 1}, "key": "17時"},
    {"q": "GPUを使いたいとき、申し込みの締め切りはいつか", "rel": {1: 2}, "key": "18時"},
    {"q": "GPUを続けて使えるのは最長で何時間か", "rel": {1: 2, 16: 1}, "key": "48時間"},
    {"q": "同時に走らせられる計算はいくつまでか", "rel": {2: 2}, "key": "4件"},
    {"q": "自分専用の保存場所には何GBまで置けるか", "rel": {3: 2, 4: 1}, "key": "50GB"},
    {"q": "みんなで使う保存場所のファイルは、何日触らないと消されるか", "rel": {3: 2, 17: 1}, "key": "90日"},
    {"q": "消してしまったファイルは、何日前のものまで戻せるか", "rel": {4: 2}, "key": "30日"},
    {"q": "/data に置いたデータは控えが取られるか", "rel": {4: 2, 3: 1}, "key": "されない"},
    {"q": "利用を申し込んでから使えるようになるまで何日かかるか", "rel": {5: 2, 13: 1}, "key": "3営業日"},
    {"q": "パスワードは最低何文字必要か", "rel": {6: 2}, "key": "12文字"},
    {"q": "ログインに何回失敗すると締め出されるか", "rel": {6: 2}, "key": "5回"},
    {"q": "お金がかからずに刷れるのは月に何枚までか", "rel": {7: 2}, "key": "300枚"},
    {"q": "部屋に持って入れる飲み物の条件は何か", "rel": {8: 2}, "key": "ふた"},
    {"q": "サーバが落ちているときはどこに知らせればよいか", "rel": {9: 2}, "key": "admin@example.org"},
    {"q": "管理者権限が要るソフトウェアを入れてもらうには、どのくらい待つか", "rel": {10: 2}, "key": "1週間"},
    {"q": "卒業したあと、アカウントはいつ使えなくなるか", "rel": {11: 2, 13: 1}, "key": "60日"},
    {"q": "計算機室に3Dプリンタは置かれているか", "rel": {}, "key": None},
    {"q": "GPUサーバの利用料金は1時間あたりいくらか", "rel": {}, "key": None},
]
ANSWERABLE = [x for x in RAG_Q if x["rel"]]
print(len(DOCS), "文書 ", len(RAG_Q), "質問（うち答えのある質問", len(ANSWERABLE), "）")

## 検索段の指標

検索器は、質問に対して文書を関連が強いと判断した順に並べる

その順位表と、質問ごとの関連文書を比べる

Precision@k、Recall@k、MRRの考え方は `mlsys-text-J-NLP-1-Basics.ipynb` で導入した
ここでは、それらをRAGの検索段に当てはめ、関連度に段階がある場合のnDCGを加える

- **Hit@1**: 答えが直接書かれた文書（根拠文書）が1位にあるか
- **Recall@k**: 関連文書のうち、上位$k$件に含まれた割合
  - LLMに渡すのは上位$k$件だけであるから、ここに入らなかった文書は存在しないのと同じである
  - 関連文書が2件ある質問では、Recall@1は最大でも0.5にしかならないことに注意する
- **MRR（Mean Reciprocal Rank、平均逆順位）**: 最初の関連文書が$r$位に現れたとき、$1/r$ を点数とし、質問全体で平均する
  - 1位なら1、2位なら0.5、3位なら0.33となり、上位での順位の違いを重く見る
- **nDCG@k（normalized Discounted Cumulative Gain）**: 関連度に段階があるときに使う
  - $i$位の文書の関連度を $rel_i$ として、次の値を計算する

$$\mathrm{DCG@}k = \sum_{i=1}^{k}\frac{rel_i}{\log_2(i+1)}$$

  - 下位にある文書ほど、分母が大きくなって寄与が小さくなる
  - 理想的な並び（関連度の高い順）でのDCGで割り、0から1の範囲に正規化したものがnDCGである
  - 関連度の代わりに $2^{rel_i}-1$ を使い、関連の強い文書をさらに重く見る定義もある

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-rank-metrics.png" width=700>

上の図は、5件の順位表に対して3つの指標を計算した例である
根拠文書が2位、もう1件の関連文書が4位にある

In [ ]:
# 検索段の指標を実装する
def recall_at_k(rank, rel, k):
    return len(set(rank[:k]) & set(rel)) / len(rel)

def reciprocal_rank(rank, rel):
    for i, d in enumerate(rank):
        if d in rel:
            return 1 / (i + 1)
    return 0.0

def ndcg_at_k(rank, rel, k):
    dcg = sum(rel.get(d, 0) / math.log2(i + 2) for i, d in enumerate(rank[:k]))
    ideal = sum(g / math.log2(i + 2) for i, g in enumerate(sorted(rel.values(), reverse=True)[:k]))
    return dcg / ideal

# 動作確認: 関連文書が 7（関連度2）と 3（関連度1）のとき
_rel = {7: 2, 3: 1}
assert recall_at_k([7, 1, 3], _rel, 2) == 0.5 and reciprocal_rank([1, 3, 7], _rel) == 0.5
assert ndcg_at_k([7, 3, 1], _rel, 3) == 1.0 and ndcg_at_k([3, 7, 1], _rel, 3) < 1.0

## 2つの検索器を比べる

性質の異なる2つの検索器を用意する

- **文字の重なりによる検索（TF-IDF）**
  - 文書と質問を、連続する2文字（文字2-gram）の出現頻度で表し、コサイン類似度で並べる
  - TF-IDFは `mlsys-text-J-NLP-1-Basics.ipynb` を参照
  - 質問と文書で同じ語が使われていれば強い
- **埋め込みによる検索**
  - 自動指標の節で使った埋め込みモデルで、質問と文書をベクトルにして並べる
  - 言い換えに強いことが期待できる

質問は、利用者が実際に尋ねそうな言い回しで書いてあり、文書の表現とは一致しないものが多い（「認証に失敗」に対して「ログインに失敗」、「障害」に対して「落ちている」など）

In [ ]:
# 2つの検索器を作り、検索段の指標で比べる
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2, 2)).fit(DOCS)
doc_tfidf = tfidf.transform(DOCS)
doc_emb = embed(DOCS, prefix="passage: ")

def rank_tfidf(q):
    return list(np.argsort(-(tfidf.transform([q]) @ doc_tfidf.T).toarray()[0]))

def rank_emb(q):
    return list(np.argsort(-(embed([q], prefix="query: ") @ doc_emb.T)[0]))

def main_doc(x):
    """答えが直接書かれている文書（関連度が最も高い文書）の番号"""
    return max(x["rel"], key=x["rel"].get)

def eval_retriever(rank_fn):
    per_q = []
    for x in ANSWERABLE:
        rank = rank_fn(x["q"])
        per_q.append({"Hit@1": float(rank[0] == main_doc(x)), "Recall@2": recall_at_k(rank, x["rel"], 2),
                      "Recall@5": recall_at_k(rank, x["rel"], 5), "MRR": reciprocal_rank(rank, x["rel"]),
                      "nDCG@5": ndcg_at_k(rank, x["rel"], 5)})
    return pd.DataFrame(per_q)

ret_tfidf, ret_emb = eval_retriever(rank_tfidf), eval_retriever(rank_emb)
print(pd.DataFrame({"tfidf": ret_tfidf.mean(), "embedding": ret_emb.mean()}).round(3))
d, lo, hi = paired_diff_ci(ret_emb["nDCG@5"], ret_tfidf["nDCG@5"])
print(f"\nnDCG@5 の差（embedding − tfidf）: {d:+.3f}  95%信頼区間 [{lo:+.3f}, {hi:+.3f}]")

# 文字の重なりによる検索が、根拠文書を1位にできなかった質問
for x in ANSWERABLE:
    rank = rank_tfidf(x["q"])
    if rank[0] != main_doc(x):
        print(f"  {x['q']} -> 1位: {DOCS[rank[0]].split(':')[0]}（根拠文書: {DOCS[main_doc(x)].split(':')[0]}、{rank.index(main_doc(x)) + 1}位）")

### 結果の読み方

- 指標によって、検索器の差の見え方が違う
  - Recall@5は「上位5件のどこかに入っていればよい」ので、順位の違いに鈍い
  - Hit@1、MRR、nDCGは、1位か2位かの違いを区別する
  - LLMに1件しか渡さないならHit@1を、数件渡すならRecall@kを重視する、というように、**後段での使い方に合わせて指標を選ぶ**
- どの指標でも、埋め込みによる検索のほうが高い
- 差の信頼区間が0をまたぐかどうかを確認する
  - 質問は16問しか無いため、信頼区間は広い
  - ブートストラップ法は、平均で定義される指標であれば、正解率以外にもそのまま使える
- 文字の重なりによる検索が失敗した質問を読むと、原因がわかる
  - 質問と根拠文書で言い回しが異なる（「戻せる」と「バックアップ」、「落ちている」と「障害」）
  - 別の文書に、質問と同じ語が多く含まれている（「サーバ」を含む文書は複数ある）

ここでは2つの検索器を別々に比べたが、実際には両者を組み合わせるハイブリッド検索や、上位の候補を並べ直すリランキングが使われる
それらをMRRで比べる実習は、`mlsys-text-M-LLM-2-Application.ipynb` にある

検索段の評価は、LLMを1回も呼ばずに実行できる
速く安いため、チャンクの大きさや検索方式を調整するときは、まず検索段の指標だけで絞り込むとよい

## 生成段を評価する

次に、検索した文書をLLMに渡して回答を生成させる

生成段の実力を検索段と切り分けるため、同じ質問に対して、渡す文書を変えた4つの条件で回答させる

| 条件 | 渡す文書 | わかること |
| --- | --- | --- |
| `none` | 何も渡さない | LLMが事前の知識だけで答えられるか |
| `tfidf` | 文字の重なりによる検索の上位2件 | 実際のRAGの性能（検索器その1） |
| `embedding` | 埋め込みによる検索の上位2件 | 実際のRAGの性能（検索器その2） |
| `gold` | 正解の根拠文書（関連度2の文書） | 検索が完璧だった場合の、生成段の上限 |

- `gold` と、検索を使った条件（`tfidf`、`embedding`）の差が、**検索の失敗による損失**である
- `gold` でも間違える分が、**生成の失敗**である

正確さは、回答に `key` の語句が含まれるかどうかで判定する
答えの無い質問では、「記載なし」と答えられたら正解とする

In [ ]:
# 4つの条件で回答を生成し、正確さを採点する
RAG_INST = "次の資料だけを根拠にして、質問に1文で答えよ\n資料に答えが書かれていない場合は「資料に記載なし」とだけ答えよ"

def format_docs(ids):
    return "\n".join(f"[資料{k + 1}] {DOCS[i]}" for k, i in enumerate(ids))

def rag_prompt(q, ids):
    if not ids:
        return f"{q}\n1文で答えよ"
    return f"{RAG_INST}\n\n{format_docs(ids)}\n\n質問: {q}"

def is_correct(x, answer):
    a = normalize(answer)
    return "記載なし" in a if x["key"] is None else normalize(x["key"]) in a

TOP_K = 2
contexts = {
    "none": [[] for _ in RAG_Q],
    "tfidf": [rank_tfidf(x["q"])[:TOP_K] for x in RAG_Q],
    "embedding": [rank_emb(x["q"])[:TOP_K] for x in RAG_Q],
    "gold": [[main_doc(x)] if x["rel"] else rank_emb(x["q"])[:TOP_K] for x in RAG_Q],  # 答えの無い質問には検索結果を渡す
}
rag_out = {}
for cond, ctx in contexts.items():
    answers = chat(LLM_L, [rag_prompt(x["q"], ids) for x, ids in zip(RAG_Q, ctx)], max_new_tokens=64)
    rag_out[cond] = pd.DataFrame({"q": [x["q"] for x in RAG_Q], "answerable": [bool(x["rel"]) for x in RAG_Q],
                                  "answer": answers, "correct": [is_correct(x, a) for x, a in zip(RAG_Q, answers)]})
    df = rag_out[cond]
    print(f"{cond:10s} 正確さ（答えのある16問）: {df[df.answerable].correct.mean():.3f}"
          f"   答えの無い2問で「記載なし」と答えた数: {int(df[~df.answerable].correct.sum())}")

print()
for cond in contexts:
    print(f"[{cond}] {rag_out[cond].answer[10]!r}")

### 結果の読み方

- `none` では、ほとんど正解できない
  - 架空の内容であるから当然であり、評価データが「文書を読まなければ答えられない」ものになっていることの確認になる
  - 文書なしでも、モデルはもっともらしい数値を答える
- `tfidf`、`embedding`、`gold` の順に正確さが上がる
  - 検索が良くなるほど、最終的な回答も良くなる
- `gold` でも満点にはならない
  - 残りは生成段の問題である
- 表示した例（「ログインに何回失敗すると締め出されるか」）では、検索した2件を渡すと「資料に記載なし」と答え、根拠文書だけを渡すと正しく答えている
  - 余分な文書が1件混ざっただけで、答えられなくなることがある

## 忠実性と関連性

正確さの採点には、質問ごとの模範解答（ここでは `key`）が必要であった

実運用では、利用者の質問に対する模範解答は存在しない

そこで、**模範解答なしで測れる**2つの性質が使われる

- **忠実性（faithfulness）**: 回答に含まれる主張が、渡された文書から読み取れるか
  - 文書に無いことを述べていれば、それが偶然正しくても、忠実ではない
  - ハルシネーションの検出に対応する
- **関連性（answer relevance）**: 回答が、質問に対する答えになっているか
  - 「資料に記載なし」という回答は、忠実ではあるが、質問には答えていない

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-rag-triangle.png" width=620>

上の図のとおり、3つの指標は、回答を何と照らし合わせるかが違う
忠実性は渡した文書と、関連性は質問と、正確さは模範解答と照らす

この2つは**両立させるのが難しい関係**にある

- 常に「資料に記載なし」と答えるシステムは、忠実性が満点になるが、役に立たない
- 常に何かを答えるシステムは、関連性が高くなるが、文書に無いことを作って答える
- 片方だけを見ると、もう片方を犠牲にした改悪を見逃す

どちらも文字列の比較では測れないため、LLMに採点させる

In [ ]:
# 忠実性と関連性を、LLMに「はい」「いいえ」で判定させる関数
def faith_prompt(ids, answer):
    return ("資料と回答を読み、回答の内容がすべて資料から読み取れるかを判定せよ\n"
            "資料に書かれていない内容、または資料と食い違う内容が回答に含まれていれば「いいえ」、そうでなければ「はい」と一語で答えよ\n\n"
            f"{format_docs(ids)}\n\n回答: {answer}")

def relev_prompt(q, answer):
    return ("回答が質問に対する答えを述べているかを判定せよ\n"
            "質問とは別の事柄を述べている、または「記載なし」「わからない」のように答えを避けていれば「いいえ」、そうでなければ「はい」と一語で答えよ\n\n"
            f"質問: {q}\n回答: {answer}")

def yes_no(prompts, judge=LLM_L):
    """各プロンプトに対する判定を 1（はい）、0（いいえ）で返す"""
    outs = chat(judge, prompts, max_new_tokens=6)
    return np.array([0 if "いいえ" in o else int("はい" in o) for o in outs])

この2つの採点者を、結果を集計する前に点検する

In [ ]:
# 採点者の確認: 答えのわかっている入力を与える
g = rag_out["gold"][rag_out["gold"].answerable & rag_out["gold"].correct]   # 正解の根拠文書から作られた、正しい回答
g_ids = [[main_doc(RAG_Q[i])] for i in g.index]
g_q, g_ans = list(g.q), list(g.answer)

def shift(xs, k=3):
    """リストを k 個ずらす（別の質問の資料や質問と組み合わせるため）"""
    return xs[k:] + xs[:k]

def corrupt_numbers(text):
    """文中の数字をすべて別の数字に書き換える"""
    return re.sub(r"\d", lambda m: str((int(m.group()) + 3) % 10), text)

num = [bool(re.search(r"\d", a)) for a in g_ans]   # 数字を含む回答
print("忠実性の採点者が「はい」と答えた割合")
print(f"  陽性対照（正しい資料 + 正しい回答）            : {yes_no([faith_prompt(i, a) for i, a in zip(g_ids, g_ans)]).mean():.3f}")
print(f"  易しい陰性対照（無関係な資料 + 正しい回答）    : {yes_no([faith_prompt(i, a) for i, a in zip(shift(g_ids), g_ans)]).mean():.3f}")
hard = [faith_prompt(i, corrupt_numbers(a)) for i, a, m in zip(g_ids, g_ans, num) if m]
print(f"  難しい陰性対照（正しい資料 + 数字を変えた回答）: {yes_no(hard).mean():.3f}  （{len(hard)}件）")
print("    例:", corrupt_numbers(next(a for a, m in zip(g_ans, num) if m)))

print("関連性の採点者が「はい」と答えた割合")
print(f"  陽性対照（質問 + その質問への正しい回答）      : {yes_no([relev_prompt(q, a) for q, a in zip(g_q, g_ans)]).mean():.3f}")
print(f"  陰性対照（別の質問 + 同じ回答）                : {yes_no([relev_prompt(q, a) for q, a in zip(shift(g_q), g_ans)]).mean():.3f}")
print(f"  陰性対照（質問 + 「資料に記載なし」）          : {yes_no([relev_prompt(q, '資料に記載なし') for q in g_q]).mean():.3f}")

### 採点者の確認

LLMによる採点の節で学んだとおり、採点者は使う前に確かめる

ここでは人手ラベルを作る代わりに、**答えのわかっている入力**を採点者に与えた

- **陽性対照**: 「はい」と判定されるべき入力
  - 正解の根拠文書と、その文書から生成された正しい回答の組
- **陰性対照**: 「いいえ」と判定されるべき入力
  - 易しいもの: 同じ回答を、無関係な文書と組み合わせる
  - 難しいもの: 回答の中の数字だけを別の数字に書き換える
    - 自動指標の節で見た、「文面はほぼ同じで内容だけが違う」誤りである

陽性対照で「はい」が多く、陰性対照で「はい」が少なければ、採点者はその性質を区別できている

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-controls.png" width=650>

上の図は、忠実性の採点者に与えた3種類の入力と、期待される判定である

結果を見ると、2つの採点者の出来は大きく異なる

- **忠実性の採点者**は、陽性対照と陰性対照をはっきり区別している
  - 数字だけを書き換えた回答も「いいえ」と判定できている
  - 資料と回答を照合するだけの課題であり、小型のモデルでもこなせる
- **関連性の採点者**は、陰性対照に対しても「はい」と答えている
  - 別の質問と組み合わせても、「資料に記載なし」という回答でも、「はい」と判定してしまう
  - この採点者は、何を渡しても「はい」と言っているだけであり、**採点結果は使えない**

もし確認をせずに関連性の採点者を使っていたら、「関連性はどの条件でもほぼ満点」という、意味の無い結果を報告していたことになる

対照を使った確認は、人手ラベルを作る前の簡易な点検として、どの採点者にも使える

この採点者が失敗した理由と、モデルを大きくすれば解決するのかは、この節の終わりで3Bモデルを使って確かめる

ここでは、関連性について別の測り方を試す

## 関連性の別の測り方

LLMに直接「はい」「いいえ」を判定させる代わりに、次の手順を取る

1. 回答だけを見せて、「この文が答えになるような質問」をLLMに書かせる
2. 逆に生成した質問と、元の質問の埋め込み類似度を計算する
3. 回答が元の質問にきちんと答えていれば、逆に生成した質問は元の質問に近くなる

LLMには判定ではなく生成をさせ、判定は埋め込みの類似度で行う

「資料に記載なし」のように答えを避けた回答は、0点とする

In [ ]:
# 関連性の別の測り方: 回答から質問を逆に作らせ、元の質問との埋め込み類似度を見る
def answer_relevance(questions, answers):
    """回答ごとの関連性の点数と、逆に生成した質問を返す（答えを避けた回答は0点とする）"""
    gen_q = chat(LLM_L, [f"次の文が答えになるような質問文を1つだけ書け\n\n文: {a}" for a in answers], max_new_tokens=40)
    sim = (embed(list(questions)) * embed(gen_q)).sum(axis=1)
    avoid = np.array(["記載なし" in normalize(a) for a in answers])
    return np.where(avoid, 0.0, sim), gen_q

rel_pos, gen_q = answer_relevance(g_q, g_ans)
rel_neg, _ = answer_relevance(shift(g_q), g_ans)
for q, a, gq in list(zip(g_q, g_ans, gen_q))[:3]:
    print(f"元の質問: {q}\n  回答: {a}\n  逆に生成した質問: {gq}")
print(f"\n陽性対照（質問 + その質問への正しい回答）の平均: {rel_pos.mean():.3f}")
print(f"陰性対照（別の質問 + 同じ回答）の平均            : {rel_neg.mean():.3f}")
print(f"陽性対照のほうが点数が高い組の割合              : {np.mean(rel_pos > rel_neg):.3f}")

### 結果の読み方

- 陽性対照の点数は、陰性対照よりも高い
  - 組ごとに比べても、ほとんどの組で陽性対照のほうが高い
  - この測り方は、回答が質問に対応しているかを、少なくとも相対的には区別できている
- 点数の差は大きくない
  - 埋め込み類似度が狭い範囲に集まる性質は、ここでも現れている
  - 絶対値に意味は無く、同じ測り方で得た点数どうしを比べる
- 短い回答（「4件まで」「1週間」など）は、逆に生成される質問が曖昧になり、点数が低めに出る
  - 測り方の癖として知っておく

この確認を済ませたうえで、4つの条件を比べる

In [ ]:
# 4つの条件について、正確さ、忠実性、関連性を並べる
rows = []
for cond, ctx in contexts.items():
    df = rag_out[cond]
    df["relevance"], _ = answer_relevance(df.q, df.answer)
    # 資料を渡していない条件では、忠実性は定義できない
    df["faithful"] = np.nan if cond == "none" else yes_no([faith_prompt(ids, a) for ids, a in zip(ctx, df.answer)])
    df["abstain"] = ["記載なし" in normalize(a) for a in df.answer]
    a = df[df.answerable]
    rows.append({"condition": cond, "correct": a.correct.mean(), "faithful": a.faithful.mean(), "relevance": a.relevance.mean(),
                 "abstain": a.abstain.mean(), "abstain(unanswerable)": df[~df.answerable].abstain.mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 検索の成否と回答の正否の関係（答えのある質問）
for cond in ["tfidf", "embedding"]:
    r = rag_out[cond][rag_out[cond].answerable].copy()
    r["retrieval_hit"] = [main_doc(x) in ids for x, ids in zip(RAG_Q, contexts[cond]) if x["rel"]]
    print(f"\n[{cond}] 検索の成否と回答の正否（件数）")
    print(pd.crosstab(r.retrieval_hit, r.correct))

# 根拠文書を渡しても正解できなかった質問
gdf = rag_out["gold"]
for q, a in zip(gdf[gdf.answerable & ~gdf.correct].q, gdf[gdf.answerable & ~gdf.correct].answer):
    print(f"\n[gold で不正解] {q} -> {a!r}")

### 結果の読み方

- **`none`（文書なし）は、関連性が最も高く、正確さが最も低い**
  - どの質問にも、もっともらしい数値を挙げて答えるからである
  - 関連性だけを見ていると、文書を読まずに作り話をするシステムが最良に見える
  - これがハルシネーションであり、関連性と忠実性（または正確さ）を**組にして見る**理由である
- 文書を渡した条件では、正確さが大きく上がる
  - 一方で、答えのある質問にも「資料に記載なし」と答えることがある（`abstain` の列）
  - そのぶん関連性は下がる
  - 作り話をしなくなった代わりに、答えを控えすぎるようになった
- 検索器の違い（`tfidf` と `embedding`）は、最終的な正確さの違いとして現れる
  - 検索段の指標で見えた差が、生成段まで伝わっている
- 答えの無い2問では、文書を渡した条件はすべて「資料に記載なし」と答えられている

`gold`（根拠文書を渡した条件）でも正解できなかった質問は、**生成の失敗**である

- 根拠文書に「30日分が保存される」と書かれていても、「何日前のものまで戻せるか」という問いと結び付けられない
- 「バックアップされない」と書かれていても、「控えが取られるか」という問いと結び付けられない
- 文書の言い回しをそのまま抜き出すことはできても、言い換えを挟んだ推論は苦手である
- この種の失敗は、検索をいくら改良しても直らない

最後の表は、**誤答の原因の切り分け**である

| 検索 | 回答 | 解釈 | 直す場所 |
| --- | --- | --- | --- |
| 成功 | 正解 | 正常 | なし |
| 失敗 | 不正解 | 検索の失敗 | 検索器、チャンクの分け方、渡す件数 |
| 成功 | 不正解 | 生成の失敗 | プロンプト、モデル |
| 失敗 | 正解 | 偶然、または他の文書から答えられた | 評価データの関連文書の付け方を見直す |

RAGを改良するときは、まずこの表を作り、件数の多い欄から手を付ける

## 生成段のモデルを大きくすると何が変わるか

大モデル（1.5B）は、根拠文書を渡しても、言い換えを挟んだ質問に答えられなかった

検索はそのままにして、生成段だけを3Bモデルに替える

これにより、「検索を直すべきか、モデルを替えるべきか」という判断の材料が得られる

In [ ]:
# 生成段のモデルを3Bモデルに替えて、同じ4つの条件で正確さを比べる
rows = []
rag_out_XL = {}
for cond, ctx in contexts.items():
    answers = chat(LLM_XL, [rag_prompt(x["q"], ids) for x, ids in zip(RAG_Q, ctx)], max_new_tokens=64)
    df = pd.DataFrame({"q": [x["q"] for x in RAG_Q], "answerable": [bool(x["rel"]) for x in RAG_Q], "answer": answers,
                       "correct": [is_correct(x, a) for x, a in zip(RAG_Q, answers)],
                       "abstain": ["記載なし" in normalize(a) for a in answers]})
    rag_out_XL[cond] = df
    base = rag_out[cond]
    rows.append({"condition": cond,
                 "correct(1.5B)": base[base.answerable].correct.mean(), "correct(3B)": df[df.answerable].correct.mean(),
                 "abstain(1.5B)": base[base.answerable].abstain.mean(), "abstain(3B)": df[df.answerable].abstain.mean(),
                 "abstain_unanswerable(3B)": df[~df.answerable].abstain.mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 1.5Bでは使えなかった「はい」「いいえ」による関連性の採点者を、3Bモデルで点検し直す
print("\n関連性の採点者（3Bモデル）が「はい」と答えた割合")
print(f"  陽性対照（質問 + その質問への正しい回答）: {yes_no([relev_prompt(q, a) for q, a in zip(g_q, g_ans)], judge=LLM_XL).mean():.3f}")
print(f"  陰性対照（別の質問 + 同じ回答）          : {yes_no([relev_prompt(q, a) for q, a in zip(shift(g_q), g_ans)], judge=LLM_XL).mean():.3f}")
print(f"  陰性対照（質問 + 「資料に記載なし」）    : {yes_no([relev_prompt(q, '資料に記載なし') for q in g_q], judge=LLM_XL).mean():.3f}")

gx = rag_out_XL["gold"]
for q, a in zip(gdf[gdf.answerable & ~gdf.correct].q, gx[gdf.answerable & ~gdf.correct].answer):
    print(f"\n[1.5Bが gold で不正解だった質問] {q}\n  3Bモデルの回答: {a!r}")
for q, a in zip(gx[~gx.answerable].q, gx[~gx.answerable].answer):
    print(f"\n[答えの無い質問] {q}\n  3Bモデルの回答: {a!r}")

### 結果の読み方

- 文書を渡した3つの条件すべてで、3Bモデルのほうが正確さが高い
  - 答えのある質問に「資料に記載なし」と答えてしまう割合（`abstain`）も下がっている
- 1.5Bが根拠文書を渡されても答えられなかった2問のうち、1問は3Bモデルが正しく答えている
  - 「バックアップされない」という記述を、「控えが取られるか」という問いに結び付けられた
  - もう1問（「何日前のものまで戻せるか」）は、3Bモデルでも答えられない
- **検索器による差は、生成段を強くしても残る**
  - `tfidf` と `embedding` の差は、どちらのモデルでも12ポイント前後ある
  - 根拠文書が渡されていなければ、どれほど優れたモデルでも答えられない
  - 生成モデルの交換は、検索の失敗を補わない

### 関連性の採点者を3Bモデルで点検し直した結果

1.5Bによる「はい」「いいえ」の関連性の採点は、陰性対照にも「はい」と答えてしまい、使えなかった

- **本来はどうなるはずか**
  - 「回答が質問に答えているか」の判定は、LLMによる採点でよく使われる観点であり、大規模なモデルを採点者にすると、人の評価と相関する採点が得られることが報告されている（Liu et al., 2023 など）
  - したがって、採点者を大きくすれば使えるようになる、と期待される
- **3Bモデルでの結果**
  - 陰性対照には、ほぼ「いいえ」と答えられるようになった
  - ところが、陽性対照（正しく答えている回答）にも、6割以上「いいえ」と答えている
  - 「何でも『はい』」が「ほとんど『いいえ』」に変わっただけで、採点者としては依然として使えない
- **考えられる理由**
  - **肯定や否定への偏り**: LLMは、内容とは別に、「はい」または「いいえ」の一方に偏って答える傾向を持つことがあり、偏りの向きはモデルによって異なる
  - **採点プロンプトの書き方**: 「別の事柄を述べている、または答えを避けていれば『いいえ』」という条件は、2つの否定的な条件を組み合わせたものであり、小型のモデルには解釈が難しい
  - **回答が短い**: 「4件まで」のような短い回答は、質問に答えているかを回答だけからは判断しにくい
  - **模範となる例が無い**: 採点プロンプトに「はい」の例と「いいえ」の例を示していない
- **断定できないこと**
  - どれが主な原因かは、この実験からは断定できない
  - 3Bモデルでも使えなかったことから、少なくとも「モデルの規模だけの問題」ではないと言える

採点者を替えるたびに対照で点検する必要があること、点検に通らない採点者は使わないことを、この結果は示している

### 表の数値を信じる前に出力を読む

`abstain_unanswerable(3B)` の列は0.5であり、表だけを見ると「3Bモデルは、答えの無い質問の半分で作り話をした」と読める

しかし、表示した出力を読むと、3Bモデルは3Dプリンタの質問に対して「資料には情報が含まれていません」という趣旨の文で、正しく答えを控えている

- 採点は、回答に「記載なし」という文字列が含まれるかどうかで行っている
- 3Bモデルは、指示した定型句を使わずに、自分の言葉で同じ内容を述べた
- 文字列の一致で採点したために、正しい振る舞いが「不正解」と数えられた

自動指標の節で見た「言い換えを正解とできない」問題が、自分で作った採点規則の中で起きている

- 採点規則を作るときは、数件の出力を読み、規則が意図どおりに働いているかを確かめる
- 定型句の一致で判定するなら、判定に使う語を増やすか、LLMによる採点に替える
- 評価対象のモデルを替えたときは、出力の書き方が変わるため、**採点規則も点検し直す**

---

# エージェントの評価

エージェントは、LLMがツールを呼び出しながら、複数のステップをかけて課題を解くシステムである（構成は `mlsys-text-M-LLM-3-Agent.ipynb` を参照）

`mlsys-text-M-LLM-3-Agent.ipynb` の「エージェントの評価と安全性」では、評価指標の一覧（タスク達成率、効率性、ツール選択精度、コスト、再現性）を挙げた

エージェントの実装、失敗の型、行動を記録するトレーサについては、`mlsys-text-M-LLM-5-Agent設計.ipynb` で詳しく扱う

この節では、それらを実際に計測する

エージェントの評価が単発の質問応答より難しい理由は3つある

- **途中経過がある**: 最終結果が合っていても、途中で不要な操作や危険な操作をしているかもしれない
- **誤りが連鎖する**: 1つのステップの成功率が90%でも、5ステップ続けて成功する確率は $0.9^5 \approx 0.59$ に下がる
- **ばらつきが大きい**: 途中の小さな違いが後のステップを変えるため、同じ課題でも成功したり失敗したりする

## 評価用の小さなエージェント

外部APIを使わずに評価の実習を行うため、大モデルを使った最小限のエージェントを作る

- 題材: 架空の商店の店員を助ける
- ツール:
  - `price`: 商品名を渡すと、1個の値段を返す
  - `stock`: 商品名を渡すと、在庫の個数を返す
  - `calc`: 四則演算の式を渡すと、計算結果を返す
  - `finish`: 最終的な答えを渡して終了する
- LLMは毎回、`{"tool": "道具名", "arg": "引数"}` という形のJSONを1つ出力する
- プログラムがツールを実行し、結果を対話履歴に追記して、再びLLMに渡す
- これを `finish` が呼ばれるか、最大ステップ数に達するまで繰り返す

値段と在庫は架空の値であり、LLMは知らない
したがって、**ツールで調べずに答えた場合は、当て推量である**と判定できる

`calc` は、LLMが書いた式を計算する
LLMの出力をそのまま `eval` に渡すと、任意のコードを実行されるおそれがあるため、四則演算だけを受け付ける計算器を自作する

In [ ]:
# エージェントが使うツールを定義する
PRICE = {"りんご": 120, "バナナ": 80, "ノート": 150, "ペン": 90, "消しゴム": 60}
STOCK = {"りんご": 35, "バナナ": 50, "ノート": 12, "ペン": 40, "消しゴム": 0}

CALC_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
            ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv, ast.USub: operator.neg}

def safe_calc(expr):
    """四則演算だけを許す計算器（LLMの出力を eval に渡さないため）"""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in CALC_OPS:
            return CALC_OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in CALC_OPS:
            return CALC_OPS[type(node.op)](ev(node.operand))
        raise ValueError("使えない式")
    return ev(ast.parse(expr.replace("×", "*").replace("÷", "/"), mode="eval").body)

assert safe_calc("120*3 + 80*5") == 760 and safe_calc("1000 // 90") == 11
try:
    safe_calc("__import__('os').system('ls')")
    raise AssertionError("危険な式が通ってしまった")
except ValueError:
    pass

def call_tool(tool, arg):
    """ツールを実行し、(結果の文字列, 正しい呼び出しだったか) を返す"""
    if tool in ("price", "stock"):
        table = PRICE if tool == "price" else STOCK
        return (str(table[arg]), True) if arg in table else ("エラー: その商品は登録されていない", False)
    if tool == "calc":
        try:
            v = safe_calc(arg)
            return (str(int(v)) if float(v).is_integer() else f"{v:.2f}"), True
        except Exception:
            return "エラー: 式には数値と + - * / だけを使うこと", False
    return "エラー: その道具は存在しない", False

print(call_tool("price", "りんご"), call_tool("calc", "120*3"), call_tool("price", "みかん"), call_tool("search", "りんご"))

## 課題を定義する

エージェントの評価データは、課題ごとに次の情報を持つ

- `task`: 依頼文
- `answer`: 期待する最終結果
- `required`: 正しく解くために必ず調べなければならない情報（ツール名と引数の組）

`required` があることで、最終結果だけでなく、**途中の行動（軌跡、trajectory）**を採点できる

In [ ]:
# エージェントの評価用の課題
AGENT_TASKS = [
    {"id": "t0", "task": "りんご3個の代金はいくらか", "answer": 360, "required": [("price", "りんご")]},
    {"id": "t1", "task": "りんご2個とバナナ5個を買うと合計いくらか", "answer": 640, "required": [("price", "りんご"), ("price", "バナナ")]},
    {"id": "t2", "task": "ノートの在庫をすべて売ると売上はいくらになるか", "answer": 1800, "required": [("stock", "ノート"), ("price", "ノート")]},
    {"id": "t3", "task": "1000円でペンは最大何本買えるか", "answer": 11, "required": [("price", "ペン")]},
    {"id": "t4", "task": "ペン1本とノート1冊を買うと合計いくらか", "answer": 240, "required": [("price", "ペン"), ("price", "ノート")]},
    {"id": "t5", "task": "バナナの在庫は何個あるか", "answer": 50, "required": [("stock", "バナナ")]},
]

## エージェントの本体

エージェントの本体は、次の繰り返しである

1. 依頼と、これまでの行動と結果を、対話履歴としてLLMに渡す
2. LLMの出力からJSONを取り出す
3. `finish` であれば終了し、それ以外であればツールを実行して、結果を履歴に追加する

評価にかかる時間を短くするため、すべての課題と試行をまとめて1ステップずつ進める

実行のたびに、消費したトークン数、誤った呼び出しの回数、呼び出したツールの列を記録しておく
これらが、あとで採点の材料になる

In [ ]:
# エージェントの本体: LLMにJSONで行動を1つずつ出力させ、ツールを実行して記録に追記する
AGENT_SYS = """あなたは商店の店員を助けるエージェントである
道具を1回に1つずつ使い、依頼に答える
毎回、次のいずれかの形のJSONを1行だけ出力する

{"tool": "price", "arg": "りんご"}    商品1個の値段（円）を調べる
{"tool": "stock", "arg": "りんご"}    商品の在庫の個数を調べる
{"tool": "calc", "arg": "15*4+7"}     数値だけの四則演算の式を計算する
{"tool": "finish", "arg": "67"}       最終的な答えの数値を報告して終了する

規則:
- 商品名は、依頼文に書かれた日本語のまま渡す
- 値段と在庫は推測せず、必ず price と stock で調べる
- calc の式には、調べて得た数値だけを書く
- 同じ道具を同じ引数で2回呼ばない
- 記録の中に答えの数値があれば、すぐに finish で報告する"""

def agent_messages(ep):
    """依頼、これまでの行動（LLMの発話）、その結果（利用者側の発話）を対話履歴として並べる"""
    msgs = [{"role": "user", "content": f"依頼: {ep['task']['task']}"}]
    for action, obs in ep["log"]:
        msgs.append({"role": "assistant", "content": action})
        msgs.append({"role": "user", "content": f"結果: {obs}"})
    return msgs

def parse_action(text):
    """出力からJSONを取り出す（道具名と引数がそろっていなければ None）"""
    m = re.search(r"\{.*?\}", text, re.S)
    try:
        act = json.loads(m.group(0))
        return str(act["tool"]), str(act["arg"])
    except Exception:
        return None

def run_agent(llm, tasks, n_trials=1, temperature=0.0, max_steps=8):
    """各課題を n_trials 回ずつ実行する（すべての実行をまとめて1ステップずつ進める）"""
    eps = [{"task": t, "trial": k, "log": [], "calls": [], "answer": None, "done": False, "invalid": 0, "tokens": 0}
           for t in tasks for k in range(n_trials)]
    tok = llm["tok"]
    for _ in range(max_steps):
        active = [e for e in eps if not e["done"]]
        if not active:
            break
        prompts = [agent_messages(e) for e in active]
        outs = chat(llm, prompts, system=AGENT_SYS, max_new_tokens=40, temperature=temperature)
        for e, p, o in zip(active, prompts, outs):
            e["tokens"] += len(tok.encode(build_prompt(llm, p, AGENT_SYS))) + len(tok.encode(o))  # 入力と出力のトークン数（コストの目安）
            act = parse_action(o)
            if act is None:
                e["invalid"] += 1
                e["log"].append((o[:60], "エラー: tool と arg の両方を持つJSONを出力すること"))
            elif act[0] == "finish":
                e["log"].append((json.dumps({"tool": "finish", "arg": act[1]}, ensure_ascii=False), "終了"))
                e["answer"], e["done"] = act[1], True
            else:
                obs, ok = call_tool(*act)
                e["calls"].append(act)
                e["invalid"] += not ok
                e["log"].append((json.dumps({"tool": act[0], "arg": act[1]}, ensure_ascii=False), obs))
    return eps

# 貪欲法で各課題を1回ずつ実行し、2つの課題の軌跡を表示する
eps_greedy = run_agent(LLM_L, AGENT_TASKS)
for e in eps_greedy[1:3]:
    print("依頼:", e["task"]["task"], " 期待する答え:", e["task"]["answer"])
    for a, o in e["log"]:
        print("   ", a, "->", o)

## 何を採点するか

1回の実行（エピソード）について、次の項目を採点する

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-agent-trajectory.png" width=720>

上の図は、1回の実行の軌跡と、採点する箇所の対応である
この例は最終結果が正しいが、途中に不要な在庫の検索が1回含まれている

| 項目 | 内容 | 採点方法 |
| --- | --- | --- |
| `success` | 最終結果が期待する答えと一致したか | `finish` に渡した数値と `answer` の完全一致 |
| `grounded` | 必要な情報をすべてツールで調べたか | `required` の呼び出しがすべて軌跡に含まれるか |
| `steps` | 終了までのステップ数 | 記録の長さ |
| `invalid` | 誤ったツール呼び出しの回数 | JSONの形式違反、存在しないツール、存在しない商品、計算できない式 |
| `redundant` | 不要な呼び出しの回数 | 同じ呼び出しの繰り返し、`required` に無い商品の検索 |
| `tokens` | 消費したトークン数 | 入力と出力の合計 |

`success` と `grounded` を分けて見ることが重要である

- `success` だが `grounded` でない: 調べずに答えて、偶然当たった
  - 値段が変われば間違える
  - 最終結果だけを見る評価では、これを正解として数えてしまう
- `grounded` だが `success` でない: 必要な情報は調べたが、計算や最終回答で誤った

In [ ]:
# エピソードを採点する
def grade_episode(e):
    t = e["task"]
    required = set(t["required"])
    lookups = [c for c in e["calls"] if c[0] in ("price", "stock")]
    return {
        "task": t["id"], "trial": e["trial"],
        "success": e["answer"] is not None and extract_int(e["answer"]) == t["answer"],
        "grounded": required <= set(lookups),
        "finished": e["done"],
        "steps": len(e["log"]),
        "invalid": e["invalid"],
        "redundant": (len(e["calls"]) - len(set(e["calls"]))) + len(set(lookups) - required),
        "tokens": e["tokens"],
    }

g_greedy = pd.DataFrame([grade_episode(e) for e in eps_greedy])
print(g_greedy.to_string(index=False))
print(f"\n成功率 {g_greedy.success.mean():.3f}   平均ステップ数 {g_greedy.steps.mean():.2f}   平均トークン数 {g_greedy.tokens.mean():.0f}")

### 結果の読み方

- 課題ごとに、成否、軌跡の妥当性、ステップ数、コストが並ぶ
- 小型のモデルにとって、複数のステップを正しくつなぐ課題は難しく、成功する課題は一部にとどまる
- 上に表示した2つの軌跡を読むと、最終結果だけではわからないことが見える
  - 1つ目は正解しているが、依頼に関係の無い在庫を2回調べている（`redundant` に数えられる）
  - 2つ目は、必要な在庫と値段を正しく調べたのに、最後の式を誤っている（`grounded` だが `success` でない）
- `grounded` が `False` の実行は、必要な情報を調べないまま答えている
  - 値段を調べずに適当な数を式に入れる、商品名を英語に直して検索して失敗する、といった軌跡が見られる
- ステップ数が増えるほど、トークン数は急に増える
  - 毎ステップ、それまでの履歴をすべて入力し直すからである
  - 同じ成功率であれば、ステップ数の少ないエージェントのほうが安く、速い
- 最大ステップ数に達して終了できなかった実行は、`finished` が `False` になる
  - 最大ステップ数は、評価のときも必ず設定する（設定しないと、評価が終わらず、費用も際限なく増える）

ここまでは各課題を1回しか実行していない

## 複数回実行したときのばらつき

実際のエージェントはサンプリングを使って動くことが多く、同じ課題でも結果が変わる

各課題を、`temperature=0.7` で5回ずつ実行する

In [ ]:
# 各課題を5回ずつ実行し、成功率のばらつきを調べる
N_TRIALS = 5
torch.manual_seed(SEED)
eps_multi = run_agent(LLM_L, AGENT_TASKS, n_trials=N_TRIALS, temperature=0.7)
g_multi = pd.DataFrame([grade_episode(e) for e in eps_multi])

per_task = g_multi.groupby("task").agg(success=("success", "mean"), grounded=("grounded", "mean"),
                                       steps=("steps", "mean"), invalid=("invalid", "mean"),
                                       redundant=("redundant", "mean"), tokens=("tokens", "mean"))
print(per_task.round(2))

m, lo, hi = bootstrap_ci(g_multi.success)
all_k = g_multi.groupby("task").success.all().mean()
any_k = g_multi.groupby("task").success.any().mean()
print(f"\n全実行での成功率: {m:.3f}  95%信頼区間 [{lo:.3f}, {hi:.3f}]（{len(g_multi)}回）")
print(f"{N_TRIALS}回すべて成功した課題の割合: {all_k:.3f}   {N_TRIALS}回のうち1回でも成功した課題の割合: {any_k:.3f}")
print("\n成否と軌跡の妥当性（回数）")
print(pd.crosstab(g_multi.success, g_multi.grounded))

ax = per_task.success.plot.bar(figsize=(6, 3), rot=0, color="tab:blue")
ax.set_ylabel(f"success rate over {N_TRIALS} runs")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### 結果の読み方

- 同じ課題でも、実行によって成功したり失敗したりする
  - 筆者の実行では、課題ごとの成功率は0割から8割までばらついた
  - 1回だけ実行して「成功した」「失敗した」と報告しても、その課題についての性能はわからない
- 「5回すべて成功した課題の割合」と「1回でも成功した課題の割合」には大きな開きがある
  - 筆者の実行では、前者は0、後者は8割を超えた
  - 前者は**信頼性**を表す
    - 利用者が同じ依頼を何度しても正しく動くか
    - 業務で使うエージェントでは、こちらが重要である
  - 後者は**能力の上限**を表す
    - うまくいけば解けるか
    - 何度も試して最良の結果を選べる用途（コード生成で、試験に通ったものだけを採用するなど）では、こちらが意味を持つ
  - 同じエージェントが、数え方しだいで「まったく使えない」とも「ほとんどの課題を解ける」とも報告できてしまう
  - どちらを報告しているかを明記する

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-agent-reliability.png" width=620>

上の図は、4つの課題を5回ずつ実行した結果の模式的な例である
同じ結果を「すべて成功」で数えると4分の1、「1回でも成功」で数えると4分の3になる

- 成否と軌跡の妥当性の表を見る
  - 失敗のおよそ半分は、「必要な情報を調べたのに失敗した」実行である
    - 情報の取得まではできており、そのあとの計算や最終回答で誤っている
    - この部分を改良するには、ツールの選び方ではなく、調べた値の使い方を直す必要がある
  - 残りの半分は、必要な情報を調べないまま答えた実行である
  - 「成功したが、必要な情報を調べていない」実行があれば、それは当て推量である
  - 失敗を種類に分けると、次に何を直すべきかが決まる
- 課題ごとの成功率は5回の実行から求めたものであり、それ自体の誤差が大きい
  - 5回中0回や5回中5回のように端に寄った割合には、ウィルソンの区間を使う（`mlsys-text-T-安全性と公平性.ipynb` を参照）
- 全実行での成功率の信頼区間は、課題が6種類しか無いことを考えると、実際にはこれより広い
  - 同じ課題の5回の実行は互いに似ているため、30回が独立な30問であるかのように扱うと、誤差を小さく見積もってしまう
  - 課題の種類を増やすことが、実行回数を増やすことよりも重要である

### 軌跡の評価の方法

この実習では、必要な呼び出しが含まれているかを規則で判定した

軌跡の評価には、ほかに次の方法がある

- **模範の軌跡との比較**: 人が作った正しい手順と、ツール呼び出しの列を照合する
  - 正しい手順が1通りでない課題では、厳しすぎる判定になる
- **最終状態の検査**: 途中の手順は問わず、環境が期待する状態になったか（ファイルが作られたか、試験が通るかなど）を調べる
  - 手順の自由度が高い課題に向く
  - コーディングエージェントのベンチマークは、多くがこの方式である
- **LLMによる採点**: 軌跡の全文を採点者に読ませ、採点基準に沿って判定させる
  - 不要な操作や危険な操作のように、規則で書きにくい観点を扱える
  - 採点者の検証が必要であることは、前の節と同じである

安全性（禁止された操作をしていないか）も軌跡から評価する
`mlsys-text-M-LLM-3-Agent.ipynb` で挙げたリスクのうち、過剰な権限の行使やコストの暴走は、最終結果だけを見る評価では検出できない

## もう少し成功する構成と比べる

大モデル（1.5B）のエージェントは、成功率が4割に届かず、5回すべて成功する課題は無かった

課題、ツール、プロンプト、採点方法はそのままにして、LLMだけを3Bモデルに替える

エージェントの評価では、こうして**1つの要素だけを替えて比べる**ことが基本になる
複数の要素を同時に替えると、どれが効いたのかがわからなくなる

In [ ]:
# 同じ課題、同じ枠組みで、LLMだけを3Bモデルに替えて5回ずつ実行する
torch.manual_seed(SEED)
eps_multi_XL = run_agent(LLM_XL, AGENT_TASKS, n_trials=N_TRIALS, temperature=0.7)
g_multi_XL = pd.DataFrame([grade_episode(e) for e in eps_multi_XL])

def agent_summary(name, g):
    by_task = g.groupby("task").success
    return {"model": name, "success": g.success.mean(), "all_5": by_task.all().mean(), "any_of_5": by_task.any().mean(),
            "grounded": g.grounded.mean(), "finished": g.finished.mean(), "steps": g.steps.mean(),
            "invalid": g.invalid.mean(), "redundant": g.redundant.mean(), "tokens": g.tokens.mean()}

print(pd.DataFrame([agent_summary("large(1.5B)", g_multi), agent_summary("3B", g_multi_XL)]).round(2).to_string(index=False))
print("\n成否と軌跡の妥当性（3Bモデル、回数）")
print(pd.crosstab(g_multi_XL.success, g_multi_XL.grounded))

cmp = pd.DataFrame({"large(1.5B)": g_multi.groupby("task").success.mean(), "3B": g_multi_XL.groupby("task").success.mean()})
ax = cmp.plot.bar(figsize=(6.5, 3), rot=0)
ax.set_ylabel(f"success rate over {N_TRIALS} runs")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

# 3Bモデルが失敗した実行の軌跡を1つ表示する
for e, ok in zip(eps_multi_XL, g_multi_XL.success):
    if not ok:
        print("依頼:", e["task"]["task"], " 期待する答え:", e["task"]["answer"])
        for a, o in e["log"]:
            print("   ", a, "->", o)
        break

### 結果の読み方

筆者の実行では、次のようになった

- **成功率は 0.37 → 0.53 に上がった**
- **信頼性は大きく変わった**
  - 5回すべて成功した課題の割合は、0 → 0.5 に上がった
  - 一方、1回でも成功した課題の割合は、0.83 → 0.67 に下がった
  - 図を見ると、3Bモデルは「できる課題は5回とも成功し、できない課題は5回とも失敗する」傾向にある
  - 1.5Bは「どの課題も、ときどき成功する」
  - 平均の成功率の差（16ポイント）よりも、この**ばらつき方の違い**のほうが、利用者にとっては大きな違いである
- **効率も良くなった**
  - 平均ステップ数、誤った呼び出し、不要な呼び出しがいずれも減った
  - ステップ数が減ったため、1回の実行で消費するトークン数も減っている
  - 大きいモデルは1トークンあたりの計算や料金が高いが、少ないステップで終われば、総額では逆転することがある
  - コストは、単価ではなく**課題を1件成功させるのにかかる総額**で比べる
- **悪化した課題がある**
  - 課題 `t1`（りんごとバナナの合計金額）は、1.5Bでは5回中4回成功していたが、3Bモデルでは1回も成功していない
  - 表示した軌跡を読むと、値段（`price`）ではなく在庫（`stock`）を調べ、その数で計算している
  - 平均では改善していても、個々の課題では退行が起きている
- **失敗の種類が変わった**
  - 3Bモデルの失敗の多くは、必要な情報を調べていない（`grounded` でない）実行である
  - 1.5Bでは、調べたあとの計算で誤る失敗が半分を占めていた
  - モデルを替えると、次に直すべき箇所も変わる

### 課題 `t1` の退行: 本来はどうなるはずか、なぜここでは違うのか

- **本来の期待**: 上位のモデルは、下位のモデルが解ける課題を同じように解ける
- **実際によく起きること**: モデルを新しい版や大きい版に替えたとき、平均は上がっても一部の入力で悪化することは珍しくない
  - だからこそ、モデルの交換は回帰試験の対象になる
- **ここで退行した理由として考えられること**
  - **試行回数が少ない**: 5回中4回と5回中0回の差は偶然だけでは起きにくいが、5回では成功率の推定の幅が広い
    - 5回中0回であっても、真の成功率が3〜4割ある可能性は否定できない
    - ブートストラップ法は、0回や全回成功のときに幅0の区間を返してしまい、この不確かさを表せない
    - 件数が少なく、割合が0や1に近いときは、ウィルソンの区間を使う（`mlsys-text-T-安全性と公平性.ipynb` を参照）
  - **ツールの説明とプロンプト**: `price` と `stock` は同じ形の説明で並んでおり、依頼文には「2個」「5個」という個数が現れる
    - 表示した軌跡では、3Bモデルは個数の表現から在庫を調べるほうへ進んでいる
    - 説明文の書き方を変えれば結果が変わる可能性がある
  - **サンプリング**: `temperature=0.7` で生成しており、最初の1手が違うと、その後の軌跡全体が変わる
- **断定できないこと**
  - どの要因が原因かは、この実験からは断定できない
  - 確かめるには、試行回数を増やす、ツールの説明文を書き換えて同じ課題を再実行する、失敗した軌跡をすべて読む、といった追加の実験が要る

「モデルを大きくしたら良くなった」という一言でまとめると、`t1` の退行と、失敗の種類の変化を見落とす

平均が上がった変更の中に隠れた悪化を見つける仕組みが、次の節の回帰試験である

---

# 回帰試験として回す

ここまでで、評価データ、採点器、誤差の見積もりという部品がそろった

最後に、これらを**繰り返し自動で回す仕組み**にまとめる

LLMを使ったシステムでは、次のような変更が頻繁に起きる

- プロンプトの文面を直す
- モデルを新しい版に替える、または安いモデルに替える
- 検索の方式や、渡す文書の件数を変える
- 生成の設定（温度、最大トークン数）を変える

どの変更も、ある入力では改善し、別の入力では悪化する

数件を手で試して「良くなった」と判断すると、試さなかった入力での悪化を見逃す

**回帰試験（regression test）**は、変更のたびに同じ評価を自動で実行し、前回の結果と比べて悪化を検出する仕組みである

ソフトウェア開発の自動試験と同じ考え方であるが、LLMの場合は次の点が異なる

- 合否が確率的であり、「全件合格」を条件にできない
  - 前回との**差**を、誤差の幅と比べて判定する
- 平均が変わらなくても、個々の事例の合否は入れ替わる
  - 合格から不合格に変わった事例を一覧にし、内容を確認する
- 絶対に間違えてはならない事例がある
  - 平均とは別に、**必ず合格すべき事例**を指定し、1件でも落ちたら不合格とする

## 枠組みの設計

小さな枠組みを、3つの関数で作る

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-regression-flow.png" width=720>

上の図は、3つの関数の関係である
変更後の実行結果と、保存しておいた基準の結果を `compare_runs` が比べ、判定を返す

| 関数 | 役割 |
| --- | --- |
| `run_suite` | 評価データの全事例を、指定した設定（モデル、システムプロンプト）で実行し、事例ごとの合否を返す |
| `save_run`、`load_run` | 実行結果を設定と一緒にJSONファイルへ保存し、読み戻す |
| `compare_runs` | 2つの実行結果を比べ、差の信頼区間、合否が入れ替わった事例、判定を返す |

評価データの各事例は、次の4つを持つ

- `id`: 事例の識別子（前回の結果と突き合わせるために必要）
- `group`: 層の名前
- `prompt`: 入力
- `grade`: 出力を受け取り、合否を返す関数

ここでは、計算問題200問と選択式問題40問を合わせた240事例を評価データとする

In [ ]:
# 回帰試験の評価データ（スイート）を組み立てる
def build_suite():
    suite = []
    for d in ARITH:
        suite.append({"id": d["id"], "group": d["kind"], "prompt": PROMPT_V1.format(expr=d["expr"]),
                      "grade": lambda out, ans=d["answer"]: extract_int(out) == ans})
    for i, m in enumerate(MC):
        order, pos = mc_layout(i)
        suite.append({"id": m["id"], "group": "mc-" + m["cat"], "prompt": mc_prompt(m, order),
                      "grade": lambda out, gold=LETTERS[pos]: extract_choice(out, strict=False) == gold})
    return suite

SUITE = build_suite()
assert len({c["id"] for c in SUITE}) == len(SUITE)  # 識別子が重複していないこと
print(len(SUITE), "事例")

次に、実行、保存、比較の3つの関数を定義する

`compare_runs` の判定は、次の規則による

- 必ず合格すべき事例が1件でも落ちた、または差の信頼区間が0より下にある: 「悪化」
- 差の信頼区間が0より上にある: 「改善」
- それ以外: 「差は誤差の範囲」

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-flip-table.png" width=560>

`compare_runs` は、判定とは別に、事例ごとの合否の入れ替わりを数える
上の図の右上（合格→不合格）が退行、左下（不合格→合格）が修正である

In [ ]:
# 実行、保存、比較の3つの関数
def run_suite(suite, llm, system=None, max_new_tokens=16, note=""):
    outs = chat(llm, [c["prompt"] for c in suite], system=system, max_new_tokens=max_new_tokens, batch_size=32)
    passed = {c["id"]: bool(c["grade"](o)) for c, o in zip(suite, outs)}
    return {"config": {"model": llm["name"], "system": system, "max_new_tokens": max_new_tokens, "note": note},
            "accuracy": float(np.mean(list(passed.values()))), "passed": passed}

def save_run(run, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, ensure_ascii=False, indent=1)

def load_run(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def compare_runs(base, new, suite=SUITE, critical=()):
    """基準の実行 base と新しい実行 new を比べ、報告を辞書で返す"""
    ids = [c["id"] for c in suite]
    b = np.array([base["passed"][i] for i in ids], dtype=float)
    n = np.array([new["passed"][i] for i in ids], dtype=float)
    diff, lo, hi = paired_diff_ci(n, b)
    regressed = [i for i, x, y in zip(ids, b, n) if x and not y]   # 合格から不合格に変わった事例
    fixed = [i for i, x, y in zip(ids, b, n) if y and not x]       # 不合格から合格に変わった事例
    broken = [i for i in critical if not new["passed"][i]]         # 必ず合格すべき事例のうち、落ちたもの
    groups = pd.DataFrame({"group": [c["group"] for c in suite], "base": b, "new": n}).groupby("group", sort=False).mean()
    if broken or hi < 0:
        verdict = "悪化"
    elif lo > 0:
        verdict = "改善"
    else:
        verdict = "差は誤差の範囲"
    return {"base": b.mean(), "new": n.mean(), "diff": diff, "ci": (lo, hi), "regressed": regressed,
            "fixed": fixed, "broken_critical": broken, "by_group": groups, "verdict": verdict}

def print_report(name, rep):
    lo, hi = rep["ci"]
    print(f"=== {name} ===")
    print(f"正解率 {rep['base']:.3f} -> {rep['new']:.3f}   差 {rep['diff']:+.3f}  95%信頼区間 [{lo:+.3f}, {hi:+.3f}]")
    print(f"合格→不合格 {len(rep['regressed'])}件   不合格→合格 {len(rep['fixed'])}件   必須事例の不合格 {len(rep['broken_critical'])}件")
    print(f"判定: {rep['verdict']}")

## 基準の結果を保存する

まず、現在の設定で実行し、**基準（baseline）**として保存する

- 結果は、設定（モデル名、システムプロンプトなど）と一緒に保存する
  - どの設定で得た結果かが後からわからなければ、比較に使えない
- 保存するのは、平均の点数だけでなく、**事例ごとの合否**である
  - 平均だけでは、対応のある比較も、合否が入れ替わった事例の特定もできない

あわせて、必ず合格すべき事例を決める

ここでは、基準の実行で合格した2桁の加算（`add2`）を、必ず合格すべき事例とする
最も基本的な機能が壊れていないことを確かめる試験にあたる

In [ ]:
# 基準の実行結果を保存し、必ず合格すべき事例を決める
base_run = run_suite(SUITE, LLM_L, note="baseline")
save_run(base_run, "eval_runs/baseline.json")
base_run = load_run("eval_runs/baseline.json")  # 保存したファイルから読み戻せることも確かめる

CRITICAL = [c["id"] for c in SUITE if c["group"] == "add2" and base_run["passed"][c["id"]]]
print(f"基準の正解率: {base_run['accuracy']:.3f}   必ず合格すべき事例: {len(CRITICAL)}件")

## 変更を加えて比べる

3種類の変更を試す

| 名前 | 変更の内容 | 変更する人の期待 |
| --- | --- | --- |
| `persona` | システムプロンプトに「正確さを重んじる助手である」という一文を加える | 害は無く、少し良くなるかもしれない |
| `tutor` | システムプロンプトで「考え方を説明しながら答えよ」と指示する | 考えてから答えるので、良くなるかもしれない |
| `small-model` | モデルを小モデルに替える | コストは下がるが、精度も下がりそうである |

実行する前に、それぞれの判定がどうなるかを予想してみよ

In [ ]:
# 3種類の変更について回帰試験を実行する
candidates = {
    "persona": dict(llm=LLM_L, system="あなたは正確さを重んじる助手である"),
    "tutor": dict(llm=LLM_L, system="あなたは親切な家庭教師である\n考え方を説明しながら答えよ"),
    "small-model": dict(llm=LLM_S, system=None),
}
reports = {}
for name, cfg in candidates.items():
    run = run_suite(SUITE, cfg["llm"], system=cfg["system"], note=name)
    save_run(run, f"eval_runs/{name}.json")
    reports[name] = compare_runs(base_run, run, critical=CRITICAL)
    print_report(name, reports[name])
    print()

判定だけでは、何が起きたのかはわからない

`tutor` について、層ごとの変化と、選択式で合格から不合格に変わった事例の出力を見る

In [ ]:
# 層ごとの変化と、合格から不合格に変わった事例を確認する
name = "tutor"
print(reports[name]["by_group"].assign(diff=lambda d: d.new - d.base).round(3))

case_by_id = {c["id"]: c for c in SUITE}
cfg = candidates[name]
sample_ids = [i for i in reports[name]["regressed"] if i.startswith("mc")][:3]  # 選択式で不合格に変わった事例
outs = chat(cfg["llm"], [case_by_id[i]["prompt"] for i in sample_ids], system=cfg["system"], max_new_tokens=16)
for i, o in zip(sample_ids, outs):
    print(f"\n[{i}] {case_by_id[i]['prompt'].splitlines()[2]}\n  出力: {o!r}")

### 結果の読み方

筆者の実行では、次の結果になった

- **`persona`: 平均の差は誤差の範囲だが、判定は「悪化」**
  - 正解率の差の信頼区間は0をまたいでおり、平均だけを見れば「変化なし」である
  - しかし、必ず合格すべき事例が1件落ちたため、「悪化」と判定された
  - 「合格→不合格」が15件、「不合格→合格」が7件あり、害の無さそうな一文でも、個々の事例の合否は20件以上入れ替わっている
- **`tutor`: 判定は「差は誤差の範囲」だが、中身は大きく変わっている**
  - 層ごとの表を見ると、計算問題の層は改善し、選択式の層は大きく悪化している
  - 改善と悪化が打ち消し合って、平均がほとんど動いていないだけである
  - 選択式で不合格に変わった事例の出力を見ると、記号で答えずに説明の文章を書き始めている
    - 知識が失われたのではなく、**出力の形式が、採点の前提と合わなくなった**
  - 同じ指示が、計算問題では有利に、選択式では不利に働いている
- **`small-model`: 明確な悪化**
  - 差の信頼区間が0より下にあり、50件以上が合格から不合格に変わっている

### 予想と違った点について

- **`tutor` は、計算問題を改善し、選択式を悪化させた**
  - 「考え方を説明しながら答えよ」という指示は、途中経過を書かせる指示であり、計算のような課題で正解率を上げることが知られている（Wei et al., 2022）
    - 計算問題の層が改善したのは、この一般的な傾向のとおりである
  - 選択式の層が悪化したのは、能力が下がったからではない
    - 出力例のとおり、モデルは正しい内容（「北海道」など）を文章で述べているが、記号を書く前に16トークンの上限で切れている
    - 最大トークン数を増やし、答えの取り出し方を合わせれば、悪化は解消すると考えられる（各自で確かめよ）
- **`persona` は、平均をほとんど変えなかったが、20件以上の合否を入れ替えた**
  - 役割を与える一文の効果は、課題やモデルによってまちまちであり、一貫して良くなるとは言えない
  - 内容に関係の無い一文でも、入力が変われば出力の確率分布が変わり、もともと際どかった事例の正誤が入れ替わる
  - どの事例が入れ替わるかに、はっきりした規則は見いだせず、この実験からは原因を断定できない

ここから、回帰試験の読み方について3つのことがわかる

1. **平均の差だけで判定してはならない**
   - 平均が変わらなくても、層ごと、事例ごとには大きく入れ替わっていることがある
   - 利用者から見れば、昨日まで正しく動いていた入力が、今日は動かなくなっている
   - 「合格→不合格」の件数と、層ごとの表を必ず確認する
2. **「誤差の範囲」は「変化が無い」ことの証明ではない**
   - 240事例で検出できるほど大きな平均の変化は無かった、という意味である
3. **落ちた事例の出力を読む**
   - 悪化の原因が、能力の低下なのか、出力の形式の変化なのかは、点数からはわからない
   - 形式の変化が原因であれば、最大トークン数や答えの取り出し方を合わせて直せば解決する

必ず合格すべき事例による判定は厳しい
確率的に動くシステムでは、内容と無関係な変更でも、1件が偶然落ちることがある

- 必須の事例は、本当に落ちてはならないものだけに絞る
- 落ちた場合は、落ちた事例の出力を読み、偶然か、再現する不具合かを確かめる

## 評価のコスト

評価そのものにも費用がかかる

`chat` 関数は、生成のたびに入力と出力のトークン数を `USAGE` に積算してきた

ここまでの実習全体で消費した量を確認する

In [ ]:
# ここまでの評価全体で消費したトークン数を確認する
total = USAGE["prompt"] + USAGE["completion"]
print(f"入力 {USAGE['prompt']:,} トークン   出力 {USAGE['completion']:,} トークン   合計 {total:,} トークン")
print("（chat 関数による生成の分だけを数えている、対数確率の計算と埋め込みは含まない）")

### 結果の読み方

- 小さな評価データばかりであったが、合計すると数十万トークンになる
  - 入力のほうが出力よりもはるかに多い
  - 評価では、短い答えを得るために、長い指示や文書を毎回入力するからである
- 外部APIを使う場合は、この量に単価を掛けたものが、評価を1回まわすたびにかかる
  - LLMによる採点を使えば、採点の分のトークンも加わる
  - エージェントは1件あたりの消費が特に多い
- 回帰試験は、変更のたびに何度も実行する
  - 評価データを大きくすれば誤差は小さくなるが、費用と時間は比例して増える
  - 「見分けたい差の大きさ」から必要な件数を決める、という評価データの節の考え方は、費用を抑えるためにも重要である

## 実務での運用

- 評価の実行を、コードの変更と同じ仕組み（継続的インテグレーション）に組み込む
  - プロンプトや設定を変更するたびに自動で実行し、判定が「悪化」であれば変更を取り込まない
- 評価データを2段階に分ける
  - 小さく速い集合: 変更のたびに実行する（数十〜数百件、数分以内）
  - 大きく網羅的な集合: 1日1回、または公開の前に実行する
- 実行結果を履歴として残す
  - 点数の推移を時系列で見られるようにする
  - 実験管理の道具（`mlsys-text-R-運用.ipynb` で扱ったwandbなど）に記録すると、設定と結果を一緒に管理できる
- 外部APIのモデルは、利用者が何も変更していなくても、提供者の更新によって振る舞いが変わることがある
  - 変更が無くても、定期的に回帰試験を実行する
- 回帰試験は、プロンプトの変更だけでなく、提供の方式を変えるときにも使う
  - モデルの量子化や、質問に応じて使うモデルを切り替えるルーティングは、速度と費用を下げる代わりに品質を変えうる
  - 変更後の品質を同じ評価データで測り、差を判定する（`mlsys-text-S-推論サービング.ipynb` を参照）
  - 評価データでの試験に通ったあと、一部の利用者だけに先に公開して様子を見るカナリア公開も、同じノートブックで扱っている
- モデルを追加学習（調整）したときも、調整の前後を同じ評価データで比べる（`mlsys-text-J-NLP-8-LLM調整.ipynb` を参照）
- 新しく見つかった失敗例は、評価データに追加する
  - 評価データを変えた場合は、基準も新しい評価データで取り直す

---

# 落とし穴

評価の仕組みを作ると、数値が手に入る

数値は説得力を持つため、その数値が何を表しているかを忘れやすい

最後に、評価を運用するなかで陥りやすい3つの落とし穴を確かめる

## 落とし穴1: 評価セットへの過適合

プロンプトを20通り試し、評価データで最も点数が高かったものを採用する

これは自然な手順であるが、採用したプロンプトの点数は、**実力よりも高く出ている**

- 評価データでの点数は、実力に偶然の上下が加わったものである
- 多数の候補から最高点を選ぶと、実力が高い候補だけでなく、偶然に上振れした候補も選ばれる
- 候補が多いほど、また評価データが小さいほど、上振れは大きくなる

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-selection-bias.png" width=620>

上の図は模式図である
真の実力が同じ20個の候補でも、評価データでの点数はばらつき、最高点の候補（赤）は実力よりかなり高く見える

`mlsys-text-2-ML基礎.ipynb` で学んだ、検証データとテストデータを分ける理由と同じである
ハイパーパラメータの代わりにプロンプトを選んでいるだけで、構造は変わらない

次のセルで、**実力がまったく同じ**（どれも真の正解率が0.70の）プロンプトを $K$ 個用意し、100問の評価データで最高点のものを選んだとき、その点数がいくつになるかを模擬する

In [ ]:
# 実力が同じK個の候補から最高点のものを選んだときの、見かけの点数を模擬する
rng = np.random.default_rng(SEED)
p_true, n_dev, trials = 0.70, 100, 5000
rows = []
for K in [1, 3, 10, 30, 100]:
    dev = rng.binomial(n_dev, p_true, size=(trials, K)) / n_dev  # K個の候補の、開発用データでの正解率
    best_dev = dev.max(axis=1)                                   # 選ばれた候補の、開発用データでの正解率
    fresh = rng.binomial(n_dev, p_true, size=trials) / n_dev     # 選ばれた候補を、新しいデータで測り直した正解率
    rows.append({"K": K, "selected score (dev)": best_dev.mean(), "same prompt on fresh data": fresh.mean()})
sel = pd.DataFrame(rows)
print(sel.round(3).to_string(index=False))

plt.figure(figsize=(6, 3))
plt.plot(sel.K, sel["selected score (dev)"], "o-", label="score used for selection")
plt.plot(sel.K, sel["same prompt on fresh data"], "s--", label="score on fresh data")
plt.xscale("log")
plt.xlabel("number of candidates K")
plt.ylabel("accuracy")
plt.legend()
plt.tight_layout()
plt.show()

### 結果の読み方

- 候補が1つのときは、評価データでの点数は実力（0.70）と一致する
- 候補を増やすと、選ばれた候補の点数は上がっていく
  - 30個から選ぶと、見かけの点数は実力より約9ポイント高くなる
- 同じ候補を新しいデータで測り直すと、点数は0.70に戻る
  - 候補の実力はすべて同じであるから、「最良の候補を選んだ」効果は存在しない
  - 上がって見えたのは、すべて偶然である

対処は、機械学習の基本と同じである

- 評価データを、**開発用（dev）**と**最終確認用（test）**に分ける
  - プロンプトの調整や候補の選択は、開発用だけで行う
  - 最終確認用は、採用するものを決めたあとに1回だけ使う
- 最終確認用のデータを何度も見て調整に使えば、それも開発用になってしまう
  - 定期的に新しい事例を追加し、入れ替える
- 開発用の評価データを大きくすれば、上振れは小さくなる

公開ベンチマークでも同じことが起きる
多くの研究者が同じベンチマークで手法を調整し、点数の高かったものが論文として公表されるため、分野全体がそのベンチマークに過適合していく

## 落とし穴2: 指標のハック（グッドハートの法則）

**グッドハートの法則（Goodhart's law）**は、「ある指標が目標にされると、その指標は良い指標ではなくなる」という経験則である

指標は、測りたい性質そのものではなく、その**代理**である

指標の値を上げること自体を目的にすると、測りたい性質を改善せずに指標だけを上げる方法が見つかり、採用されてしまう

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-goodhart.png" width=620>

上の図の2つの円は、測りたい性質を満たす回答と、指標が高くなる回答である
2つは重なるが一致はせず、指標だけを追うと、重なりの外（右側）へ進んでしまう

RAGの節で使った正確さの判定（回答に `key` の語句が含まれるか）を例に、指標を「攻略」してみる

In [ ]:
# 語句が含まれるかどうかで採点する指標を、内容の無い回答で攻略する
cheat = "、".join(x["key"] for x in ANSWERABLE)  # すべての質問の正解の語句を並べただけの回答
print("ずるい回答:", cheat)
print(f"語句による正確さ: {np.mean([is_correct(x, cheat) for x in ANSWERABLE]):.3f}")

# 別の観点の指標と組み合わせると、不正を検出できる
honest = list(rag_out["gold"][rag_out["gold"].answerable].answer)
docs_gold = [[main_doc(x)] for x in ANSWERABLE]
faith_cheat = yes_no([faith_prompt(ids, cheat) for ids in docs_gold])
faith_honest = yes_no([faith_prompt(ids, a) for ids, a in zip(docs_gold, honest)])
print(f"忠実性（根拠文書に照らして）: ずるい回答 {faith_cheat.mean():.3f}   通常の回答 {faith_honest.mean():.3f}")
print(f"回答の長さ（文字数）: ずるい回答 {len(cheat)}   通常の回答の平均 {np.mean([len(a) for a in honest]):.1f}")

### 結果の読み方

- 正解の語句をすべて並べただけの回答は、どの質問に対しても「正解」と判定され、語句による正確さは満点になる
  - 利用者にとっては、まったく役に立たない回答である
- この例は極端だが、同じ構造の現象は実際に起きる
  - 再現率に基づく指標は、長い回答ほど有利になる
    - 指標を上げようとすると、回答が冗長になっていく
  - LLMによる採点が長く丁寧な回答を好む場合、内容の正しさと関係なく、回答を長くすれば点が上がる
  - 選択式の評価で特定の記号が正解になりやすい場合、その記号を答えやすいモデルが高得点になる
- 人が意図して攻略しなくても、プロンプトを点数で選ぶ作業を繰り返せば、指標の弱点を突く方向に自然に進んでいく

対処は次のとおりである

- **性質の異なる複数の指標を組にして見る**
  - 正確さと長さ、関連性と忠実性のように、片方を攻略するともう片方が下がる組合せを選ぶ
  - 上のセルでは、忠実性や回答の長さを見れば、異常に気づける
- **定期的に出力を人が読む**
  - 指標の値が上がったとき、出力が本当に良くなったのかを、無作為に取り出した事例で確かめる
- **指標の弱点が見つかったら、採点方法を直す**
  - 評価の仕組みも、評価される側と同じように改良し続ける対象である

## 落とし穴3: ベンチマークの飽和

ベンチマークは、モデルの性能が上がるにつれて、モデルを見分ける力を失う
これを**飽和（saturation）**と呼ぶ

評価データの節で見た、層ごとの正解率を思い出そう

- 2桁の加算（`add2`）は、大モデルも小モデルもほぼ全問正解であった
- この層の問題を何問増やしても、2つのモデルの違いはわからない

ある問題がモデルの比較に役立つのは、**一方のモデルだけが正解する**場合である

<img src="https://class.west.sd.keio.ac.jp/dataai/text/eval-saturation.png" width=620>

上の図は模式図である
比べたいモデルの実力の範囲（黄色の帯）で曲線が平らになっているベンチマークは、実力の差を点数に反映しない

次のセルで、層ごとに、2つのモデルの正誤が分かれた問題の数を数える

In [ ]:
# 層ごとに、2つのモデルの正誤が分かれた問題（比較に役立つ問題）を数える
info = pd.DataFrame({
    "kind": res_L.kind,
    "both_correct": res_L.correct & res_S.correct,
    "both_wrong": ~res_L.correct & ~res_S.correct,
    "only_large": res_L.correct & ~res_S.correct,
    "only_small": ~res_L.correct & res_S.correct,
}).groupby("kind", sort=False).sum()
info["informative"] = info.only_large + info.only_small
print(info)

### 結果の読み方

- `add2` は、ほぼすべての問題で両方のモデルが正解しており、比較に役立つ問題がほとんど無い
  - この層は**飽和**している
- 比較に役立つ問題は、特定の層に集中している
- 200問あっても、モデルの違いについての情報を持つ問題は、その一部にすぎない

飽和には、もう1つ問題がある

- 正解率が100%に近づくと、残りの誤答の多くは、問題の側の誤り（正解ラベルの間違い、曖昧な問題文）になる
- 正解率が98%から99%に上がっても、それがモデルの進歩なのか、誤ったラベルに合わせただけなのかを区別できない

飽和したベンチマークで競っても、得られる情報は少ない

- ベンチマークの順位表を読むときは、上位のモデルの点数が上限に張り付いていないかを確認する
- 自分の評価データでも、ほぼ全問正解になった層は、より難しい事例に入れ替える
  - ただし、易しい事例にも役割はある
  - 回帰試験では、「以前からできていたことが、今もできるか」を確かめるために残しておく

## 3つの落とし穴に共通すること

どの落とし穴も、**点数を見ることと、実際の出力を見ることを切り離したとき**に起きる

- 点数が上がったら、何が良くなったのかを事例で確かめる
- 点数が下がったら、何が悪くなったのかを事例で確かめる
- 点数が変わらなくても、ときどき事例を読む

評価の自動化は、人が出力を読む作業をなくすためのものではなく、読むべき事例を絞り込むためのものである

---

# まとめ

本テキストでは、LLMとその応用システムを評価する技術を一通り学んだ

| 節 | 内容 | 核心 |
| --- | --- | --- |
| LLMの評価が難しい理由 | 正解の多様性、確率的な出力、プロンプトへの敏感さ、汚染 | 評価の手順全体が点数を決める |
| 評価データの作り方 | 設計、層別、ブートストラップ法、信頼区間 | 点数には誤差の幅を付け、差は対応のある比較で判定する |
| 自動指標 | 完全一致、F1、BLEU、ROUGE-L、埋め込み類似度、パープレキシティ | 文面の近さと内容の正しさは別である |
| 選択式ベンチマーク | 対数尤度による採点、生成と抽出、順序への敏感さ | 採点方法と選択肢の並びで点数が変わる |
| LLMによる採点 | 採点基準、ペア比較、位置バイアス、Cohenのκ | 採点者は、使う前に人の判定と照らして検証する |
| RAGの評価 | 検索段の指標、忠実性、関連性 | 検索と生成を分けて測り、誤答の原因を切り分ける |
| エージェントの評価 | 成否、軌跡、コスト、成功率のばらつき | 最終結果だけでなく途中の行動を見て、複数回実行する |
| 回帰試験 | 実行、保存、比較の枠組み | 変更のたびに回し、合否が入れ替わった事例を読む |
| 落とし穴 | 過適合、グッドハートの法則、飽和 | 点数と実際の出力を切り離さない |

評価に関する判断は、次の4つの問いに集約できる

1. **何を測っているか**: その指標は、知りたい性質のどの側面を、どの程度反映しているか
2. **誤差はどれくらいか**: 件数は十分か、差は信頼区間の外にあるか
3. **計測器は信用できるか**: 採点器、採点者、取り出しの規則を検証したか
4. **点数の裏で何が起きているか**: 実際の出力を読んだか

本テキストで使ったモデルは小型であり、偏りが大きかった

モデルを0.5B、1.5B、3Bと大きくしていくと、次のことが観察された

- 選択式での記号の偏り、ペア比較での位置バイアスは小さくなり、採点者としての人との一致は上がった
- しかし、どれも無くなりはしなかった
- プロンプトへの敏感さは減らず、プロンプトによっては大きいモデルが小さいモデルに負けた
- エージェントは、平均の成功率が上がる一方で、退行する課題もあった

ここで観察した現象は、さらに大規模なモデルでも、程度を変えて同じように現れる
モデルが良くなっても、評価の手順を省いてよい理由にはならない

**測れないものは改良できず、誤って測ったものは誤った方向に改良される**

---

# 課題1（評価データの設計と信頼区間）

自分で課題を1つ決め、評価データを作成して、2つの設定を比較せよ

1. 測りたい能力を1文で述べる
2. 3つ以上の層を持つ評価データを、合計60件以上作成する（プログラムによる生成でもよい）
3. 採点方法（出力から答えを取り出す規則を含む）を定め、採点関数を実装する
4. 2つの設定（モデルの違い、またはプロンプトの違い）で実行し、層ごとの正解率を表にする
5. それぞれの正解率の95%信頼区間と、差の95%信頼区間（対応のあるブートストラップ）を求める
6. 差があると言えるかを判断し、言えない場合は、あと何件あれば判断できそうかを見積もる

# 課題2（採点者の検証）

文章で答える課題について、LLMによる採点者を作成し、人の判定と比較せよ

1. 質問と模範解答の組を20問以上用意し、本テキストのモデル（または任意のモデル）に回答させて、回答を保存する
2. 保存したすべての回答に、自分で正誤を付ける（判定の基準を先に文章で書いておくこと）
3. 採点プロンプトを2種類以上作成し、それぞれについて一致率とCohenのκを求める
4. 人の判定と食い違った事例をすべて読み、原因を「採点者の誤り」「採点基準の不備」「人の誤り」に分類する
5. 分類の結果をもとに採点プロンプトを1回改良し、κがどう変わったかを報告する

# 課題3（RAGの評価と回帰試験）

自分で選んだ題材でRAGの評価データを作成し、設定の変更を回帰試験で判定せよ

1. 10件以上の文書と、15問以上の質問（関連文書と正解の語句を付けたもの）を用意する
2. 検索段の指標（Recall@k、MRR、nDCG@k）を2つ以上の検索の設定について求める
3. 生成段について、文書なし、検索結果、正解の根拠文書の3条件で正確さを求め、誤答を「検索の失敗」と「生成の失敗」に分ける
4. 本テキストの `run_suite`、`compare_runs` にならって、このRAG用の回帰試験を実装する
5. 設定を1つ変更し（渡す文書の件数、プロンプト、モデルなど）、回帰試験の報告（差の信頼区間、合否が入れ替わった事例、判定）を示す
6. 合格から不合格に変わった事例を2件以上取り上げ、出力を読んで原因を考察する

# 参考文献

- Papineni, K., Roukos, S., Ward, T., Zhu, W.-J. (2002). "BLEU: a Method for Automatic Evaluation of Machine Translation." ACL 2002.
- Lin, C.-Y. (2004). "ROUGE: A Package for Automatic Evaluation of Summaries." Text Summarization Branches Out (ACL Workshop).
- Rajpurkar, P., Zhang, J., Lopyrev, K., Liang, P. (2016). "SQuAD: 100,000+ Questions for Machine Comprehension of Text." EMNLP 2016.
- Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., Artzi, Y. (2020). "BERTScore: Evaluating Text Generation with BERT." ICLR 2020.
- Efron, B. (1979). "Bootstrap Methods: Another Look at the Jackknife." The Annals of Statistics.
- Koehn, P. (2004). "Statistical Significance Tests for Machine Translation Evaluation." EMNLP 2004.\n- Wei, J. et al. (2022). "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models." NeurIPS 2022.
- Card, D., Henderson, P., Khandelwal, U., Jia, R., Mahowald, K., Jurafsky, D. (2020). "With Little Power Comes Great Responsibility." EMNLP 2020.
- Miller, E. (2024). "Adding Error Bars to Evals: A Statistical Approach to Language Model Evaluations." arXiv.
- Cohen, J. (1960). "A Coefficient of Agreement for Nominal Scales." Educational and Psychological Measurement.
- Hendrycks, D., Burns, C., Basart, S., Zou, A., Mazeika, M., Song, D., Steinhardt, J. (2021). "Measuring Massive Multitask Language Understanding." ICLR 2021.
- Liang, P. et al. (2023). "Holistic Evaluation of Language Models." Transactions on Machine Learning Research.
- Sclar, M., Choi, Y., Tsvetkov, Y., Suhr, A. (2024). "Quantifying Language Models' Sensitivity to Spurious Features in Prompt Design." ICLR 2024.
- Pezeshkpour, P., Hruschka, E. (2024). "Large Language Models Sensitive to the Order of Options in Multiple-Choice Questions." Findings of NAACL 2024.
- Zheng, C., Zhou, H., Meng, F., Zhou, J., Huang, M. (2024). "Large Language Models Are Not Robust Multiple Choice Selectors." ICLR 2024.
- Zheng, L. et al. (2023). "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena." NeurIPS 2023 Datasets and Benchmarks Track.
- Liu, Y., Iter, D., Xu, Y., Wang, S., Xu, R., Zhu, C. (2023). "G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment." EMNLP 2023.
- Wang, P. et al. (2024). "Large Language Models are not Fair Evaluators." ACL 2024.
- Panickssery, A., Bowman, S. R., Feng, S. (2024). "LLM Evaluators Recognize and Favor Their Own Generations." NeurIPS 2024.
- Järvelin, K., Kekäläinen, J. (2002). "Cumulated Gain-Based Evaluation of IR Techniques." ACM Transactions on Information Systems.
- Es, S., James, J., Espinosa-Anke, L., Schockaert, S. (2024). "RAGAS: Automated Evaluation of Retrieval Augmented Generation." EACL 2024 System Demonstrations.
- Jimenez, C. E. et al. (2024). "SWE-bench: Can Language Models Resolve Real-World GitHub Issues?" ICLR 2024.
- Yao, S., Shinn, N., Razavi, P., Narasimhan, K. (2024). "τ-bench: A Benchmark for Tool-Agent-User Interaction in Real-World Domains." arXiv.
- Kapoor, S., Stroebl, B., Siegel, Z. S., Nadgir, N., Narayanan, A. (2024). "AI Agents That Matter." arXiv.
- Ribeiro, M. T., Wu, T., Guestrin, C., Singh, S. (2020). "Beyond Accuracy: Behavioral Testing of NLP Models with CheckList." ACL 2020.
- Recht, B., Roelofs, R., Schmidt, L., Shankar, V. (2019). "Do ImageNet Classifiers Generalize to ImageNet?" ICML 2019.
- Kiela, D. et al. (2021). "Dynabench: Rethinking Benchmarking in NLP." NAACL 2021.
- Goodhart, C. A. E. (1975). "Problems of Monetary Management: The U.K. Experience." Papers in Monetary Economics, Reserve Bank of Australia.
- Wang, L., Yang, N., Huang, X., Yang, L., Majumder, R., Wei, F. (2024). "Multilingual E5 Text Embeddings: A Technical Report." arXiv.
- Qwen Team (2024). "Qwen2.5-1.5B-Instruct" モデルカード. https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
- Qwen Team (2024). "Qwen2.5-3B-Instruct" モデルカード. https://huggingface.co/Qwen/Qwen2.5-3B-Instruct
- Hugging Face. "Transformers Documentation." https://huggingface.co/docs/transformers/